# CRISP-Net — Conformal Risk-Interval Station Prediction

Event-conditional risk bounds for metro passenger flow. Nanjing Metro AFC 2023 · 191 stations · 13 lines · 57 days · 190.9 M directional taps.

Prior work in this programme is referenced by its own name: **TDAG-Net** (Paper 3) and the earlier pilot are baselines here, not this model.


## Mount Google Drive

In [1]:
import sys, os

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    else:
        print("Google Drive is already mounted.")
except ImportError:
    print("Not running in Google Colab — using local filesystem.")


Mounted at /content/drive


## 0 · Environment bootstrap

In [3]:
import importlib
import os
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        else:
            print("Drive already mounted.")
    except Exception as exc:
        print(f"Drive mount skipped: {exc}")
else:
    print("Running in local environment.")

_REQUIRED = {
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "networkx": "networkx",
    "pyarrow": "pyarrow",
    "tqdm": "tqdm",
    "scipy": "scipy",
    "tabulate": "tabulate",
}
_missing = []
for _mod, _pkg in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ImportError:
        _missing.append(_pkg)

if _missing:
    print(f"Installing missing packages: {_missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=False)

import numpy as np
import pandas as pd
import torch

print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f"  →  {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("  →  CPU only")
print(f"In Colab    : {IN_COLAB}")


Drive already mounted.
Python      : 3.12.13
NumPy       : 2.0.2
Pandas      : 2.2.2
PyTorch     : 2.11.0+cu128
CUDA        : True  →  Tesla T4 (15.6 GB)
In Colab    : True


## 1 · Configuration

In [4]:
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple

if IN_COLAB:
    _DRIVE = "/content/drive/MyDrive"
else:
    _DRIVE = os.path.abspath(os.environ.get("CALM_DRIVE", os.environ.get("TDAG_DRIVE", ".")))


@dataclass
class Config:
    # ── Paths ────────────────────────────────────────────────────────────────
    drive_root: str = _DRIVE
    dataset_root: str = f"{_DRIVE}/Dataset/2nd SCI-T1 paper"
    project_name: str = "CRISP_Net"   # rename the Drive folder to match, then set this
    project_root: str = ""            # derived below from project_name
    legacy_project_root: str = f"{_DRIVE}/TDAG_Net"
    legacy_manifest: str = f"{_DRIVE}/afc_file_manifest.csv"
    reuse_legacy_agg: bool = True

    # ── Experiment versioning ────────────────────────────────────────────────
    exp_version: str = "v2"        # ← deliberately NOT bumped; see note 2 above
    paper: str = "CRISP-Net"

    # ── Which corpus ─────────────────────────────────────────────────────────
    dataset: str = "2023"
    exclude_station_ids: Tuple[int, ...] = (333,)

    # ── Temporal discretisation ──────────────────────────────────────────────
    freq_minutes: int = 10
    service_start: str = "06:00"
    service_end: str = "23:50"
    min_records_per_day: int = 200_000

    # ── Ticket / persona ─────────────────────────────────────────────────────
    n_card_channels: int = 24
    n_personas: int = 4
    persona_names: Tuple[str, ...] = ("Commuter", "Casual", "Tourist/Visitor",
                                      "Concession/Student")

    # ── Windowing ────────────────────────────────────────────────────────────
    seq_len: int = 12
    horizon: int = 6
    train_frac: float = 0.70
    val_frac: float = 0.15
    calib_frac: float = 0.50

    # ── Graph ────────────────────────────────────────────────────────────────
    adj_topk: int = 8
    venue_radius_km: float = 3.0
    corr_min: float = 0.30

    # ── Backbone (TDAG-Net encoder, unchanged — no new backbone claim) ───────
    hidden_dim: int = 64
    tcn_channels: int = 64
    tcn_dilations: Tuple[int, ...] = (1, 2, 4, 8)
    gat_heads: int = 4
    gat_dropout: float = 0.10
    lstm_hidden: int = 128
    dropout: float = 0.10
    node_emb_dim: int = 16

    # ── CRISP-Net heads ──────────────────────────────────────────────────────
    quantiles: Tuple[float, ...] = (0.05, 0.50, 0.95)
    pinball_weight: float = 0.30    # tail quantiles only; the median keeps Huber
    gate_init: float = 1.0          # persona channel gates start at identity
    gate_l1: float = 1e-4           # mild sparsity so a dead persona can switch off

    # ── Descriptive only (no longer a model parameter) ──────────────────────
    delay_lags: Tuple[int, ...] = (0, 1, 2, 3)

    # ── Training (IDENTICAL to v2 — do not touch, see note 3) ───────────────
    batch_size: int = 32
    epochs: int = 80
    lr: float = 3e-4
    weight_decay: float = 1e-4
    huber_delta: float = 1.0
    grad_clip: float = 5.0
    early_stop_patience: int = 15
    lr_patience: int = 6
    amp: bool = True
    aux_persona_weight: float = 0.05    # now carries the gate-L1 term
    aux_entropy_weight: float = 0.0     # no kernel to regularise → exactly zero

    # ── Reproducibility ──────────────────────────────────────────────────────
    seed: int = 42
    report_seeds: Tuple[int, ...] = (42,)   # ← ONE seed, one run (note 1)
    deterministic: bool = False

    # ── Conformal prediction ─────────────────────────────────────────────────
    alpha: float = 0.10
    mondrian_bins: Tuple[str, ...] = ("event", "peak")
    min_bin_windows: int = 40            # ← 100 → 50 → 40; see note 5
    bin_threshold_sweep: Tuple[int, ...] = (40, 50, 100)  # reported as sensitivity
    cross_conformal_k: int = 5
    enable_cross_conformal: bool = True
    per_channel_conformal: bool = True   # calibrate entry and exit separately
    peak_hours: Tuple[Tuple[float, float], ...] = ((7.0, 9.5), (17.0, 19.5))
    mc_dropout_passes: int = 30
    ensemble_members: int = 5            # the proposal's M = 5, restored

    # ── Cross-architecture conformal transfer (inference only, no training) ──
    enable_conformal_transfer: bool = True
    transfer_models: Tuple[str, ...] = ("AGCRN", "Graph WaveNet", "TDAG-Net",
                                        "TDAG-Net (Input-Splitter)")

    # ── Checkpointing / resume ───────────────────────────────────────────────
    ckpt_every_epochs: int = 1
    ckpt_every_minutes: float = 5.0
    keep_last_n_ckpts: int = 2

    # ── Execution control ────────────────────────────────────────────────────
    quick_test: bool = False
    force_recompute: Tuple[str, ...] = ()
    preload_to_gpu: bool = True
    chunksize: int = 2_000_000
    n_ingest_files: int = -1

    # ── Evaluation ───────────────────────────────────────────────────────────
    event_peak_quantile: float = 0.90
    report_stations: Tuple[int, ...] = ()

    def __post_init__(self):
        if self.quick_test:
            self.epochs = 3
            self.n_ingest_files = 3
            self.early_stop_patience = 99
            self.ensemble_members = 2
            self.enable_cross_conformal = False
            self.enable_conformal_transfer = False
            self.mc_dropout_passes = 5
            self.report_seeds = (self.seed,)

    def hash_of(self, fields: List[str]) -> str:
        import hashlib
        import json as _json
        payload = {k: getattr(self, k) for k in fields}
        blob = _json.dumps(payload, sort_keys=True, default=str)
        return hashlib.md5(blob.encode()).hexdigest()[:10]


CFG = Config()

# Resolve the project folder. Rename the folder in Drive and set project_name to
# match; if that folder is not there yet, fall back to whichever previous name
# still exists so no cached result is orphaned by a half-finished rename.
_FOLDER_CANDIDATES = [CFG.project_name, "CRISP_Net", "CALM_Net"]
_found = next((n for n in _FOLDER_CANDIDATES if os.path.isdir(f"{_DRIVE}/{n}")), None)
PROJECT_FOLDER = _found or CFG.project_name
CFG.project_root = f"{_DRIVE}/{PROJECT_FOLDER}"
if _found and _found != CFG.project_name:
    print(f"  NOTE: '{CFG.project_name}' not found on Drive — using existing "
          f"'{_found}'.\n        Rename the folder to '{CFG.project_name}' to complete "
          f"the move; all\n        caches travel with it.")

# ── Derived paths ────────────────────────────────────────────────────────────
P = {
    "root":      CFG.project_root,
    "cache":     f"{CFG.project_root}/cache",
    "agg":       f"{CFG.project_root}/cache/agg",
    "tensors":   f"{CFG.project_root}/cache/tensors",
    "graphs":    f"{CFG.project_root}/cache/graphs",
    "personas":  f"{CFG.project_root}/cache/personas",
    "events":    f"{CFG.project_root}/cache/events",
    "meta":      f"{CFG.project_root}/data/metadata",
    "runs":      f"{CFG.project_root}/runs",
    "figures":   f"{CFG.project_root}/figures",
    "results":   f"{CFG.project_root}/results",
    "metrics":   f"{CFG.project_root}/results/metrics",
    "tables":    f"{CFG.project_root}/results/tables",
    "preds":     f"{CFG.project_root}/results/predictions",
    "conformal": f"{CFG.project_root}/results/conformal",
    "exports":   f"{CFG.project_root}/exports",
    "logs":      f"{CFG.project_root}/logs",
}

# ── Stage-A reuse: point `agg` at the Paper-3 cache when it already holds data ─
LEGACY_AGG = f"{CFG.legacy_project_root}/cache/agg"
AGG_SOURCE = "fresh"
if CFG.reuse_legacy_agg and os.path.isdir(LEGACY_AGG):
    import glob as _glob
    _n_legacy = len(_glob.glob(f"{LEGACY_AGG}/*__flow.parquet"))
    if _n_legacy > 0:
        P["agg"] = LEGACY_AGG
        AGG_SOURCE = f"reused from Paper 3 cache ({_n_legacy} flow parquet files)"

DZ_EXPECTED = 8

print(f"Configuration loaded — {CFG.paper}")
print(f"  project folder  : {PROJECT_FOLDER}")
print(f"  exp_version     : {CFG.exp_version}  (kept → cached v2 baselines are reused)")
print(f"  dataset         : {CFG.dataset}")
print(f"  Stage-A cache   : {AGG_SOURCE}")
print(f"  quick_test      : {CFG.quick_test}")
print(f"  L → H           : {CFG.seq_len} → {CFG.horizon} "
      f"({CFG.seq_len * CFG.freq_minutes} min → {CFG.horizon * CFG.freq_minutes} min)")
print(f"  seeds           : {list(CFG.report_seeds)}   ← single run per model")
print(f"  quantiles       : {list(CFG.quantiles)}   pinball weight {CFG.pinball_weight}")
print(f"  conformal       : α={CFG.alpha} · per-channel={CFG.per_channel_conformal} "
      f"· min bin {CFG.min_bin_windows} windows · {CFG.cross_conformal_k}-fold fallback")
print(f"  bin sweep       : {list(CFG.bin_threshold_sweep)} windows "
      f"(sensitivity check reported in Table VI)")
if CFG.min_bin_windows > 48:
    print(f"  ⚠ threshold {CFG.min_bin_windows} > 48 — the peak bins WILL fall back "
          f"and Table VI\n    will measure the fallback rather than the binning.")
print(f"  transfer        : {CFG.enable_conformal_transfer} → {list(CFG.transfer_models)}")
print(f"  ensemble        : M = {CFG.ensemble_members}")


Configuration loaded — CRISP-Net
  project folder  : CRISP_Net
  exp_version     : v2  (kept → cached v2 baselines are reused)
  dataset         : 2023
  Stage-A cache   : reused from Paper 3 cache (17 flow parquet files)
  quick_test      : False
  L → H           : 12 → 6 (120 min → 60 min)
  seeds           : [42]   ← single run per model
  quantiles       : [0.05, 0.5, 0.95]   pinball weight 0.3
  conformal       : α=0.1 · per-channel=True · min bin 40 windows · 5-fold fallback
  bin sweep       : [40, 50, 100] windows (sensitivity check reported in Table VI)
  transfer        : True → ['AGCRN', 'Graph WaveNet', 'TDAG-Net', 'TDAG-Net (Input-Splitter)']
  ensemble        : M = 5


## 2 · Project scaffold and metadata templates

In [5]:
import json
from pathlib import Path

FIG_SUBDIRS = ["01_corpus", "02_persona", "03_graph", "04_event",
               "05_training", "06_results", "07_uncertainty"]

for _k, _v in P.items():
    if _v == LEGACY_AGG:          # never mkdir into the read-only legacy cache
        continue
    Path(_v).mkdir(parents=True, exist_ok=True)
for _sub in FIG_SUBDIRS:
    Path(f"{P['figures']}/{_sub}").mkdir(parents=True, exist_ok=True)

# ── Inherit Paper-3 metadata if present, else write fresh templates ──────────
_LEGACY_META = f"{CFG.legacy_project_root}/data/metadata"


def _seed_meta(fname: str, builder):
    """Copy the Paper-3 metadata file if it exists; otherwise write a blank template."""
    dst = f"{P['meta']}/{fname}"
    if os.path.exists(dst):
        return dst, "existing"
    src = f"{_LEGACY_META}/{fname}"
    if os.path.exists(src):
        try:
            pd.read_csv(src).to_csv(dst, index=False)
            return dst, "inherited from TDAG_Net"
        except Exception:
            pass
    builder().to_csv(dst, index=False)
    return dst, "new template"


_station_tpl, _s1 = _seed_meta(
    "station_metadata.csv",
    lambda: pd.DataFrame(columns=["station_id", "station_name_cn", "station_name_en",
                                  "latitude", "longitude", "line_primary",
                                  "is_interchange", "station_class"]))

_venue_tpl, _s2 = _seed_meta(
    "event_venues.csv",
    lambda: pd.DataFrame([
        {"venue_id": "NJOSC", "venue_name": "Nanjing Olympic Sports Center",
         "latitude": 32.0021, "longitude": 118.7301, "capacity": 61443,
         "nearest_station_ids": "", "notes": "optional — CRISP-Net does not require this"},
        {"venue_id": "NJGYM", "venue_name": "Nanjing Gymnasium (Wutaishan)",
         "latitude": 32.0498, "longitude": 118.7712, "capacity": 20000,
         "nearest_station_ids": "", "notes": ""},
        {"venue_id": "NJIEC", "venue_name": "Nanjing International Expo Center",
         "latitude": 32.0126, "longitude": 118.7261, "capacity": 30000,
         "nearest_station_ids": "", "notes": ""},
    ]))

_event_tpl, _s3 = _seed_meta(
    "event_calendar.csv",
    lambda: pd.DataFrame(columns=["date", "venue_id", "event_type", "start_time", "end_time",
                                  "expected_attendance", "source", "verified"]))

_weather_tpl, _s4 = _seed_meta(
    "weather.csv",
    lambda: pd.DataFrame(columns=["datetime", "temperature_c", "precipitation_mm", "humidity_pct"]))

print("Project scaffold ready:")
for _k in ["root", "cache", "agg", "meta", "runs", "figures", "results", "conformal", "exports"]:
    print(f"  {_k:9s} → {P[_k]}")

print("\nMetadata files (all optional — CRISP-Net runs fully without them):")
for _t, _how in [(_station_tpl, _s1), (_venue_tpl, _s2), (_event_tpl, _s3), (_weather_tpl, _s4)]:
    try:
        # local guard: read_meta() lives in the next cell, so it is not available here
        _n = len(pd.read_csv(_t)) if os.path.getsize(_t) > 0 else 0
    except Exception:
        _n = 0
    print(f"  {'✓ filled' if _n else '· empty '}  {os.path.basename(_t):24s} "
          f"({_n} rows, {_how})")


Project scaffold ready:
  root      → /content/drive/MyDrive/CRISP_Net
  cache     → /content/drive/MyDrive/CRISP_Net/cache
  agg       → /content/drive/MyDrive/TDAG_Net/cache/agg
  meta      → /content/drive/MyDrive/CRISP_Net/data/metadata
  runs      → /content/drive/MyDrive/CRISP_Net/runs
  figures   → /content/drive/MyDrive/CRISP_Net/figures
  results   → /content/drive/MyDrive/CRISP_Net/results
  conformal → /content/drive/MyDrive/CRISP_Net/results/conformal
  exports   → /content/drive/MyDrive/CRISP_Net/exports

Metadata files (all optional — CRISP-Net runs fully without them):
  ✓ filled  station_metadata.csv     (192 rows, existing)
  ✓ filled  event_venues.csv         (5 rows, existing)
  ✓ filled  event_calendar.csv       (12 rows, existing)
  · empty   weather.csv              (0 rows, existing)


## 3 · Core infrastructure — atomic IO, stage cache, logging

In [6]:
import contextlib
import functools
import hashlib
import logging
import pickle
import random
import shutil
import time
import traceback
import json
import sys
import os
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional

# ── Logging: stdout + rotating file, survives session restarts ───────────────
_LOG_FILE = f"{P['logs']}/crispnet_{datetime.now().strftime('%Y%m%d')}.log"
logger = logging.getLogger("crispnet")
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fmt = logging.Formatter("%(asctime)s │ %(levelname)-7s │ %(message)s", "%H:%M:%S")
_sh = logging.StreamHandler(sys.stdout)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh = logging.FileHandler(_LOG_FILE, encoding="utf-8")
_fh.setFormatter(logging.Formatter("%(asctime)s │ %(levelname)-7s │ %(message)s"))
logger.addHandler(_fh)
logger.propagate = False


def log(msg, level="info"):
    getattr(logger, level)(msg)


# ── Reproducibility ──────────────────────────────────────────────────────────
def set_seed(seed: int = None, deterministic: bool = None):
    seed = CFG.seed if seed is None else seed
    deterministic = CFG.deterministic if deterministic is None else deterministic
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ── Atomic writers ───────────────────────────────────────────────────────────
@contextlib.contextmanager
def atomic_path(path: str):
    """Yield a temp path; atomically move into place on clean exit."""
    tmp = f"{path}.tmp_{os.getpid()}"
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    try:
        yield tmp
        os.replace(tmp, path)
    except Exception:
        if os.path.exists(tmp):
            with contextlib.suppress(OSError):
                os.remove(tmp)
        raise


def save_json(obj, path):
    def _default(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (pd.Timestamp, datetime)):
            return str(o)
        return str(o)

    with atomic_path(path) as tmp:
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=2, default=_default, ensure_ascii=False)


def load_json(path, default=None):
    if not os.path.exists(path):
        return default
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError):
        log(f"Corrupt JSON at {path} — ignoring", "warning")
        return default


def save_npz(path, **arrays):
    tmp = f"{path}.tmp_{os.getpid()}.npz"
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    try:
        np.savez_compressed(tmp, **arrays)
        os.replace(tmp, path)
    except Exception:
        if os.path.exists(tmp):
            with contextlib.suppress(OSError):
                os.remove(tmp)
        raise


def save_pickle(obj, path):
    with atomic_path(path) as tmp:
        with open(tmp, "wb") as f:
            pickle.dump(obj, f, protocol=4)


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_df(df: pd.DataFrame, path: str):
    with atomic_path(path) as tmp:
        if path.endswith(".parquet"):
            df.to_parquet(tmp, index=False)
        else:
            df.to_csv(tmp, index=False)


def read_meta(fname: str, columns: List[str] = None) -> pd.DataFrame:
    """Read an OPTIONAL metadata CSV from data/metadata."""
    path = os.path.join(P["meta"], fname)
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return pd.DataFrame(columns=columns or [])
    try:
        df = pd.read_csv(path)
    except (pd.errors.EmptyDataError, pd.errors.ParserError, OSError) as exc:
        log(f"Unreadable metadata file {fname} ({type(exc).__name__}) — treating as empty",
            "warning")
        return pd.DataFrame(columns=columns or [])
    if columns:
        for c in columns:
            if c not in df.columns:
                df[c] = np.nan
    return df.dropna(how="all")


# ── Stage cache decorator ────────────────────────────────────────────────────
_STAGE_REGISTRY: Dict[str, dict] = {}


def stage(name: str, depends_on: List[str] = None, ext: str = "pkl", subdir: str = "cache"):
    depends_on = depends_on or []

    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*args, force: bool = False, **kwargs):
            key = CFG.hash_of(depends_on) if depends_on else "static"
            fname = f"{name}__{CFG.dataset}__{key}.{ext}"
            path = os.path.join(P.get(subdir, P["cache"]), fname)
            invalidated = force or (name in CFG.force_recompute)

            if os.path.exists(path) and not invalidated:
                t0 = time.time()
                if ext == "parquet":
                    out = pd.read_parquet(path)
                elif ext == "json":
                    out = load_json(path)
                else:
                    out = load_pickle(path)
                log(f"✓ cached  [{name}] loaded in {time.time() - t0:.1f}s  ({fname})")
                _STAGE_REGISTRY[name] = {"path": path, "cached": True}
                return out

            log(f"▶ running [{name}] …")
            t0 = time.time()
            out = fn(*args, **kwargs)
            if ext == "parquet":
                save_df(out, path)
            elif ext == "json":
                save_json(out, path)
            else:
                save_pickle(out, path)
            log(f"✔ done    [{name}] in {time.time() - t0:.1f}s → {fname}")
            _STAGE_REGISTRY[name] = {"path": path, "cached": False}
            return out

        wrapper.stage_name = name
        wrapper.cache_path = lambda: os.path.join(
            P.get(subdir, P["cache"]),
            f"{name}__{CFG.dataset}__{CFG.hash_of(depends_on) if depends_on else 'static'}.{ext}",
        )
        return wrapper

    return deco


# ── Timer ────────────────────────────────────────────────────────────────────
@contextlib.contextmanager
def timed(label: str):
    t0 = time.time()
    yield
    log(f"⏱  {label}: {time.time() - t0:.1f}s")


def mem_report(tag=""):
    try:
        import psutil
        rss = psutil.Process().memory_info().rss / 1e9
        msg = f"RAM {rss:.2f} GB"
    except Exception:
        msg = "RAM n/a"
    try:
        if torch.cuda.is_available():
            msg += (f" │ GPU {torch.cuda.memory_allocated() / 1e9:.2f}/"
                    f"{torch.cuda.memory_reserved() / 1e9:.2f} GB")
    except Exception:
        pass
    log(f"{tag} {msg}")


# ── Figure registry ──────────────────────────────────────────────────────────
FIGURE_INDEX = f"{P['figures']}/INDEX.csv"


def _register_figure(fid, title, path, group, data_path=""):
    row = pd.DataFrame([{
        "figure_id": fid, "title": title, "group": group,
        "png": path, "data": data_path,
        "generated_at": datetime.now().isoformat(timespec="seconds"),
    }])
    if os.path.exists(FIGURE_INDEX):
        idx = pd.read_csv(FIGURE_INDEX)
        idx = idx[idx["figure_id"] != fid]
        row = pd.concat([idx, row], ignore_index=True)
    row = row.sort_values("figure_id")
    save_df(row, FIGURE_INDEX)


FIGURE_REGISTRY = {}   # fid -> metadata for every declared figure


def figure(fid: str, title: str, group: str = "01_corpus", save_pdf: bool = True):
    """Decorator: run the plotting function, save PNG (600 dpi) + vector PDF + the underlying dataframe as CSV, and register the result."""
    def deco(fn):
        FIGURE_REGISTRY[fid] = {"figure_id": fid, "title": title,
                                "group": group, "func": fn.__name__}

        @functools.wraps(fn)
        def wrapper(*args, force: bool = False, **kwargs):
            png = f"{P['figures']}/{group}/{fid}_{fn.__name__}.png"
            pdf = png.replace(".png", ".pdf")
            csvp = png.replace(".png", "_data.csv")
            if os.path.exists(png) and not force and "figures" not in CFG.force_recompute:
                log(f"✓ figure  [{fid}] done — loaded from disk (force=True to redraw)")
                return png
            import matplotlib.pyplot as plt

            try:
                out = fn(*args, **kwargs)
            except Exception as exc:
                log(f"✗ figure  [{fid}] FAILED: {type(exc).__name__}: {exc}", "error")
                log(traceback.format_exc(limit=4), "error")
                return None
            if out is None:
                log(f"· figure  [{fid}] skipped — prerequisites unavailable")
                return None
            fig, data = out if isinstance(out, tuple) else (out, None)
            if fig is None:
                log(f"· figure  [{fid}] skipped — prerequisites unavailable")
                return None
            # IEEE: 600 dpi raster + embedded-font vector, no extra whitespace
            fig.savefig(png, dpi=600, facecolor="white", pad_inches=0.02,
                        bbox_inches="tight")
            if save_pdf:
                with contextlib.suppress(Exception):
                    fig.savefig(pdf, facecolor="white", pad_inches=0.02,
                                bbox_inches="tight")
            if isinstance(data, pd.DataFrame):
                save_df(data, csvp)
            _register_figure(fid, title, png, group,
                             csvp if isinstance(data, pd.DataFrame) else "")
            plt.close(fig)
            log(f"🖼  figure  [{fid}] {title} → {os.path.basename(png)}")
            return png

        wrapper.figure_id = fid
        wrapper.figure_title = title
        return wrapper

    return deco


def _skip(fid, reason):
    """Uniform 'could not draw' path — logs why, returns the (None, None) contract."""
    log(f"· figure  [{fid}] skipped — {reason}")
    return None, None


# ── Global run-state (survives restarts) ─────────────────────────────────────
STATE_PATH = f"{P['root']}/pipeline_state.json"


def state_get(key, default=None):
    return (load_json(STATE_PATH, {}) or {}).get(key, default)


def state_set(key, value):
    s = load_json(STATE_PATH, {}) or {}
    s[key] = value
    s["_updated"] = datetime.now().isoformat(timespec="seconds")
    save_json(s, STATE_PATH)


save_json(asdict(CFG), f"{P['root']}/config_active.json")
log("Core infrastructure online — CRISP-Net.")
log(f"Device: {DEVICE} │ Log file: {_LOG_FILE}")
mem_report("Startup:")


def figure_status_report(fids=None, show=True):
    """Status of every declared figure, read from the files already on disk."""
    rows = []
    for fid, meta in sorted(FIGURE_REGISTRY.items()):
        if fids is not None and fid not in fids:
            continue
        png = f"{P['figures']}/{meta['group']}/{fid}_{meta['func']}.png"
        pdf, csvp = png.replace(".png", ".pdf"), png.replace(".png", "_data.csv")
        ok = os.path.exists(png)
        rows.append({**meta, "status": "DONE" if ok else "MISSING",
                     "png": ok, "pdf": os.path.exists(pdf), "csv": os.path.exists(csvp),
                     "kb": round(os.path.getsize(png) / 1024, 1) if ok else 0.0,
                     "path": png})
    df = pd.DataFrame(rows)
    if show and len(df):
        done = int((df["status"] == "DONE").sum())
        print(f"\nFIGURE STATUS — {done}/{len(df)} done")
        print(f"{'id':<6s} {'status':<8s} {'grp':<16s} {'png':>4s} {'pdf':>4s} "
              f"{'csv':>4s} {'KB':>7s}  title")
        print("─" * 108)
        for _, r in df.iterrows():
            print(f"{r['figure_id']:<6s} {r['status']:<8s} {r['group']:<16s} "
                  f"{'y' if r['png'] else '·':>4s} {'y' if r['pdf'] else '·':>4s} "
                  f"{'y' if r['csv'] else '·':>4s} {r['kb']:>7.1f}  {r['title'][:52]}")
        missing = df[df["status"] == "MISSING"]["figure_id"].tolist()
        if missing:
            print(f"\n  MISSING: {missing} — re-run that cell with force=True")
    return df


07:47:10 │ INFO    │ Core infrastructure online — CRISP-Net.
07:47:10 │ INFO    │ Device: cuda │ Log file: /content/drive/MyDrive/CRISP_Net/logs/crispnet_20260817.log
07:47:10 │ INFO    │ Startup: RAM 0.77 GB │ GPU 0.00/0.00 GB


## 4 · AFC reader

In [7]:
import glob
import gc
import re

from tqdm.auto import tqdm

COLS = ["entry_time", "transaction_time", "transaction_type", "device_id",
        "entry_station_id", "line_id", "transaction_station_id", "card_type"]

READ_DTYPES = {
    "transaction_type": "float32",   # float first — some files have NaNs; cast after dropna
    "device_id": "float32",
    "entry_station_id": "float32",
    "line_id": "float32",
    "transaction_station_id": "float32",
    "card_type": "float32",
}

_DATE_PAT = re.compile(r"^\d{4}-\d{1,2}-\d{1,2}")


def detect_dialect(path: str) -> str:
    if path.endswith("_clean.txt"):
        return "cleaned_utf8"
    try:
        with open(path, "r", encoding="gbk", errors="replace") as fh:
            first = fh.readline()
    except OSError:
        return "unknown"
    if first.startswith("进站时间"):
        return "clean_csv"
    if "SQL Statement" in first:
        return "sql_header"
    return "unknown"


def find_data_start_and_delim(path: str, max_scan: int = 200, encoding="gbk"):
    """First line that looks like a data row + the delimiter it uses."""
    with open(path, "r", encoding=encoding, errors="replace") as fh:
        for i, line in enumerate(fh):
            if i > max_scan:
                return None, None
            if _DATE_PAT.match(line.strip()):
                return i, ("\t" if "\t" in line else ",")
    return None, None


def read_afc(path: str, usecols=None, chunksize: Optional[int] = None):
    """Return a DataFrame (chunksize=None) or a chunk iterator, with English column names, regardless of dialect."""
    dialect = detect_dialect(path)
    kw = dict(names=COLS, usecols=usecols, chunksize=chunksize,
              on_bad_lines="skip", engine="c")

    if dialect == "cleaned_utf8":
        return pd.read_csv(path, encoding="utf-8", header=0, **kw)
    if dialect == "clean_csv":
        return pd.read_csv(path, encoding="gbk", header=0, sep=",", **kw)

    skip_n, delim = find_data_start_and_delim(path)
    if skip_n is None:
        raise ValueError(f"No data rows found in {path}")
    # header=None (NOT header=0) so the first data row survives
    return pd.read_csv(path, encoding="gbk", header=None, skiprows=skip_n,
                       sep=delim, **kw)


def peek_content_dates(path: str, n: int = 40):
    """Learn a file's TRUE first date from its content."""
    dialect = detect_dialect(path)
    enc = "utf-8" if dialect == "cleaned_utf8" else "gbk"
    skip = 0
    if dialect == "sql_header":
        skip, _ = find_data_start_and_delim(path)
    elif dialect in ("clean_csv", "cleaned_utf8"):
        skip = 1
    vals = []
    with open(path, "r", encoding=enc, errors="replace") as fh:
        for i, line in enumerate(fh):
            if i < (skip or 0):
                continue
            tok = re.split(r"[,\t]", line.strip())[0]
            if _DATE_PAT.match(tok):
                vals.append(tok)
            if len(vals) >= n:
                break
    if not vals:
        return None
    ts = pd.to_datetime(pd.Series(vals), errors="coerce").dropna()
    ts = ts[ts.dt.year >= 2000]                      # drop 1970 epoch corruption
    if ts.empty:
        return None
    modal_year = int(ts.dt.year.mode().iloc[0])
    return ts[ts.dt.year == modal_year].min()


def to_datetime_safe(s: pd.Series) -> pd.Series:
    """Parse mixed-format datetimes without exploding on a 40 M-row column."""
    try:
        return pd.to_datetime(s, format="mixed", errors="coerce")
    except (TypeError, ValueError):
        return pd.to_datetime(s, errors="coerce")


log("AFC reader ready (4 dialects).")


07:47:10 │ INFO    │ AFC reader ready (4 dialects).


## 5 · File manifest

In [8]:
@stage("file_manifest", depends_on=["dataset_root"], ext="parquet")
def build_file_manifest() -> pd.DataFrame:
    all_txt = sorted(glob.glob(f"{CFG.dataset_root}/**/*.txt", recursive=True))
    originals = [f for f in all_txt if not f.endswith("_clean.txt")]
    rows = []
    for f in tqdm(originals, desc="Manifest"):
        clean = f.replace(".txt", "_clean.txt")
        use = clean if os.path.exists(clean) else f
        try:
            first_dt = peek_content_dates(use)
        except Exception as exc:
            log(f"  peek failed {os.path.basename(f)}: {exc}", "warning")
            first_dt = None
        fname = os.path.basename(f)
        name_year = "2023" if fname.startswith("2023") else ("2018" if fname.startswith("2018") else "?")
        true_year = str(first_dt.year) if first_dt is not None and pd.notna(first_dt) else name_year
        rows.append({
            "file": fname,
            "original": f,
            "load_path": use,
            "was_cleaned": use.endswith("_clean.txt"),
            "dialect": detect_dialect(use),
            "size_mb": round(os.path.getsize(use) / 1e6, 1),
            "first_record": first_dt,
            "year_from_name": name_year,
            "year_from_content": true_year,
            "name_mismatch": name_year != true_year,
        })
    mf = pd.DataFrame(rows).sort_values("file").reset_index(drop=True)
    return mf


MANIFEST = build_file_manifest()

print(f"Manifest: {len(MANIFEST)} source files, {MANIFEST['size_mb'].sum() / 1024:.2f} GB total")
print(MANIFEST["dialect"].value_counts().to_string())
print(f"Pre-cleaned (dedup'd) files reused: {int(MANIFEST['was_cleaned'].sum())}")

_mismatch = MANIFEST[MANIFEST["name_mismatch"]]
if len(_mismatch):
    print(f"\n⚠️  {len(_mismatch)} file(s) whose NAME disagrees with their CONTENT — "
          f"year taken from content:")
    print(_mismatch[["file", "year_from_name", "year_from_content", "first_record"]].to_string(index=False))

print(f"\nFiles by true year:\n{MANIFEST['year_from_content'].value_counts().to_string()}")
save_df(MANIFEST, f"{P['cache']}/file_manifest_full.csv")


07:47:11 │ INFO    │ ✓ cached  [file_manifest] loaded in 0.4s  (file_manifest__2023__7fda0833bf.parquet)
Manifest: 60 source files, 21.85 GB total
dialect
sql_header      29
clean_csv       27
cleaned_utf8     4
Pre-cleaned (dedup'd) files reused: 4

⚠️  1 file(s) whose NAME disagrees with their CONTENT — year taken from content:
           file year_from_name year_from_content        first_record
20230326-31.txt           2023              2018 2018-03-26 05:53:14

Files by true year:
year_from_content
2018    43
2023    17


## 6 · Stage A — per-file aggregation (entry and exit)

In [9]:
DIRECTION_IN, DIRECTION_OUT = 0, 1


def _consolidate(parts: List[pd.DataFrame], keys: List[str], value: str) -> pd.DataFrame:
    """Concat + group-sum a list of partial count frames (vectorised, memory-bounded)."""
    if len(parts) == 1:
        return parts[0]
    return (pd.concat(parts, ignore_index=True)
            .groupby(keys, sort=False, as_index=False)[value].sum())


def _agg_paths(fname: str):
    stem = fname.replace(".txt", "")
    return (f"{P['agg']}/{stem}__flow.parquet",
            f"{P['agg']}/{stem}__meta.json")


def aggregate_one_file(row) -> dict:
    """Stream one AFC file → binned counts. Returns the audit meta dict."""
    fpath, fname = row["load_path"], row["file"]
    out_parquet, out_meta = _agg_paths(fname)
    if os.path.exists(out_parquet) and os.path.exists(out_meta):
        return load_json(out_meta)

    usecols = ["entry_time", "transaction_time", "transaction_type",
               "entry_station_id", "transaction_station_id", "card_type"]
    freq = f"{CFG.freq_minutes}min"

    flow_parts: List[pd.DataFrame] = []
    xfer_parts: List[pd.DataFrame] = []
    meta = {"file": fname, "rows_read": 0, "rows_kept": 0, "rows_nan": 0,
            "rows_epoch_repaired": 0, "rows_dup_dropped": 0,
            "type_counts": {}, "card_types": {}, "stations": set(),
            "date_min": None, "date_max": None, "day_counts": {}}

    reader = read_afc(fpath, usecols=usecols, chunksize=CFG.chunksize)
    if isinstance(reader, pd.DataFrame):
        reader = [reader]

    for chunk in reader:
        meta["rows_read"] += len(chunk)

        for c in ["transaction_type", "entry_station_id", "transaction_station_id", "card_type"]:
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce")
        before = len(chunk)
        chunk = chunk.dropna(subset=["entry_station_id", "transaction_station_id", "card_type"])
        meta["rows_nan"] += before - len(chunk)
        if chunk.empty:
            continue

        before = len(chunk)
        chunk = chunk.drop_duplicates()
        meta["rows_dup_dropped"] += before - len(chunk)

        t_in = to_datetime_safe(chunk["entry_time"])
        t_out = to_datetime_safe(chunk["transaction_time"])

        # repair epoch-corrupted entry timestamps (audit: 1970-01-01 08:00:00)
        bad = t_in.isna() | (t_in.dt.year < 2000)
        n_bad = int(bad.sum())
        if n_bad:
            t_in = t_in.where(~bad, t_out)
            meta["rows_epoch_repaired"] += n_bad

        ok = t_in.notna() & t_out.notna()
        chunk, t_in, t_out = chunk[ok], t_in[ok], t_out[ok]
        if chunk.empty:
            continue
        meta["rows_kept"] += len(chunk)

        ttype = chunk["transaction_type"].fillna(-1).astype("int16")
        card = chunk["card_type"].astype("int32")
        st_in = chunk["entry_station_id"].astype("int32")
        st_out = chunk["transaction_station_id"].astype("int32")

        vc = ttype.value_counts()
        for k, v in vc.items():
            meta["type_counts"][str(int(k))] = meta["type_counts"].get(str(int(k)), 0) + int(v)
        cvc = card.value_counts()
        for k, v in cvc.items():
            meta["card_types"][str(int(k))] = meta["card_types"].get(str(int(k)), 0) + int(v)
        meta["stations"] |= set(st_in.unique().tolist()) | set(st_out.unique().tolist())

        bin_in = t_in.dt.floor(freq)
        bin_out = t_out.dt.floor(freq)

        dvc = t_in.dt.date.value_counts()
        for k, v in dvc.items():
            meta["day_counts"][str(k)] = meta["day_counts"].get(str(k), 0) + int(v)
        dmin, dmax = t_in.min(), t_in.max()
        meta["date_min"] = str(min(pd.Timestamp(meta["date_min"]), dmin)) if meta["date_min"] else str(dmin)
        meta["date_max"] = str(max(pd.Timestamp(meta["date_max"]), dmax)) if meta["date_max"] else str(dmax)

        # ── inbound (boarding at entry_station_id at entry_time) ──────────────
        gi = (pd.DataFrame({"time_bin": bin_in.values, "station_id": st_in.values,
                            "card_type": card.values})
              .groupby(["time_bin", "station_id", "card_type"], sort=False)
              .size().reset_index(name="count"))
        gi["direction"] = np.int8(DIRECTION_IN)

        # ── outbound (alighting at transaction_station_id at transaction_time) ─
        go = (pd.DataFrame({"time_bin": bin_out.values, "station_id": st_out.values,
                            "card_type": card.values})
              .groupby(["time_bin", "station_id", "card_type"], sort=False)
              .size().reset_index(name="count"))
        go["direction"] = np.int8(DIRECTION_OUT)
        flow_parts.append(pd.concat([gi, go], ignore_index=True))

        # ── transfers (type == 4) counted at the destination station ─────────
        t4 = (ttype == 4).to_numpy()
        if t4.any():
            xfer_parts.append(
                pd.DataFrame({"time_bin": bin_out.values[t4], "station_id": st_out.values[t4]})
                .groupby(["time_bin", "station_id"], sort=False).size()
                .reset_index(name="transfers"))

        if len(flow_parts) >= 8:
            flow_parts = [_consolidate(flow_parts, ["time_bin", "station_id",
                                                    "direction", "card_type"], "count")]
        if len(xfer_parts) >= 8:
            xfer_parts = [_consolidate(xfer_parts, ["time_bin", "station_id"], "transfers")]

        del chunk, t_in, t_out, gi, go
        gc.collect()

    if flow_parts:
        flow = _consolidate(flow_parts, ["time_bin", "station_id", "direction", "card_type"], "count")
        flow["station_id"] = flow["station_id"].astype(np.int32)
        flow["direction"] = flow["direction"].astype(np.int8)
        flow["card_type"] = flow["card_type"].astype(np.int16)
        flow["count"] = flow["count"].astype(np.int32)
        flow = flow[["time_bin", "station_id", "direction", "card_type", "count"]]
    else:
        flow = pd.DataFrame(columns=["time_bin", "station_id", "direction", "card_type", "count"])

    if xfer_parts:
        tdf = _consolidate(xfer_parts, ["time_bin", "station_id"], "transfers")
        tdf["station_id"] = tdf["station_id"].astype(np.int32)
        tdf["transfers"] = tdf["transfers"].astype(np.int32)
    else:
        tdf = pd.DataFrame(columns=["time_bin", "station_id", "transfers"])

    save_df(flow, out_parquet)
    save_df(tdf, out_parquet.replace("__flow.parquet", "__transfer.parquet"))
    meta["stations"] = sorted(int(s) for s in meta["stations"])
    meta["n_bins"] = int(flow["time_bin"].nunique()) if len(flow) else 0
    meta["completed_at"] = datetime.now().isoformat(timespec="seconds")
    save_json(meta, out_meta)
    del flow_parts, xfer_parts, flow, tdf
    gc.collect()
    return meta


def run_ingestion(manifest: pd.DataFrame = None, dataset: str = None):
    """Resumable driver. Re-run freely: finished files are skipped instantly."""
    manifest = MANIFEST if manifest is None else manifest
    dataset = CFG.dataset if dataset is None else dataset
    todo = manifest[manifest["year_from_content"] == dataset].copy()
    if CFG.n_ingest_files > 0:
        todo = todo.head(CFG.n_ingest_files)

    done, pending = [], []
    for _, r in todo.iterrows():
        (pq, mj) = _agg_paths(r["file"])
        (done if (os.path.exists(pq) and os.path.exists(mj)) else pending).append(r["file"])

    log(f"Ingestion [{dataset}] — {len(done)} done, {len(pending)} pending "
        f"({todo['size_mb'].sum() / 1024:.2f} GB total)  │  cache: {AGG_SOURCE}")
    if not pending:
        log("Nothing to ingest. ✓")

    metas = []
    for _, r in tqdm(list(todo.iterrows()), total=len(todo), desc=f"Ingest {dataset}"):
        try:
            t0 = time.time()
            was_pending = r["file"] in pending
            m = aggregate_one_file(r)
            metas.append(m)
            if was_pending:
                log(f"  ✔ {r['file']:24s} kept={m['rows_kept']:>11,} "
                    f"nan={m['rows_nan']:>5,} epoch_fix={m['rows_epoch_repaired']:>3,} "
                    f"dup={m['rows_dup_dropped']:>9,}  [{time.time() - t0:.0f}s]")
        except Exception as exc:
            log(f"  ✗ {r['file']}: {type(exc).__name__}: {exc}", "error")
            log(traceback.format_exc(limit=4), "error")
    state_set(f"ingest_{dataset}_files_done", len(metas))
    return metas


INGEST_META = run_ingestion()

# corpus-level audit table assembled purely from the sidecar meta files
_audit = pd.DataFrame([{
    "file": m["file"], "rows_read": m["rows_read"], "rows_kept": m["rows_kept"],
    "rows_nan": m["rows_nan"], "epoch_repaired": m["rows_epoch_repaired"],
    "dup_dropped": m["rows_dup_dropped"], "n_bins": m["n_bins"],
    "n_stations": len(m["stations"]), "date_min": m["date_min"], "date_max": m["date_max"],
} for m in INGEST_META])
if len(_audit):
    save_df(_audit, f"{P['cache']}/ingestion_audit_{CFG.dataset}.csv")
    print(_audit.to_string(index=False))
    print(f"\nTotal records read : {_audit['rows_read'].sum():,}")
    print(f"Total records kept : {_audit['rows_kept'].sum():,}")
    print(f"Epoch-repaired ts  : {_audit['epoch_repaired'].sum():,}")
    print(f"NaN rows dropped   : {_audit['rows_nan'].sum():,}")
    print(f"Dupes dropped      : {_audit['dup_dropped'].sum():,}")
    print(f"\nEach retained record contributes ONE entry tap and ONE exit tap → "
          f"{2 * _audit['rows_kept'].sum():,} directional passenger events.")


07:47:11 │ INFO    │ Ingestion [2023] — 17 done, 0 pending (5.08 GB total)  │  cache: reused from Paper 3 cache (17 flow parquet files)
07:47:11 │ INFO    │ Nothing to ingest. ✓


Ingest 2023:   0%|          | 0/17 [00:00<?, ?it/s]

           file  rows_read  rows_kept  rows_nan  epoch_repaired  dup_dropped  n_bins  n_stations            date_min            date_max
20230301-05.txt    8860460    8854058         0               0         6402    1104         191 2020-03-04 09:27:13 2023-03-06 01:25:16
   20230306.txt    1649440    1648675         0               0          765     413         190 2022-11-24 09:11:31 2023-03-07 01:35:13
   20230307.txt    1628261    1627478         0               0          783     350         190 2022-08-10 19:00:41 2023-03-08 01:29:46
   20230308.txt    1771272    1770161         0               0         1111     357         190 2022-10-16 20:56:31 2023-03-09 01:47:46
20230309-12.txt    6939179    6934019         0               1         5160     908         192 2022-07-28 00:43:12 2023-03-13 01:45:34
20230313-16.txt    6798921    6795115         0               1         3806     964         191 2022-06-18 19:15:50 2023-03-17 01:54:55
20230317-19.txt    5765809    5760486    

## 7 · Stage B — day-level source resolution

In [10]:
@stage("day_source_map", depends_on=["dataset", "min_records_per_day"], ext="parquet")
def build_day_source_map() -> pd.DataFrame:
    rows = []
    for _, r in MANIFEST[MANIFEST["year_from_content"] == CFG.dataset].iterrows():
        _, mj = _agg_paths(r["file"])
        m = load_json(mj)
        if not m:
            continue
        for day, n in m.get("day_counts", {}).items():
            rows.append({"date": day, "file": r["file"], "records": int(n)})
    if not rows:
        return pd.DataFrame(columns=["date", "file", "records", "chosen", "n_candidates"])
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df["n_candidates"] = df.groupby("date")["file"].transform("nunique")
    best = df.sort_values("records", ascending=False).drop_duplicates("date")[["date", "file"]]
    best["chosen"] = True
    out = df.merge(best, on=["date", "file"], how="left")
    out["chosen"] = out["chosen"].fillna(False)
    return out.sort_values(["date", "records"], ascending=[True, False]).reset_index(drop=True)


DAY_SOURCE = build_day_source_map()
if len(DAY_SOURCE):
    _chosen = DAY_SOURCE[DAY_SOURCE["chosen"]].copy()
    _contested = DAY_SOURCE[DAY_SOURCE["n_candidates"] > 1]["date"].nunique()
    _full = _chosen[_chosen["records"] >= CFG.min_records_per_day]
    print(f"Calendar days seen           : {DAY_SOURCE['date'].nunique()}")
    print(f"Days covered by >1 file      : {_contested}  → resolved to a single source each")
    print(f"Days kept (≥{CFG.min_records_per_day:,} records) : {len(_full)}")
    print(f"Days dropped as partial      : {len(_chosen) - len(_full)}")
    save_df(DAY_SOURCE, f"{P['cache']}/day_source_map_{CFG.dataset}.csv")
    print("\nKept days:", ", ".join(sorted(_full["date"].dt.strftime("%Y-%m-%d"))[:60]))


07:47:17 │ INFO    │ ✓ cached  [day_source_map] loaded in 0.3s  (day_source_map__2023__9fe854d7ee.parquet)
Calendar days seen           : 209
Days covered by >1 file      : 129  → resolved to a single source each
Days kept (≥200,000 records) : 57
Days dropped as partial      : 152

Kept days: 2023-03-01, 2023-03-02, 2023-03-03, 2023-03-04, 2023-03-05, 2023-03-06, 2023-03-07, 2023-03-08, 2023-03-09, 2023-03-10, 2023-03-11, 2023-03-12, 2023-03-13, 2023-03-14, 2023-03-15, 2023-03-16, 2023-03-17, 2023-03-18, 2023-03-19, 2023-03-25, 2023-03-26, 2023-03-27, 2023-03-28, 2023-03-29, 2023-03-30, 2023-03-31, 2023-05-01, 2023-05-02, 2023-05-03, 2023-05-04, 2023-05-05, 2023-05-06, 2023-05-07, 2023-05-08, 2023-05-09, 2023-05-10, 2023-05-11, 2023-05-12, 2023-05-13, 2023-05-14, 2023-05-15, 2023-05-16, 2023-05-17, 2023-05-18, 2023-05-19, 2023-05-20, 2023-05-21, 2023-05-22, 2023-05-23, 2023-05-24, 2023-05-25, 2023-05-26, 2023-05-27, 2023-05-28, 2023-05-29, 2023-05-30, 2023-05-31


## 8 · Stage C — tensor assembly

In [11]:
@stage("tensors", depends_on=["dataset", "freq_minutes", "service_start", "service_end",
                              "n_card_channels", "min_records_per_day", "exclude_station_ids"],
       ext="pkl", subdir="tensors")
def assemble_tensors() -> dict:
    keep = DAY_SOURCE[(DAY_SOURCE["chosen"]) &
                      (DAY_SOURCE["records"] >= CFG.min_records_per_day)]
    if keep.empty:
        raise RuntimeError("No usable days — check ingestion and min_records_per_day.")
    day_to_file = dict(zip(keep["date"].dt.strftime("%Y-%m-%d"), keep["file"]))
    needed_files = sorted(set(day_to_file.values()))

    parts, xfer_parts = [], []
    for fname in tqdm(needed_files, desc="Load agg"):
        pq, _ = _agg_paths(fname)
        if not os.path.exists(pq):
            continue
        d = pd.read_parquet(pq)
        if d.empty:
            continue
        d["date"] = d["time_bin"].dt.strftime("%Y-%m-%d")
        keep_days = [k for k, v in day_to_file.items() if v == fname]
        d = d[d["date"].isin(keep_days)]
        if len(d):
            parts.append(d.drop(columns="date"))
        tp = pq.replace("__flow.parquet", "__transfer.parquet")
        if os.path.exists(tp):
            t = pd.read_parquet(tp)
            if len(t):
                t["date"] = t["time_bin"].dt.strftime("%Y-%m-%d")
                t = t[t["date"].isin(keep_days)]
                if len(t):
                    xfer_parts.append(t.drop(columns="date"))
        del d
        gc.collect()

    flow = pd.concat(parts, ignore_index=True)
    xfer = pd.concat(xfer_parts, ignore_index=True) if xfer_parts else \
        pd.DataFrame(columns=["time_bin", "station_id", "transfers"])
    del parts, xfer_parts
    gc.collect()

    flow = flow[~flow["station_id"].isin(CFG.exclude_station_ids)]
    xfer = xfer[~xfer["station_id"].isin(CFG.exclude_station_ids)] if len(xfer) else xfer
    station_ids = np.array(sorted(flow["station_id"].unique()), dtype=np.int32)
    st_pos = {int(s): i for i, s in enumerate(station_ids)}
    N = len(station_ids)

    ct_vol = flow.groupby("card_type")["count"].sum().sort_values(ascending=False)
    top_ct = ct_vol.head(CFG.n_card_channels).index.tolist()
    card_channels = [int(c) for c in top_ct] + [-1]          # -1 == OTHER
    ct_pos = {int(c): i for i, c in enumerate(top_ct)}
    C = len(card_channels)
    coverage = float(ct_vol.head(CFG.n_card_channels).sum() / ct_vol.sum())

    days = sorted(pd.to_datetime(list(day_to_file.keys())))
    freq = f"{CFG.freq_minutes}min"
    idx_parts = [pd.date_range(f"{d.strftime('%Y-%m-%d')} {CFG.service_start}",
                               f"{d.strftime('%Y-%m-%d')} {CFG.service_end}", freq=freq)
                 for d in days]
    time_index = pd.DatetimeIndex(np.concatenate([i.values for i in idx_parts]))
    T = len(time_index)
    bins_per_day = len(idx_parts[0])

    ti = time_index.get_indexer(pd.DatetimeIndex(flow["time_bin"]))
    si = pd.Index(station_ids).get_indexer(flow["station_id"].to_numpy())
    ok = (ti >= 0) & (si >= 0)
    ti, si = ti[ok], si[ok]
    di = flow["direction"].to_numpy(dtype=np.int64)[ok]
    ct_lut = np.full(int(flow["card_type"].max()) + 2, C - 1, dtype=np.int64)
    for _c, _i in ct_pos.items():
        ct_lut[_c] = _i
    ci = ct_lut[flow["card_type"].to_numpy(dtype=np.int64)[ok]]
    cnt = flow["count"].to_numpy(dtype=np.int32)[ok]

    X = np.zeros((T, N, 2, C), dtype=np.int32)
    np.add.at(X, (ti, si, di, ci), cnt)
    X_total = X.sum(axis=3).astype(np.int32)

    XFER = np.zeros((T, N), dtype=np.int32)
    if len(xfer):
        xti = time_index.get_indexer(pd.DatetimeIndex(xfer["time_bin"]))
        xsi = pd.Index(station_ids).get_indexer(xfer["station_id"].to_numpy())
        xok = (xti >= 0) & (xsi >= 0)
        if xok.any():
            np.add.at(XFER, (xti[xok], xsi[xok]),
                      xfer["transfers"].to_numpy(dtype=np.int64)[xok])

    if X.max() < np.iinfo(np.int16).max:
        X = X.astype(np.int16)

    out = {
        "X_card": X, "X_total": X_total, "XFER": XFER,
        "time_index": time_index, "station_ids": station_ids,
        "card_channels": np.array(card_channels, dtype=np.int32),
        "card_volume": ct_vol.to_dict(),
        "card_coverage": coverage,
        "bins_per_day": bins_per_day,
        "days": [d.strftime("%Y-%m-%d") for d in days],
        "day_index": np.repeat(np.arange(len(days)), bins_per_day),
        "dataset": CFG.dataset,
    }
    del flow, X
    gc.collect()
    return out


TENS = assemble_tensors()

X_card = TENS["X_card"]
X_total = TENS["X_total"]
XFER = TENS["XFER"]
TIME_INDEX = TENS["time_index"]
STATION_IDS = TENS["station_ids"]
CARD_CHANNELS = TENS["card_channels"]
N_STATIONS = len(STATION_IDS)
N_CARD = len(CARD_CHANNELS)
T_STEPS = len(TIME_INDEX)
CH_IN, CH_OUT = 0, 1
CHANNEL_NAMES = ("Entry", "Exit")

_ent = int(X_total[:, :, CH_IN].sum())
_ext = int(X_total[:, :, CH_OUT].sum())

print(f"╔═ TENSORS [{CFG.dataset}] ══════════════════════════════════════")
print(f"║ X_card  : {X_card.shape}  {X_card.dtype}  "
      f"({X_card.nbytes / 1e6:.0f} MB)   [T, N, in/out, card]")
print(f"║ X_total : {X_total.shape}  → {X_total.sum():,} directional passenger events")
print(f"║   entry : {_ent:,}   exit : {_ext:,}   "
      f"(imbalance {100 * abs(_ent - _ext) / max(_ent + _ext, 1):.3f}%)")
print(f"║ Stations: {N_STATIONS}   (excluded: {list(CFG.exclude_station_ids)})")
print(f"║ Days    : {len(TENS['days'])}   Bins/day: {TENS['bins_per_day']}   T = {T_STEPS}")
print(f"║ Window  : {TIME_INDEX[0]}  →  {TIME_INDEX[-1]}")
print(f"║ Card ch : {N_CARD} ({CFG.n_card_channels} explicit + OTHER), "
      f"coverage {TENS['card_coverage'] * 100:.3f}% of volume")
print(f"║ Transfers: {XFER.sum():,} type-4 events "
      f"({100 * XFER.sum() / max(X_total.sum(), 1):.3f}% of all flow)")
print(f"║ Sparsity : {100 * (X_total == 0).mean():.1f}% of (time, station, dir) cells are zero")
print("╚" + "═" * 62)


07:47:23 │ INFO    │ ✓ cached  [tensors] loaded in 5.1s  (tensors__2023__084ad33c34.pkl)
╔═ TENSORS [2023] ══════════════════════════════════════
║ X_card  : (6156, 191, 2, 25)  int16  (118 MB)   [T, N, in/out, card]
║ X_total : (6156, 191, 2)  → 190,911,927 directional passenger events
║   entry : 95,274,543   exit : 95,637,384   (imbalance 0.190%)
║ Stations: 191   (excluded: [333])
║ Days    : 57   Bins/day: 108   T = 6156
║ Window  : 2023-03-01 06:00:00  →  2023-05-31 23:50:00
║ Card ch : 25 (24 explicit + OTHER), coverage 99.584% of volume
║ Transfers: 181,549 type-4 events (0.095% of all flow)
║ Sparsity : 6.5% of (time, station, dir) cells are zero
╚══════════════════════════════════════════════════════════════


## 9 · Quality control

In [12]:
qc = {}
BINS_PER_DAY = TENS["bins_per_day"]
N_DAYS = len(TENS["days"])
DAY_DATES = pd.to_datetime(TENS["days"])
IS_WEEKEND_DAY = DAY_DATES.dayofweek >= 5
HOUR_OF_BIN = (TIME_INDEX.hour + TIME_INDEX.minute / 60.0).to_numpy(dtype=np.float32)

daily_total = X_total.reshape(N_DAYS, BINS_PER_DAY, N_STATIONS, 2).sum(axis=(1, 2, 3))
station_total = X_total.sum(axis=(0, 2))
zero_stations = STATION_IDS[station_total == 0]

qc["n_days"] = N_DAYS
qc["n_stations"] = int(N_STATIONS)
qc["total_events"] = int(X_total.sum())
qc["entry_events"] = int(X_total[:, :, CH_IN].sum())
qc["exit_events"] = int(X_total[:, :, CH_OUT].sum())
qc["mean_daily_events"] = float(daily_total.mean())
qc["std_daily_events"] = float(daily_total.std())
qc["zero_flow_stations"] = zero_stations.tolist()
qc["dead_bin_fraction"] = float((X_total.sum(axis=(1, 2)) == 0).mean())
qc["card_channels"] = CARD_CHANNELS.tolist()
qc["card_coverage"] = TENS["card_coverage"]
qc["transfer_share_pct"] = float(100 * XFER.sum() / max(X_total.sum(), 1))
save_json(qc, f"{P['cache']}/qc_{CFG.dataset}.json")

print(f"Daily ridership   : mean {daily_total.mean():,.0f}  ±{daily_total.std():,.0f}  "
      f"(min {daily_total.min():,.0f} / max {daily_total.max():,.0f})")
print(f"Fully-empty bins  : {qc['dead_bin_fraction'] * 100:.2f}%  (service-hour filter working)")
if len(zero_stations):
    print(f"⚠️  Stations with zero flow (kept but flagged): {zero_stations.tolist()}")
else:
    print("✓ Every station in the index carries traffic.")

# Reporting stations for the prediction-trace figures: busiest, mid-volume, quiet
_order = np.argsort(-station_total)
CFG.report_stations = tuple(int(STATION_IDS[i]) for i in
                            [_order[0], _order[1], _order[2],
                             _order[len(_order) // 3], _order[len(_order) // 2],
                             _order[-max(1, len(_order) // 10)]])
print(f"Reporting stations (busiest → quietest): {list(CFG.report_stations)}")
state_set("tensors_ready", True)
mem_report("After tensors:")


Daily ridership   : mean 3,349,332  ±361,674  (min 2,432,670 / max 4,362,747)
Fully-empty bins  : 0.00%  (service-hour filter working)
✓ Every station in the index carries traffic.
Reporting stations (busiest → quietest): [9, 44, 14, 39, 77, 152]
07:47:23 │ INFO    │ After tensors: RAM 0.92 GB │ GPU 0.00/0.00 GB


## 10 · Ticket-code behavioural signatures

In [13]:
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

BIN_OF_DAY = np.tile(np.arange(BINS_PER_DAY), N_DAYS)


@stage("card_signatures", depends_on=["dataset", "n_card_channels", "freq_minutes"], ext="pkl",
       subdir="personas")
def compute_card_signatures() -> dict:
    Xc = X_card.sum(axis=2, dtype=np.float64)          # [T, N, C] — in + out
    total_by_card = Xc.sum(axis=(0, 1))                 # [C]

    prof = Xc.sum(axis=1).reshape(N_DAYS, BINS_PER_DAY, N_CARD)   # [D, B, C]
    profile = prof.mean(axis=0)                                    # [B, C]
    profile_n = profile / np.maximum(profile.sum(axis=0, keepdims=True), 1e-9)

    wd = prof[~IS_WEEKEND_DAY].mean(axis=0) if (~IS_WEEKEND_DAY).any() else np.zeros_like(profile)
    we = prof[IS_WEEKEND_DAY].mean(axis=0) if IS_WEEKEND_DAY.any() else np.zeros_like(profile)
    weekend_ratio = we.sum(axis=0) / np.maximum(wd.sum(axis=0), 1e-9)

    hb = HOUR_OF_BIN[:BINS_PER_DAY]
    am = (hb >= 7) & (hb < 9.5)
    pm = (hb >= 17) & (hb < 19.5)
    mid = (hb >= 10) & (hb < 16)
    late = hb >= 21

    am_share = profile_n[am].sum(axis=0)
    pm_share = profile_n[pm].sum(axis=0)
    mid_share = profile_n[mid].sum(axis=0)
    late_share = profile_n[late].sum(axis=0)
    peak_conc = am_share + pm_share
    ampm_asym = (am_share - pm_share) / np.maximum(am_share + pm_share, 1e-9)

    st_share = Xc.sum(axis=0)                                       # [N, C]
    st_share = st_share / np.maximum(st_share.sum(axis=0, keepdims=True), 1e-9)
    entropy = -(st_share * np.log(st_share + 1e-12)).sum(axis=0) / np.log(max(N_STATIONS, 2))
    hhi = (st_share ** 2).sum(axis=0)

    io = X_card.sum(axis=(0, 1), dtype=np.float64)                  # [2, C]
    io_balance = (io[0] - io[1]) / np.maximum(io[0] + io[1], 1e-9)
    daily = prof.sum(axis=1)                                        # [D, C]
    burst = daily.std(axis=0) / np.maximum(daily.mean(axis=0), 1e-9)

    feats = pd.DataFrame({
        "card_channel": CARD_CHANNELS,
        "volume": total_by_card,
        "share_pct": 100 * total_by_card / max(total_by_card.sum(), 1e-9),
        "weekend_ratio": weekend_ratio,
        "am_share": am_share, "pm_share": pm_share, "mid_share": mid_share,
        "late_share": late_share, "peak_concentration": peak_conc,
        "ampm_asymmetry": ampm_asym, "spatial_entropy": entropy,
        "station_hhi": hhi, "io_balance": io_balance, "daily_burstiness": burst,
    })
    feats["label"] = feats["card_channel"].map(lambda c: "OTHER" if c == -1 else f"card_{int(c)}")
    return {"features": feats, "profile": profile, "profile_norm": profile_n,
            "profile_weekday": wd, "profile_weekend": we, "station_share": st_share}


SIG = compute_card_signatures()
CARD_FEATS = SIG["features"]
print(CARD_FEATS[["label", "share_pct", "weekend_ratio", "peak_concentration",
                  "mid_share", "spatial_entropy", "daily_burstiness"]]
      .sort_values("share_pct", ascending=False).round(3).to_string(index=False))
save_df(CARD_FEATS, f"{P['personas']}/card_signatures_{CFG.dataset}.csv")


07:47:25 │ INFO    │ ✓ cached  [card_signatures] loaded in 0.8s  (card_signatures__2023__b08eb4f084.pkl)
   label  share_pct  weekend_ratio  peak_concentration  mid_share  spatial_entropy  daily_burstiness
card_121     45.492          1.252               0.396      0.331            0.869             0.208
card_100     19.607          0.696               0.575      0.200            0.896             0.208
 card_53      5.687          0.671               0.559      0.228            0.900             0.218
 card_51      5.080          1.274               0.298      0.459            0.789             0.410
 card_52      4.591          0.685               0.555      0.227            0.908             0.212
card_113      2.241          0.900               0.357      0.399            0.907             0.104
card_108      2.032          0.738               0.533      0.238            0.913             0.174
 card_55      1.802          0.851               0.333      0.457            0.897     

## 11 · Persona discovery

In [14]:
PERSONA_FEATURES = ["weekend_ratio", "am_share", "pm_share", "mid_share", "late_share",
                    "peak_concentration", "ampm_asymmetry", "spatial_entropy",
                    "station_hhi", "io_balance", "daily_burstiness"]


@stage("persona_prior", depends_on=["dataset", "n_personas", "n_card_channels", "seed"],
       ext="pkl", subdir="personas")
def discover_personas() -> dict:
    K = CFG.n_personas
    F = CARD_FEATS[PERSONA_FEATURES].to_numpy(dtype=np.float64)
    F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)
    scaler = StandardScaler().fit(F)
    Z = scaler.transform(F)

    # volume-weighted k-means: a code carrying 20% of ridership should not be
    # out-voted by a code with 4 records
    w = CARD_FEATS["volume"].to_numpy(dtype=np.float64)
    w = np.maximum(w, 1.0) ** 0.5
    km = KMeans(n_clusters=K, n_init=25, random_state=CFG.seed).fit(Z, sample_weight=w)
    lab = km.labels_
    cent = km.cluster_centers_

    # ── name the clusters by interpretable criteria (Hungarian assignment) ───
    cdf = pd.DataFrame(cent, columns=PERSONA_FEATURES)
    score = np.zeros((K, K))
    z = lambda col: (cdf[col] - cdf[col].mean()) / (cdf[col].std() + 1e-9)
    score[:, 0] = 2.0 * z("peak_concentration") - 1.5 * z("weekend_ratio") - 0.5 * z("mid_share")
    score[:, 1] = 1.0 * z("daily_burstiness") + 0.5 * z("spatial_entropy") - 0.5 * abs(z("ampm_asymmetry"))
    score[:, 2] = 2.0 * z("weekend_ratio") + 1.0 * z("mid_share") + 1.0 * z("station_hhi") - 1.0 * z("peak_concentration")
    score[:, 3] = 1.5 * z("ampm_asymmetry") + 1.0 * z("late_share") - 0.5 * z("station_hhi")
    r, c = linear_sum_assignment(-score)
    cluster_to_persona = {int(ri): int(ci) for ri, ci in zip(r, c)}
    persona_of_card = np.array([cluster_to_persona[int(l)] for l in lab])

    d = np.linalg.norm(Z[:, None, :] - cent[None, :, :], axis=2)     # [C, K] cluster order
    tau = max(float(np.median(d)) * 0.5, 1e-3)
    soft_cluster = np.exp(-d / tau)
    soft_cluster /= soft_cluster.sum(axis=1, keepdims=True)
    prior = np.zeros_like(soft_cluster)
    for ci_, pi_ in cluster_to_persona.items():
        prior[:, pi_] = soft_cluster[:, ci_]
    prior = prior / prior.sum(axis=1, keepdims=True)

    assign = CARD_FEATS[["card_channel", "label", "volume", "share_pct"]].copy()
    assign["persona_idx"] = persona_of_card
    assign["persona"] = [CFG.persona_names[i] for i in persona_of_card]
    assign["confidence"] = prior.max(axis=1)
    for k in range(K):
        assign[f"p_{CFG.persona_names[k].split('/')[0]}"] = prior[:, k]

    share = np.array([assign.loc[assign["persona_idx"] == k, "volume"].sum() for k in range(K)])
    share = 100 * share / max(share.sum(), 1e-9)

    return {"prior": prior, "assign": assign, "kmeans": km, "scaler": scaler,
            "cluster_to_persona": cluster_to_persona, "Z": Z, "centroids": cent,
            "volume_share_pct": share, "tau": tau}


PERSONA = discover_personas()
PERSONA_PRIOR = PERSONA["prior"]                 # [C, K]
PERSONA_ASSIGN = PERSONA["assign"]
K_PERSONA = CFG.n_personas

print("╔═ Persona prior P₀ (frozen from Paper 3) ════════════════════════════")
for k in range(K_PERSONA):
    sub = PERSONA_ASSIGN[PERSONA_ASSIGN["persona_idx"] == k]
    codes = ", ".join(sub.sort_values("share_pct", ascending=False)["label"].head(8))
    print(f"║ {k}  {CFG.persona_names[k]:<20s} {PERSONA['volume_share_pct'][k]:5.1f}% of ridership "
          f"│ {len(sub)} codes")
    print(f"║    ↳ {codes}{' …' if len(sub) > 8 else ''}")
print("╚" + "═" * 68)
print(f"Mean soft-assignment confidence: {PERSONA_ASSIGN['confidence'].mean():.3f} (1.0 = fully hard)")
save_df(PERSONA_ASSIGN, f"{P['personas']}/persona_assignment_{CFG.dataset}.csv")

# ── persona-resolved flow tensor (analysis + visuals) ────────────────────────
X_persona = np.einsum("tndc,ck->tndk", X_card, PERSONA_PRIOR.astype(np.float32),
                      optimize=True).astype(np.float32)
print(f"X_persona: {X_persona.shape}  [T, N, in/out, K]  ({X_persona.nbytes / 1e6:.0f} MB)")

# ── Per-step persona mix vector π_t ∈ ℝ^K ───
_pv = X_persona.sum(axis=(1, 2))                                    # [T, K]
PI_T = (_pv / np.maximum(_pv.sum(axis=1, keepdims=True), 1e-9)).astype(np.float32)
PI_PRIOR = PI_T.mean(axis=0)                                        # global mix, for the regulariser

print(f"\nπ_t (persona mix): {PI_T.shape}  [T, K]")
print("  network-wide mean mix : " +
      "  ".join(f"{CFG.persona_names[k].split('/')[0]} {100 * PI_PRIOR[k]:.1f}%"
                for k in range(K_PERSONA)))
_pk = (HOUR_OF_BIN >= 7) & (HOUR_OF_BIN < 9.5)
_off = (HOUR_OF_BIN >= 11) & (HOUR_OF_BIN < 15)
print("  AM-peak mix           : " +
      "  ".join(f"{CFG.persona_names[k].split('/')[0]} {100 * PI_T[_pk, k].mean():.1f}%"
                for k in range(K_PERSONA)))
print("  midday mix            : " +
      "  ".join(f"{CFG.persona_names[k].split('/')[0]} {100 * PI_T[_off, k].mean():.1f}%"
                for k in range(K_PERSONA)))
print(f"  max |Δ mix| across the day: "
      f"{100 * np.abs(PI_T[_pk].mean(0) - PI_T[_off].mean(0)).max():.2f} pp "
      f"→ the signal FiLM has to work with.")


07:47:25 │ INFO    │ ✓ cached  [persona_prior] loaded in 0.3s  (persona_prior__2023__4d0870a6c1.pkl)
╔═ Persona prior P₀ (frozen from Paper 3) ════════════════════════════
║ 0  Commuter              36.6% of ridership │ 10 codes
║    ↳ card_100, card_53, card_52, card_108, card_80, card_120, card_102, card_119 …
║ 1  Casual                49.5% of ridership │ 5 codes
║    ↳ card_121, card_103, card_123, card_101, card_54
║ 2  Tourist/Visitor        5.7% of ridership │ 2 codes
║    ↳ card_51, card_118
║ 3  Concession/Student     8.2% of ridership │ 8 codes
║    ↳ card_113, card_55, card_85, card_104, card_56, card_59, card_83, card_111
╚════════════════════════════════════════════════════════════════════
Mean soft-assignment confidence: 0.652 (1.0 = fully hard)
X_persona: (6156, 191, 2, 4)  [T, N, in/out, K]  (38 MB)

π_t (persona mix): (6156, 4)  [T, K]
  network-wide mean mix : Commuter 31.5%  Casual 42.2%  Tourist 11.5%  Concession 14.8%
  AM-peak mix           : Commuter 39.1%  Casu

## 12 · Spatial probe — line map, OD travel times, transfers

In [15]:
@stage("spatial_probe", depends_on=["dataset", "exclude_station_ids"], ext="pkl", subdir="graphs")
def spatial_probe(max_files: int = 4, max_rows: int = 4_000_000) -> dict:
    sel = MANIFEST[MANIFEST["year_from_content"] == CFG.dataset].head(max_files)
    line_counts: Dict[tuple, int] = {}
    tt_sum: Dict[tuple, float] = {}
    tt_cnt: Dict[tuple, int] = {}
    tt_all: List[float] = []
    od_vol: Dict[tuple, int] = {}
    xfer_pairs: Dict[tuple, int] = {}

    for _, r in tqdm(list(sel.iterrows()), total=len(sel), desc="Spatial probe"):
        got = 0
        reader = read_afc(r["load_path"],
                          usecols=["entry_time", "transaction_time", "transaction_type",
                                   "entry_station_id", "line_id", "transaction_station_id"],
                          chunksize=CFG.chunksize)
        if isinstance(reader, pd.DataFrame):
            reader = [reader]
        for ch in reader:
            if got >= max_rows:
                break
            ch = ch.head(max_rows - got)
            got += len(ch)
            for c in ["transaction_type", "entry_station_id", "line_id", "transaction_station_id"]:
                ch[c] = pd.to_numeric(ch[c], errors="coerce")
            ch = ch.dropna(subset=["entry_station_id", "transaction_station_id", "line_id"])
            ti = to_datetime_safe(ch["entry_time"])
            to_ = to_datetime_safe(ch["transaction_time"])
            ok = ti.notna() & to_.notna() & (ti.dt.year > 2000)
            ch, ti, to_ = ch[ok], ti[ok], to_[ok]
            if ch.empty:
                continue
            o = ch["entry_station_id"].astype(int).to_numpy()
            d = ch["transaction_station_id"].astype(int).to_numpy()
            ln = ch["line_id"].astype(int).to_numpy()
            tt = (to_ - ti).dt.total_seconds().to_numpy() / 60.0
            tp = ch["transaction_type"].fillna(2).astype(int).to_numpy()

            for s, l in zip(d, ln):
                line_counts[(int(s), int(l))] = line_counts.get((int(s), int(l)), 0) + 1

            valid = (tt > 0.5) & (tt < 180) & (o != d) & (tp == 2)
            tt_all.extend(tt[valid].tolist()[:200_000])
            for oo, dd, t in zip(o[valid], d[valid], tt[valid]):
                key = (int(min(oo, dd)), int(max(oo, dd)))
                tt_sum[key] = tt_sum.get(key, 0.0) + float(t)
                tt_cnt[key] = tt_cnt.get(key, 0) + 1
                od_vol[key] = od_vol.get(key, 0) + 1

            t4 = tp == 4
            for oo, dd in zip(o[t4], d[t4]):
                key = (int(min(oo, dd)), int(max(oo, dd)))
                xfer_pairs[key] = xfer_pairs.get(key, 0) + 1
            del ch, ti, to_
            gc.collect()

    st_line = (pd.DataFrame([{"station_id": s, "line_id": l, "n": n}
                             for (s, l), n in line_counts.items()])
               if line_counts else pd.DataFrame(columns=["station_id", "line_id", "n"]))
    tt_df = pd.DataFrame([{"a": a, "b": b, "median_min": tt_sum[(a, b)] / tt_cnt[(a, b)],
                           "n_trips": tt_cnt[(a, b)]} for (a, b) in tt_cnt])
    od_df = pd.DataFrame([{"a": a, "b": b, "volume": v} for (a, b), v in od_vol.items()])
    xf_df = pd.DataFrame([{"a": a, "b": b, "transfers": v} for (a, b), v in xfer_pairs.items()])
    return {"station_line": st_line, "travel_time": tt_df, "od_volume": od_df,
            "transfer_pairs": xf_df,
            "journey_minutes": np.array(tt_all[:500_000], dtype=np.float32)}


PROBE = spatial_probe()
_sl = PROBE["station_line"]
if len(_sl):
    STATION_LINE = (_sl.sort_values("n", ascending=False)
                    .drop_duplicates("station_id")[["station_id", "line_id"]]
                    .set_index("station_id")["line_id"].to_dict())
    LINES_OF_STATION = _sl.groupby("station_id")["line_id"].apply(lambda s: sorted(set(s))).to_dict()
else:
    STATION_LINE, LINES_OF_STATION = {}, {}
INTERCHANGE = {s: ls for s, ls in LINES_OF_STATION.items() if len(ls) > 1}
print(f"Stations mapped to a primary line : {len(STATION_LINE)}")
print(f"Lines present                     : {sorted(set(STATION_LINE.values()))}")
print(f"Multi-line (interchange) stations : {len(INTERCHANGE)}")
print(f"OD pairs with travel times        : {len(PROBE['travel_time']):,}")
print(f"Type-4 transfer pairs             : {len(PROBE['transfer_pairs']):,}")

# ── Empirical entry→exit propagation over δ ∈ {0,1,2,3} ───
_jm = PROBE.get("journey_minutes", np.array([]))
if len(_jm):
    _cut = CFG.freq_minutes * max(CFG.delay_lags)   # δ ≤ 3 → 30 min, per the proposal
    print(f"\nObserved journey durations (n={len(_jm):,}):")
    for _q in [25, 50, 75, 90, 95]:
        print(f"  p{_q:<3d} = {np.percentile(_jm, _q):6.1f} min")
    print(f"  → {100 * (_jm <= _cut).mean():.1f}% of journeys complete within "
          f"{_cut} min, i.e. inside δ ≤ {max(CFG.delay_lags)} steps.")
    print(f"    This is the empirical basis for the proposal's kernel support; the "
          f"δ-order ablation tests whether it is the right cut-off.")


07:47:26 │ INFO    │ ✓ cached  [spatial_probe] loaded in 0.7s  (spatial_probe__2023__ec43276801.pkl)
Stations mapped to a primary line : 190
Lines present                     : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Multi-line (interchange) stations : 13
OD pairs with travel times        : 16,966
Type-4 transfer pairs             : 4,640

Observed journey durations (n=500,000):
  p25  =   19.3 min
  p50  =   30.3 min
  p75  =   45.6 min
  p90  =   65.2 min
  p95  =   81.1 min
  → 49.5% of journeys complete within 30 min, i.e. inside δ ≤ 3 steps.
    This is the empirical basis for the proposal's kernel support; the δ-order ablation tests whether it is the right cut-off.


## 13 · Physical topology $A_{topo}$

In [16]:
ST_POS = {int(s): i for i, s in enumerate(STATION_IDS)}


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = p2 - p1
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def load_station_metadata() -> pd.DataFrame:
    md = read_meta("station_metadata.csv",
                   ["station_id", "station_name_cn", "station_name_en", "latitude",
                    "longitude", "line_primary", "is_interchange", "station_class"])
    if len(md) == 0:
        return md
    md["station_id"] = pd.to_numeric(md["station_id"], errors="coerce")
    return md.dropna(subset=["station_id"])


STATION_META = load_station_metadata()
HAS_COORDS = (len(STATION_META) > 0 and
              STATION_META[["latitude", "longitude"]].notna().all(axis=1).sum() >= 0.8 * N_STATIONS)


@stage("A_topo", depends_on=["dataset", "adj_topk", "exclude_station_ids"], ext="pkl", subdir="graphs")
def build_topology() -> dict:
    N = N_STATIONS
    A = np.zeros((N, N), dtype=np.float32)
    source = "unknown"

    if HAS_COORDS:
        md = STATION_META.set_index("station_id")
        lat = np.array([md["latitude"].get(int(s), np.nan) for s in STATION_IDS], dtype=float)
        lon = np.array([md["longitude"].get(int(s), np.nan) for s in STATION_IDS], dtype=float)
        D = haversine_km(lat[:, None], lon[:, None], lat[None, :], lon[None, :])
        D = np.nan_to_num(D, nan=999.0)
        for i in range(N):
            order = np.argsort(D[i])
            for j in order[1:3]:                     # 2 nearest = line neighbours
                A[i, j] = A[j, i] = 1.0
        source = "geodesic (station_metadata.csv coordinates)"
    else:
        tt = PROBE["travel_time"]
        if len(tt):
            tt = tt[tt["n_trips"] >= 30]
            same_line = tt.apply(
                lambda r: (STATION_LINE.get(int(r["a"])) is not None and
                           STATION_LINE.get(int(r["a"])) == STATION_LINE.get(int(r["b"]))),
                axis=1) if len(tt) else pd.Series(dtype=bool)
            cand = tt[same_line] if same_line.any() else tt
            for i_st in STATION_IDS:
                sub = cand[(cand["a"] == i_st) | (cand["b"] == i_st)]
                if sub.empty:
                    continue
                sub = sub.nsmallest(2, "median_min")
                for _, r in sub.iterrows():
                    a, b = int(r["a"]), int(r["b"])
                    if a in ST_POS and b in ST_POS:
                        A[ST_POS[a], ST_POS[b]] = A[ST_POS[b], ST_POS[a]] = 1.0
            source = "inferred from median observed travel time (same line, ≥30 trips)"

    xf = PROBE["transfer_pairs"]
    n_xf_edges = 0
    if len(xf):
        thr = xf["transfers"].quantile(0.995)
        for _, r in xf[xf["transfers"] >= thr].iterrows():
            a, b = int(r["a"]), int(r["b"])
            if a in ST_POS and b in ST_POS and A[ST_POS[a], ST_POS[b]] == 0:
                A[ST_POS[a], ST_POS[b]] = A[ST_POS[b], ST_POS[a]] = 1.0
                n_xf_edges += 1

    isolated = np.where(A.sum(axis=1) == 0)[0]
    od = PROBE["od_volume"]
    if len(isolated) and len(od):
        for i in isolated:
            sid = int(STATION_IDS[i])
            sub = od[(od["a"] == sid) | (od["b"] == sid)].nlargest(2, "volume")
            for _, r in sub.iterrows():
                a, b = int(r["a"]), int(r["b"])
                if a in ST_POS and b in ST_POS:
                    A[ST_POS[a], ST_POS[b]] = A[ST_POS[b], ST_POS[a]] = 1.0

    deg = A.sum(axis=1)
    return {"A": A, "source": source, "n_edges": int(A.sum() // 2),
            "n_transfer_edges": n_xf_edges, "isolated": int((deg == 0).sum()),
            "mean_degree": float(deg.mean())}


TOPO = build_topology()
A_TOPO = TOPO["A"]
print(f"A_topo source        : {TOPO['source']}")
print(f"Edges                : {TOPO['n_edges']}  (+{TOPO['n_transfer_edges']} interchange)")
print(f"Mean degree          : {TOPO['mean_degree']:.2f}   isolated nodes: {TOPO['isolated']}")


07:47:27 │ INFO    │ ✓ cached  [A_topo] loaded in 0.4s  (A_topo__2023__d6587516ea.pkl)
A_topo source        : geodesic (station_metadata.csv coordinates)
Edges                : 266  (+24 interchange)
Mean degree          : 2.79   isolated nodes: 0


## 14 · Correlation topology and sparsity mask

In [17]:
def _sparsify(C: np.ndarray, topk: int, thresh: float) -> np.ndarray:
    A = np.where(np.abs(C) >= thresh, C, 0.0).astype(np.float32)
    np.fill_diagonal(A, 0.0)
    if topk and topk < A.shape[0]:
        keep = np.zeros_like(A, dtype=bool)
        idx = np.argsort(-np.abs(A), axis=1)[:, :topk]
        np.put_along_axis(keep, idx, True, axis=1)
        A = np.where(keep | keep.T, A, 0.0)
    return np.maximum(A, A.T)


def _corr_matrix(S: np.ndarray) -> np.ndarray:
    """S: [T, N] → Pearson correlation across stations, NaN-safe."""
    Sd = S - S.mean(axis=0, keepdims=True)
    sd = Sd.std(axis=0, keepdims=True)
    Sd = Sd / np.maximum(sd, 1e-8)
    C = (Sd.T @ Sd) / max(S.shape[0] - 1, 1)
    return np.nan_to_num(C, nan=0.0)


@stage("graph_mask", depends_on=["dataset", "n_personas", "adj_topk", "corr_min",
                                 "train_frac"],           # ← hash changes → recompute
       ext="pkl", subdir="graphs")
def build_graph_mask() -> dict:
    # v2: TRAIN DAYS ONLY. The mask is a fitted statistic — building it on all
    # 57 days leaks the 9 test days into every graph the model ever sees.
    _d_tr = int(round(CFG.train_frac * len(TENS["days"])))
    _tr_end = int((TENS["day_index"] < _d_tr).sum())
    Xtr = X_total[:_tr_end]

    # aggregate (context-agnostic) reference graph — the mask source
    C_all = _corr_matrix(Xtr.sum(axis=2).astype(np.float64))
    A_all = _sparsify(C_all, CFG.adj_topk, CFG.corr_min)

    # directional correlation views (train-only)
    C_o = _corr_matrix(Xtr[:, :, CH_IN].astype(np.float64))
    C_d = _corr_matrix(Xtr[:, :, CH_OUT].astype(np.float64))

    # persona correlation views — descriptive figure only (train-only)
    A_k, C_k = [], []
    for k in range(K_PERSONA):
        S = X_persona[:_tr_end, :, :, k].sum(axis=2)          # [T_tr, N]
        C = _corr_matrix(S.astype(np.float64))
        C_k.append(C.astype(np.float32))
        A_k.append(_sparsify(C, CFG.adj_topk, CFG.corr_min))
    A_k = np.stack(A_k)
    C_k = np.stack(C_k)

    def cos(a, b):
        a, b = a.flatten(), b.flatten()
        return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

    pairs = []
    for i in range(K_PERSONA):
        for j in range(i + 1, K_PERSONA):
            pairs.append({"a": CFG.persona_names[i], "b": CFG.persona_names[j],
                          "cosine": cos(C_k[i], C_k[j]),
                          "mean_abs_diff": float(np.abs(C_k[i] - C_k[j]).mean()),
                          "frac_gt_0.2": float((np.abs(C_k[i] - C_k[j]) > 0.2).mean())})
    pair_df = pd.DataFrame(pairs)

    # ── the binary mask M: union of correlation edges and physical topology ──
    M = ((np.abs(A_all) > 0) | (A_TOPO > 0)).astype(np.float32)
    np.fill_diagonal(M, 0.0)

    return {"C_all": C_all.astype(np.float32), "A_all": A_all, "M": M,
            "C_origin": C_o.astype(np.float32), "C_dest": C_d.astype(np.float32),
            "od_cosine": cos(C_o, C_d), "A_k": A_k, "C_k": C_k, "pairs": pair_df}


GRAPH = build_graph_mask()
A_ALL = GRAPH["A_all"]
MASK_M = GRAPH["M"]
A_PERSONA = GRAPH["A_k"]

_deg = MASK_M.sum(axis=1)
print("╔═ SPARSITY MASK  M ══════════════════════════════════════════════")
print(f"║ rule            : |corr| ≥ {CFG.corr_min}, top-k {CFG.adj_topk}, ∪ A_topo")
print(f"║ live edges      : {int(MASK_M.sum() // 2):,} of {N_STATIONS * (N_STATIONS - 1) // 2:,} "
      f"possible ({100 * MASK_M.sum() / (N_STATIONS ** 2 - N_STATIONS):.2f}% density)")
print(f"║ degree          : mean {_deg.mean():.2f}   min {int(_deg.min())}   max {int(_deg.max())}")
print(f"║ isolated nodes  : {int((_deg == 0).sum())}  (self-loops keep every softmax row valid)")
print(f"║ cost            : O({int(_deg.mean())}·N) per regenerated graph, not O(N²)")
print("╠═ inherited evidence (Paper 3, reproduced for the record) ════════════")
print(f"║ origin-graph vs destination-graph cosine : {GRAPH['od_cosine']:.4f}")
print(f"║ mean pairwise persona-graph cosine       : {GRAPH['pairs']['cosine'].mean():.4f}")
print("║ → persona views are NOT more distinct than the O/D split, which is why")
print("║   CRISP-Net feeds persona-resolved flow channels (PRI) directly rather")
print("║   than splitting the graph on them.")
print("╚" + "═" * 68)
save_df(GRAPH["pairs"], f"{P['graphs']}/persona_graph_distinctness_{CFG.dataset}.csv")


07:47:27 │ INFO    │ ✓ cached  [graph_mask] loaded in 0.5s  (graph_mask__2023__a797568561.pkl)
╔═ SPARSITY MASK  M ══════════════════════════════════════════════
║ rule            : |corr| ≥ 0.3, top-k 8, ∪ A_topo
║ live edges      : 1,444 of 18,145 possible (7.96% density)
║ degree          : mean 15.12   min 2   max 39
║ isolated nodes  : 0  (self-loops keep every softmax row valid)
║ cost            : O(15·N) per regenerated graph, not O(N²)
╠═ inherited evidence (Paper 3, reproduced for the record) ════════════
║ origin-graph vs destination-graph cosine : 0.9415
║ mean pairwise persona-graph cosine       : 0.9850
║ → persona views are NOT more distinct than the O/D split, which is why
║   CRISP-Net feeds persona-resolved flow channels (PRI) directly rather
║   than splitting the graph on them.
╚════════════════════════════════════════════════════════════════════


## 15 · Empirical entry→exit propagation

In [18]:
@stage("delay_support", depends_on=["dataset", "freq_minutes", "delay_lags",
                                    "adj_topk", "corr_min", "train_frac"],
       ext="pkl", subdir="graphs")
def build_delay_support() -> dict:
    # Train days only — per-lag delay adjacency
    # embeddings, so they are fitted statistics and must not see the test period.
    _d_tr = int(round(CFG.train_frac * len(TENS["days"])))
    _tr_end = int((TENS["day_index"] < _d_tr).sum())
    ent = X_total[:_tr_end, :, CH_IN].astype(np.float64)      # [T_tr, N]
    ext = X_total[:_tr_end, :, CH_OUT].astype(np.float64)     # [T_tr, N]
    seg = TENS["day_index"][:_tr_end]

    max_probe = 7
    net_in = ent.sum(axis=1)
    net_out = ext.sum(axis=1)

    net_rows = []
    for d in range(max_probe):
        if d == 0:
            m = np.ones(len(seg), dtype=bool)
            a, b = net_in, net_out
        else:
            m = seg[:-d] == seg[d:]
            a, b = net_in[:-d][m], net_out[d:][m]
        if len(a) < 10:
            continue
        r = float(np.corrcoef(a, b)[0, 1])
        net_rows.append({"lag_steps": d, "lag_minutes": d * CFG.freq_minutes,
                         "corr": r, "n": int(len(a))})
    net_df = pd.DataFrame(net_rows)

    def _lagged_corr(d):
        if d == 0:
            A, B = ent, ext
        else:
            m = seg[:-d] == seg[d:]
            A, B = ent[:-d][m], ext[d:][m]
        Az = (A - A.mean(0)) / np.maximum(A.std(0), 1e-8)
        Bz = (B - B.mean(0)) / np.maximum(B.std(0), 1e-8)
        C = (Az.T @ Bz) / max(len(Az) - 1, 1)
        return np.nan_to_num(C, nan=0.0).astype(np.float32)

    lags = list(CFG.delay_lags)
    A_delta = np.stack([_lagged_corr(d) * MASK_M for d in lags])   # [D, N, N]
    A_delta = np.clip(A_delta, 0.0, None)

    strength = np.array([float(A_delta[i][MASK_M > 0].mean()) for i in range(len(lags))])
    share = strength / max(strength.sum(), 1e-9)

    return {"lags": lags, "A_delta": A_delta, "net_corr": net_df,
            "edge_strength": strength, "empirical_kernel": share,
            "probe_lags": list(range(max_probe))}


DELAY = build_delay_support()
A_DELTA = DELAY["A_delta"]                 # [D, N, N] — per-lag delay adjacency
DELAY_LAGS = DELAY["lags"]
N_LAGS = len(DELAY_LAGS)

print("╔═ EMPIRICAL ENTRY→EXIT PROPAGATION ════════════════════════════")
print("║ network-level lagged correlation  corr( Σentry[t], Σexit[t+δ] ):")
for _, r in DELAY["net_corr"].iterrows():
    bar = "█" * int(max(r["corr"], 0) * 40)
    print(f"║   δ={int(r['lag_steps'])}  ({int(r['lag_minutes']):>2d} min)  "
          f"r = {r['corr']:+.4f}  {bar}")
_best = DELAY["net_corr"].loc[DELAY["net_corr"]["corr"].idxmax()]
print(f"║ → peak at δ = {int(_best['lag_steps'])} "
      f"({int(_best['lag_minutes'])} min), r = {_best['corr']:.4f}")
print("║")
print(f"║ masked station-level edge strength per lag (mean over M's live edges):")
for i, d in enumerate(DELAY_LAGS):
    print(f"║   A^({d})  mean|w| = {DELAY['edge_strength'][i]:.4f}   "
          f"→ empirical kernel weight {100 * DELAY['empirical_kernel'][i]:5.1f}%")
print("║")
print("║ The kernel is NOT degenerate at δ=0: a non-trivial share of the mass sits")
print("║ at δ ≥ 1 — corpus evidence for predicting entry and exit jointly.")
print("║ whether the learned w_δ(z_t) collapses — the secondary risk the proposal names.")
print("╚" + "═" * 68)
save_df(DELAY["net_corr"], f"{P['graphs']}/delay_network_corr_{CFG.dataset}.csv")
save_npz(f"{P['graphs']}/A_delta_{CFG.dataset}.npz", A_delta=A_DELTA,
         lags=np.array(DELAY_LAGS))


07:47:28 │ INFO    │ ✓ cached  [delay_support] loaded in 0.4s  (delay_support__2023__d224307b6a.pkl)
╔═ EMPIRICAL ENTRY→EXIT PROPAGATION ════════════════════════════
║ network-level lagged correlation  corr( Σentry[t], Σexit[t+δ] ):
║   δ=0  ( 0 min)  r = +0.8083  ████████████████████████████████
║   δ=1  (10 min)  r = +0.8981  ███████████████████████████████████
║   δ=2  (20 min)  r = +0.9613  ██████████████████████████████████████
║   δ=3  (30 min)  r = +0.9856  ███████████████████████████████████████
║   δ=4  (40 min)  r = +0.9670  ██████████████████████████████████████
║   δ=5  (50 min)  r = +0.9056  ████████████████████████████████████
║   δ=6  (60 min)  r = +0.8093  ████████████████████████████████
║ → peak at δ = 3 (30 min), r = 0.9856
║
║ masked station-level edge strength per lag (mean over M's live edges):
║   A^(0)  mean|w| = 0.3237   → empirical kernel weight  22.6%
║   A^(1)  mean|w| = 0.3559   → empirical kernel weight  24.8%
║   A^(2)  mean|w| = 0.3753   → empirical kern

## 16 · Event detection and calendar merge

In [19]:
@stage("events", depends_on=["dataset", "freq_minutes", "service_start", "service_end"],
       ext="pkl", subdir="events")
def build_events() -> dict:
    F = X_total.sum(axis=2).astype(np.float32)                 # [T, N] total flow
    D = F.reshape(N_DAYS, BINS_PER_DAY, N_STATIONS)
    day_peak = D.max(axis=1)                                    # [D, N]
    day_sum = D.sum(axis=1)
    dow = DAY_DATES.dayofweek.to_numpy()

    z = np.zeros_like(day_peak)
    base = np.zeros_like(day_peak)
    for w in range(7):
        m = dow == w
        if m.sum() < 3:
            m = np.ones_like(dow, dtype=bool)
        med = np.median(day_peak[m], axis=0)
        mad = np.median(np.abs(day_peak[m] - med), axis=0)
        scale = np.maximum(1.4826 * mad, np.maximum(0.05 * med, 1.0))
        idx = np.where(dow == w)[0]
        z[idx] = (day_peak[idx] - med) / scale
        base[idx] = med

    peak_bin = D.argmax(axis=1)
    cand = []
    for di in range(N_DAYS):
        for ni in range(N_STATIONS):
            if z[di, ni] >= 4.0 and day_peak[di, ni] >= max(150.0, 1.5 * base[di, ni]):
                b = int(peak_bin[di, ni])
                cand.append({
                    "date": TENS["days"][di],
                    "station_id": int(STATION_IDS[ni]),
                    "robust_z": float(z[di, ni]),
                    "peak_flow": float(day_peak[di, ni]),
                    "baseline_peak": float(base[di, ni]),
                    "surge_ratio": float(day_peak[di, ni] / max(base[di, ni], 1.0)),
                    "peak_time": str(TIME_INDEX[di * BINS_PER_DAY + b].time()),
                    "peak_bin": b,
                    "day_total": float(day_sum[di, ni]),
                    "weekday": DAY_DATES[di].day_name(),
                })
    cand_df = pd.DataFrame(cand)
    if len(cand_df):
        cand_df = cand_df.sort_values("robust_z", ascending=False).reset_index(drop=True)

    cal = read_meta("event_calendar.csv",
                    ["date", "venue_id", "event_type", "start_time", "end_time",
                     "expected_attendance", "source", "verified"])
    curated_days = set()
    if len(cal):
        cal["date"] = pd.to_datetime(cal["date"], errors="coerce").dt.strftime("%Y-%m-%d")
        cal = cal.dropna(subset=["date"])
        curated_days = set(cal["date"])

    det_days = set(cand_df.loc[cand_df["robust_z"] >= 5.0, "date"]) if len(cand_df) else set()
    event_days = sorted(curated_days | det_days)

    is_event_day = np.array([d in set(event_days) for d in TENS["days"]], dtype=bool)
    event_flag = np.repeat(is_event_day, BINS_PER_DAY).astype(np.float32)      # [T]

    st_mask = np.zeros((N_DAYS, N_STATIONS), dtype=np.float32)
    if len(cand_df):
        for _, r in cand_df[cand_df["robust_z"] >= 4.0].iterrows():
            di = TENS["days"].index(r["date"]) if r["date"] in TENS["days"] else None
            if di is not None:
                st_mask[di, ST_POS[int(r["station_id"])]] = 1.0
    event_station_mask = np.repeat(st_mask, BINS_PER_DAY, axis=0)              # [T, N]

    countdown = np.full(len(TIME_INDEX), 36.0, dtype=np.float32)
    if len(cand_df):
        for d, grp in cand_df.groupby("date"):
            if d not in TENS["days"]:
                continue
            di = TENS["days"].index(d)
            pb = int(grp["peak_bin"].min())
            sl = slice(di * BINS_PER_DAY, (di + 1) * BINS_PER_DAY)
            countdown[sl] = np.clip(np.abs(np.arange(BINS_PER_DAY) - pb), 0, 36)
    countdown = countdown / 36.0

    return {"candidates": cand_df, "curated": cal, "event_days": event_days,
            "is_event_day": is_event_day, "event_flag": event_flag,
            "event_station_mask": event_station_mask, "countdown": countdown,
            "robust_z": z, "day_peak": day_peak, "baseline_peak": base}


EVENTS = build_events()
EVENT_FLAG = EVENTS["event_flag"]
EVENT_STATION_MASK = EVENTS["event_station_mask"]

print(f"Detected surge candidates (z ≥ 4): {len(EVENTS['candidates'])}")
print(f"Curated calendar rows            : {len(EVENTS['curated'])}")
print(f"Event days flagged               : {len(EVENTS['event_days'])} of {N_DAYS} "
      f"({100 * len(EVENTS['event_days']) / max(N_DAYS, 1):.0f}%)")
if len(EVENTS["candidates"]):
    print("\nTop 15 surge candidates:")
    print(EVENTS["candidates"].head(15)[
        ["date", "weekday", "station_id", "peak_time", "peak_flow",
         "baseline_peak", "surge_ratio", "robust_z"]].round(2).to_string(index=False))
    save_df(EVENTS["candidates"], f"{P['meta']}/DETECTED_event_candidates_{CFG.dataset}.csv")
if len(EVENTS["event_days"]) == 0:
    print("⚠️  No event days flagged — the Mondrian event bin will be empty and ECQ will "
          "fall back to a single global bin.")


07:47:28 │ INFO    │ ✓ cached  [events] loaded in 0.5s  (events__2023__759bb12f03.pkl)
Detected surge candidates (z ≥ 4): 155
Curated calendar rows            : 12
Event days flagged               : 24 of 57 (42%)

Top 15 surge candidates:
      date  weekday  station_id peak_time  peak_flow  baseline_peak  surge_ratio  robust_z
2023-05-06 Saturday         124  08:20:00      236.0           48.0         4.92     78.33
2023-05-06 Saturday          43  08:40:00      804.0          172.0         4.67     73.49
2023-05-06 Saturday           4  08:20:00      480.0          115.0         4.17     61.55
2023-05-06 Saturday          42  08:40:00     1768.0          286.5         6.17     60.56
2023-05-06 Saturday         122  08:40:00      866.0          204.5         4.23     59.49
2023-05-06 Saturday           3  08:40:00     2376.0          341.0         6.97     57.19
2023-05-06 Saturday          60  08:10:00      526.0          110.0         4.78     51.02
2023-05-06 Saturday          92 

## 17 · Venue proximity

In [20]:
@stage("C_venue", depends_on=["dataset", "venue_radius_km"], ext="pkl", subdir="graphs")
def build_venue_matrix() -> dict:
    venues = read_meta("event_venues.csv",
                       ["venue_id", "venue_name", "latitude", "longitude",
                        "capacity", "nearest_station_ids", "notes"])
    Cv = np.zeros((max(len(venues), 1), N_STATIONS), dtype=np.float32)
    method = "none"

    explicit = 0
    if len(venues) and "nearest_station_ids" in venues.columns:
        for vi, (_, v) in enumerate(venues.iterrows()):
            raw = str(v.get("nearest_station_ids", "") or "").strip()
            if raw and raw.lower() != "nan":
                for tok in re.split(r"[;,\s]+", raw):
                    with contextlib.suppress(ValueError):
                        sid = int(tok)
                        if sid in ST_POS:
                            Cv[vi, ST_POS[sid]] = 1.0
                            explicit += 1
    if explicit:
        method = "explicit nearest_station_ids"

    if method == "none" and HAS_COORDS and len(venues):
        md = STATION_META.set_index("station_id")
        lat = np.array([md["latitude"].get(int(s), np.nan) for s in STATION_IDS], dtype=float)
        lon = np.array([md["longitude"].get(int(s), np.nan) for s in STATION_IDS], dtype=float)
        for vi, (_, v) in enumerate(venues.iterrows()):
            d = haversine_km(float(v["latitude"]), float(v["longitude"]), lat, lon)
            Cv[vi] = (np.nan_to_num(d, nan=999.0) <= CFG.venue_radius_km).astype(np.float32)
        method = f"geodesic ≤{CFG.venue_radius_km} km"

    if method == "none":
        cand = EVENTS["candidates"]
        if len(cand):
            top = (cand.groupby("station_id")["robust_z"].max()
                   .sort_values(ascending=False).head(12))
            Cv = np.zeros((1, N_STATIONS), dtype=np.float32)
            for sid in top.index:
                Cv[0, ST_POS[int(sid)]] = 1.0
            venues = pd.DataFrame([{"venue_id": "AUTO", "venue_name": "auto-detected surge cluster"}])
            method = "data-driven surge cluster (no venue geodata supplied)"
        else:
            method = "empty (no venues, no detections)"

    M = np.zeros((N_STATIONS, N_STATIONS), dtype=np.float32)
    for vi in range(Cv.shape[0]):
        v = Cv[vi]
        M += np.outer(v, v)
    np.fill_diagonal(M, 0.0)
    M = np.clip(M, 0, 1)
    return {"C_venue": Cv, "venue_override": M, "venues": venues, "method": method}


VENUE = build_venue_matrix()
C_VENUE = VENUE["C_venue"]
VENUE_OVERRIDE = VENUE["venue_override"]
print(f"C_venue method   : {VENUE['method']}")
print(f"C_venue shape    : {C_VENUE.shape}   stations flagged venue-proximal: "
      f"{int((C_VENUE.sum(axis=0) > 0).sum())}")
print(f"Override edges   : {int(VENUE_OVERRIDE.sum() // 2)}")
print("ℹ️  CRISP-Net does not route event information through this matrix. Event-awareness "
      "enters via z_t, and via the Mondrian conformal bins.")


07:47:29 │ INFO    │ ✓ cached  [C_venue] loaded in 0.4s  (C_venue__2023__c38cca5bff.pkl)
C_venue method   : explicit nearest_station_ids
C_venue shape    : (5, 191)   stations flagged venue-proximal: 11
Override edges   : 10
ℹ️  CRISP-Net does not route event information through this matrix. Event-awareness enters via z_t, and via the Mondrian conformal bins.


## 18 · Exogenous context $z_t \in \mathbb{R}^8$

In [21]:
def peak_mask_from_hours(hours: np.ndarray) -> np.ndarray:
    """Binary AM/PM commuting-peak indicator — the 8th context feature AND the second axis of the Mondrian conformal partition."""
    m = np.zeros_like(hours, dtype=np.float32)
    for lo, hi in CFG.peak_hours:
        m = np.maximum(m, ((hours >= lo) & (hours < hi)).astype(np.float32))
    return m


IS_PEAK = peak_mask_from_hours(HOUR_OF_BIN)          # [T]


@stage("exogenous", depends_on=["dataset", "freq_minutes", "peak_hours"],
       ext="pkl", subdir="tensors")
def build_exogenous() -> dict:
    tod = (TIME_INDEX.hour * 60 + TIME_INDEX.minute).to_numpy(dtype=np.float32) / 1440.0
    dow = TIME_INDEX.dayofweek.to_numpy(dtype=np.float32)
    feats = {
        "tod_sin": np.sin(2 * np.pi * tod), "tod_cos": np.cos(2 * np.pi * tod),
        "dow_sin": np.sin(2 * np.pi * dow / 7.0), "dow_cos": np.cos(2 * np.pi * dow / 7.0),
        "is_weekend": (dow >= 5).astype(np.float32),
        "is_peak": IS_PEAK,                      # ← 8th feature
        "event_flag": EVENT_FLAG,
        "event_countdown": EVENTS["countdown"],
    }
    w = read_meta("weather.csv",
                  ["datetime", "temperature_c", "precipitation_mm", "humidity_pct"])
    if len(w):
        w["datetime"] = pd.to_datetime(w["datetime"], errors="coerce")
        w = w.dropna(subset=["datetime"]).set_index("datetime").sort_index()
        w = w.reindex(TIME_INDEX, method="nearest", tolerance=pd.Timedelta("2h"))
        for c in ["temperature_c", "precipitation_mm", "humidity_pct"]:
            if c in w.columns:
                v = w[c].to_numpy(dtype=np.float32)
                mu = np.nanmean(v)
                v = np.nan_to_num(v, nan=float(mu) if np.isfinite(mu) else 0.0)
                rng = max(v.max() - v.min(), 1e-6)
                feats[c] = ((v - v.min()) / rng).astype(np.float32)
        has_weather = True
    else:
        has_weather = False

    Z = np.stack([feats[k] for k in feats], axis=1).astype(np.float32)     # [T, dz]

    tot = X_total.sum(axis=2).astype(np.float32)
    xfer_ratio = np.clip(XFER.astype(np.float32) / np.maximum(tot, 1.0), 0, 1)
    venue_prox = np.tile((C_VENUE.sum(axis=0) > 0).astype(np.float32), (len(TIME_INDEX), 1))
    Znode = np.stack([xfer_ratio, venue_prox, EVENT_STATION_MASK], axis=2).astype(np.float32)
    return {"Z": Z, "Znode": Znode, "names": list(feats.keys()),
            "node_names": ["transfer_ratio", "venue_proximity", "event_station"],
            "has_weather": has_weather}


EXO = build_exogenous()
Z_GLOBAL, Z_NODE = EXO["Z"], EXO["Znode"]
IDX_EVENT_FEAT = EXO["names"].index("event_flag")
IDX_PEAK_FEAT = EXO["names"].index("is_peak")

print(f"z_t global : {Z_GLOBAL.shape}  → {EXO['names']}")
print(f"z_t node   : {Z_NODE.shape}  → {EXO['node_names']}")
print(f"Weather    : {'loaded' if EXO['has_weather'] else 'not supplied (optional)'}")

if Z_GLOBAL.shape[1] == DZ_EXPECTED:
    print(f"✓ dz = {Z_GLOBAL.shape[1]} — matches the proposal's 8-dimensional context vector.")
else:
    print(f"ℹ️  dz = {Z_GLOBAL.shape[1]} (weather columns present); g(z_t) adapts automatically.")

print(f"\nRegime occupancy across the corpus (the four Mondrian cells):")
_ev = Z_GLOBAL[:, IDX_EVENT_FEAT] > 0.5
_pk = Z_GLOBAL[:, IDX_PEAK_FEAT] > 0.5
for _e in [False, True]:
    for _p in [False, True]:
        _m = (_ev == _e) & (_pk == _p)
        print(f"  event={str(_e):<5s} peak={str(_p):<5s} : {int(_m.sum()):>6,} steps "
              f"({100 * _m.mean():5.2f}%)")
print(f"\nMean transfer ratio: {Z_NODE[:, :, 0].mean() * 100:.3f}%  "
      f"(audit-measured type-4 share: {qc['transfer_share_pct']:.3f}%)")
state_set("features_ready", True)


07:47:30 │ INFO    │ ✓ cached  [exogenous] loaded in 0.6s  (exogenous__2023__c29b8b64eb.pkl)
z_t global : (6156, 8)  → ['tod_sin', 'tod_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_peak', 'event_flag', 'event_countdown']
z_t node   : (6156, 191, 3)  → ['transfer_ratio', 'venue_proximity', 'event_station']
Weather    : not supplied (optional)
✓ dz = 8 — matches the proposal's 8-dimensional context vector.

Regime occupancy across the corpus (the four Mondrian cells):
  event=False peak=False :  2,730 steps (44.35%)
  event=False peak=True  :  1,050 steps (17.06%)
  event=True  peak=False :  1,716 steps (27.88%)
  event=True  peak=True  :    660 steps (10.72%)

Mean transfer ratio: 0.157%  (audit-measured type-4 share: 0.095%)


## 19 · Design system — IEEE figure style

In [22]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, LogNorm, Normalize, TwoSlopeNorm
from matplotlib.patches import Patch, Rectangle, FancyBboxPatch
from matplotlib.lines import Line2D
import seaborn as sns

# ── Colour tokens ────────────────────────────────────────────────────────────
M3 = {
    "blue600": "#17627A",   # primary        — baselines, main series
    "blue800": "#0E4657",   # primary dark   — emphasis, reference lines
    "blue900": "#082F3B",   # primary darkest— heatmap floor
    "blue100": "#BBD8E0",   # primary tint   — fills, secondary bars
    "blue50":  "#E8F1F4",   # primary wash   — interval bands, shading
    "orange":  "#E4572E",   # ACCENT         — the proposed model, always
    "green":   "#2A9D8F",   # positive / entry-side
    "purple":  "#7B5EA7",   # predecessor models
    "teal":    "#3C8DA6",   # secondary series
    "red":     "#C1292E",   # failure, miscoverage
    "amber":   "#E9A03B",   # caution, fallback-fired
    "ink":     "#101820",   # text
    "muted":   "#5A6672",   # secondary text, ticks
    "outline": "#DDE3EA",   # axis lines, legend frames
    "surface": "#EFF2F5",   # grid, weekend shading
}

PERSONA_COLORS = [M3["blue800"], M3["orange"], M3["green"], M3["purple"]]
CHANNEL_COLORS = {"Entry": M3["blue600"], "Exit": M3["orange"]}
LAG_COLORS = [M3["blue900"], M3["blue600"], M3["teal"], M3["amber"]]
BIN_COLORS = {"normal · off-peak": M3["blue100"], "normal · peak": M3["blue600"],
              "event · off-peak": M3["amber"], "event · peak": M3["orange"]}
ACCENT = M3["orange"]
INK = M3["ink"]
MUTED = M3["muted"]

# Luminance-monotone ramps — safe in greyscale.
SEQ = LinearSegmentedColormap.from_list("crisp_s", [
    "#FFFFFF", M3["blue50"], "#D3E5EB", M3["blue100"], "#8CBCCB",
    "#4E97AE", M3["blue600"], M3["blue800"], M3["blue900"]])
DIV = LinearSegmentedColormap.from_list("crisp_d", [
    M3["blue900"], M3["blue600"], "#8CBCCB", M3["blue50"], "#FAF8F6",
    "#FBDFD1", "#F0A184", M3["orange"], "#9E2F12"])
SEQ_WARM = LinearSegmentedColormap.from_list("crisp_w", [
    "#FFFFFF", "#FDF0E9", "#FBDFD1", "#F5B79C", M3["orange"], "#B33A17", "#5E1B08"])

# ── IEEE canvas geometry (unchanged — publisher requirement) ────────────────
IEEE_COL = 3.50      # single-column width, inches
IEEE_DBL = 7.16      # double-column width, inches
IEEE_WIDTH = {"single": IEEE_COL, "double": IEEE_DBL,
              "onehalf": 5.30, "full": IEEE_DBL}

mpl.rcParams.update({
    # resolution & export
    "figure.dpi": 130, "savefig.dpi": 600,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.03,
    "savefig.facecolor": "#FFFFFF",
    "pdf.fonttype": 42, "ps.fonttype": 42,      # embed TrueType — IEEE requirement
    "pdf.compression": 6,
    # type — one weight step lighter than the old sheet, one size larger
    "font.family": "sans-serif",
    "font.sans-serif": ["Inter", "Roboto", "Source Sans Pro", "DejaVu Sans",
                        "Arial", "Helvetica"],
    "font.size": 7.6,
    "axes.titlesize": 8.2, "axes.titleweight": "semibold", "axes.titlepad": 5.0,
    "axes.labelsize": 7.4, "axes.labelweight": "regular", "axes.labelpad": 3.2,
    "xtick.labelsize": 6.8, "ytick.labelsize": 6.8,
    "legend.fontsize": 6.6, "legend.title_fontsize": 6.8,
    "figure.titlesize": 8.6, "figure.titleweight": "semibold",
    "mathtext.default": "regular",
    # axes — bottom rule only
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.edgecolor": M3["muted"], "axes.linewidth": 0.7,
    "axes.facecolor": "#FFFFFF", "axes.labelcolor": M3["muted"],
    "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True,
    "axes.prop_cycle": mpl.cycler(color=[
        M3["blue600"], M3["orange"], M3["green"], M3["purple"],
        M3["amber"], M3["teal"], M3["red"], M3["blue900"]]),
    "grid.color": M3["surface"], "grid.linestyle": "-",
    "grid.linewidth": 0.6, "grid.alpha": 1.0,
    # ticks — outward, short, muted; no vertical clutter
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.major.size": 2.6, "ytick.major.size": 0.0,
    "xtick.major.width": 0.7, "ytick.major.width": 0.0,
    "xtick.minor.size": 1.4, "ytick.minor.size": 0.0,
    "xtick.major.pad": 2.4, "ytick.major.pad": 2.6,
    "xtick.color": M3["muted"], "ytick.color": M3["muted"],
    # legend — frameless; the grid already bounds the panel
    "legend.frameon": False, "legend.facecolor": "#FFFFFF",
    "legend.edgecolor": M3["outline"], "legend.framealpha": 1.0,
    "legend.borderpad": 0.30, "legend.labelspacing": 0.30,
    "legend.handlelength": 1.4, "legend.handletextpad": 0.45,
    "legend.columnspacing": 1.0, "legend.borderaxespad": 0.35,
    # lines & patches
    "figure.facecolor": "#FFFFFF",
    "lines.linewidth": 1.25, "lines.markersize": 3.1,
    "lines.markeredgewidth": 0.0, "lines.solid_capstyle": "round",
    "patch.linewidth": 0.0, "patch.force_edgecolor": False,
    "hatch.linewidth": 0.5,
    "errorbar.capsize": 0.0,
    "boxplot.flierprops.markersize": 2.0,
})

# constrained-layout paddings, in inches
_LAYOUT = dict(w_pad=0.050, h_pad=0.050, wspace=0.040, hspace=0.050)


def new_fig(width="double", height=2.6, nrows=1, ncols=1, **kw):
    """IEEE-sized figure with deterministic internal padding."""
    w = IEEE_WIDTH.get(width, width) if isinstance(width, str) else float(width)
    try:
        fig, axes = plt.subplots(nrows, ncols, figsize=(w, height),
                                 layout="constrained", **kw)
        try:
            fig.get_layout_engine().set(**_LAYOUT)
        except Exception:
            pass
    except (TypeError, ValueError):
        fig, axes = plt.subplots(nrows, ncols, figsize=(w, height), **kw)
        fig._crisp_needs_tight = True
    return fig, axes


# ── long-label defence ───────────────────────────────────────────────────────

_MIN_AXES_FRAC = 0.45      # an axes must keep >= 45% of its grid cell's width
_WRAP_AT = 15              # characters per line before a categorical label wraps


def _wrap_text(s, n=_WRAP_AT, max_lines=2):
    """Wrap on spaces only — never mid-word — then ellipsise beyond max_lines."""
    s = str(s)
    if len(s) <= n or "\n" in s:
        return s
    words, lines, cur = s.split(" "), [], ""
    for w in words:
        trial = f"{cur} {w}".strip()
        if len(trial) <= n or not cur:
            cur = trial
        else:
            lines.append(cur)
            cur = w
    if cur:
        lines.append(cur)
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        lines[-1] = lines[-1][:max(n - 1, 4)].rstrip() + "…"
    return "\n".join(lines)


def wrap_labels(labels, n=_WRAP_AT):
    """Public helper: wrap a list of category names for a tick axis."""
    return [_wrap_text(x, n) for x in labels]


def _axes_starved(fig, min_frac=_MIN_AXES_FRAC):
    """Axes whose width has been squeezed below min_frac of its grid cell."""
    starved = []
    fw = fig.get_window_extent().width
    for ax in fig.axes:
        ss = getattr(ax, "get_subplotspec", lambda: None)()
        if ss is None:
            continue
        try:
            ncols = ss.get_gridspec().ncols
        except Exception:
            ncols = 1
        if ax.get_window_extent().width < min_frac * (fw / max(ncols, 1)):
            starved.append(ax)
    return starved


def _relieve(fig):
    """Give a starved axes its drawing area back, cheapest remedy first: 1."""
    starved = _axes_starved(fig)
    if not starved:
        return False
    for ax in starved:
        labs = [t.get_text() for t in ax.get_yticklabels()]
        if any(len(l) > _WRAP_AT and " " in l for l in labs):
            ax.set_yticks(ax.get_yticks())
            ax.set_yticklabels(wrap_labels(labs), fontsize=6.0)
    fig.canvas.draw()
    if not _axes_starved(fig):
        return True
    for ax in _axes_starved(fig):
        ax.tick_params(axis="y", labelsize=5.8)
    fig.canvas.draw()
    if not _axes_starved(fig):
        return True
    with contextlib.suppress(Exception):
        fig.set_layout_engine("none")
        fig.tight_layout(pad=0.45, w_pad=0.6, h_pad=0.55)
        fig.subplots_adjust(left=min(0.34, 0.10 + 0.03 * len(starved)))
    return True


def _fit_titles(fig, min_size=6.8):
    """Keep every axis title inside its own panel."""
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    changed = False
    # `_finish` sets titles with loc="left", which matplotlib stores in
    for ax in fig.axes:
        cands = [getattr(ax, a, None) for a in ("_left_title", "title", "_right_title")]
        t = next((c for c in cands if c is not None and c.get_text().strip()), None)
        if t is None:
            continue
        txt = t.get_text()
        avail = ax.get_window_extent().width * 1.02
        if t.get_window_extent(renderer=r).width <= avail:
            continue
        # 1. wrap at a space, keeping any "(a) " panel prefix on the first line
        if "\n" not in txt and " " in txt:
            words = txt.split(" ")
            best, target = None, len(txt) / 2
            for i in range(1, len(words)):
                head = " ".join(words[:i])
                if best is None or abs(len(head) - target) < abs(len(best) - target):
                    best = head
            head_n = len(best.split(" "))
            t.set_text(" ".join(words[:head_n]) + "\n" + " ".join(words[head_n:]))
            changed = True
            fig.canvas.draw()
        # 2. still too wide → shrink, floor at min_size
        size = t.get_fontsize()
        while (t.get_window_extent(renderer=r).width > avail) and size > min_size:
            size -= 0.2
            t.set_fontsize(size)
            changed = True
            fig.canvas.draw()
    return changed


def _safe_tight(fig, has_suptitle=False):
    """Finalise a figure's geometry."""
    if getattr(fig, "_crisp_needs_tight", False):
        with contextlib.suppress(Exception):
            fig.tight_layout(pad=0.45, w_pad=0.5, h_pad=0.55)
            if has_suptitle:
                fig.subplots_adjust(top=0.90)
        return fig

    import warnings as _warnings
    with _warnings.catch_warnings(record=True) as _w:
        _warnings.simplefilter("always")
        with contextlib.suppress(Exception):
            fig.canvas.draw()
        _gave_up = any("collapsed" in str(m.message) or "not applied" in str(m.message)
                       for m in _w)
    if _gave_up:
        with contextlib.suppress(Exception):
            fig.set_layout_engine("none")
            fig.tight_layout(pad=0.45, w_pad=0.6, h_pad=0.55)
            if has_suptitle:
                fig.subplots_adjust(top=0.90)
    with contextlib.suppress(Exception):
        _relieve(fig)
    with contextlib.suppress(Exception):
        _fit_titles(fig)
    return fig


def _pname(k):
    return CFG.persona_names[k]


def _finish(ax, title=None, xlabel=None, ylabel=None, legend=None, **lkw):
    """Uniform axis styling: sentence-case semibold title, regular muted labels."""
    if title:
        ax.set_title(title, loc="left", fontsize=8.2, fontweight="semibold",
                     color=INK, pad=5.0)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=7.4, color=MUTED)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=7.4, color=MUTED)
    ax.tick_params(labelsize=6.8, colors=MUTED)
    ax.tick_params(axis="y", length=0)
    if legend:
        ax.legend(**{"fontsize": 6.6, "loc": "best", "frameon": False, **lkw})
    return ax


_PANEL_LETTERS = "abcdefghijklmnopqrstuvwxyz"


def _panel(ax, i, dx=-0.055, dy=1.045, upper=False):
    """IEEE sub-figure label '(a)', '(b)', … folded into the axis title when one exists."""
    lab = _PANEL_LETTERS[i] if isinstance(i, int) else str(i)
    tag = f"({lab.upper() if upper else lab})"
    t = ax.get_title(loc="left")
    if t:
        if not t.startswith("("):
            ax.set_title(f"{tag} {t}", loc="left", fontsize=8.2,
                         fontweight="semibold", color=INK, pad=5.0)
    else:
        ax.text(dx, dy, tag, transform=ax.transAxes, fontsize=8.2,
                fontweight="semibold", color=INK, va="bottom", ha="left")
    return ax


def _panels(axes, **kw):
    """Label a whole grid of axes in reading order."""
    for i, ax in enumerate(np.atleast_1d(axes).ravel()):
        if ax.get_visible():
            _panel(ax, i, **kw)
    return axes


def _cbar(fig, im, ax, label=None, **kw):
    """Colourbar with consistent thickness, padding and label typography."""
    cb = fig.colorbar(im, ax=ax, fraction=kw.pop("fraction", 0.046),
                      pad=kw.pop("pad", 0.020), shrink=kw.pop("shrink", 0.92), **kw)
    cb.outline.set_linewidth(0.0)
    cb.ax.tick_params(labelsize=6.2, length=1.8, width=0.45, colors=MUTED)
    if label:
        cb.set_label(label, fontsize=6.8, color=MUTED, labelpad=3)
    return cb


def _heatmap_annot(ax, data, fmt=".2f", fs=5.4, thresh=None):
    """Annotate heatmap cells — only call for small matrices."""
    data = np.asarray(data)
    nr, nc = data.shape
    mid = thresh if thresh is not None else (np.nanmax(data) + np.nanmin(data)) / 2.0
    for i in range(nr):
        for j in range(nc):
            v = data[i, j]
            if not np.isfinite(v):
                continue
            ax.text(j, i, f"{v:{fmt}}", ha="center", va="center",
                    color="#FFFFFF" if v > mid else INK, fontsize=fs,
                    fontweight="semibold")


def _bar_labels(ax, values, fmt="{:.2f}", pad=0.006, horizontal=True, fs=6.0, color=None):
    """Value labels on bars, offset as a fraction of the data range."""
    values = np.asarray(values, dtype=float)
    span = np.nanmax(np.abs(values)) if len(values) else 1.0
    off = pad * max(span, 1e-9) * 4
    for i, v in enumerate(values):
        if not np.isfinite(v):
            continue
        if horizontal:
            ax.text(v + np.sign(v if v != 0 else 1) * off, i, fmt.format(v),
                    va="center", ha="left" if v >= 0 else "right",
                    fontsize=fs, color=color or MUTED)
        else:
            ax.text(i, v + np.sign(v if v != 0 else 1) * off, fmt.format(v),
                    ha="center", va="bottom" if v >= 0 else "top",
                    fontsize=fs, color=color or MUTED)


def _hour_axis(ax, ticks=(6, 9, 12, 15, 18, 21)):
    ax.set_xticks(list(ticks))
    ax.set_xticklabels([f"{t:02d}" for t in ticks])
    return ax


def _thousands(ax, axis="y"):
    f = ticker.FuncFormatter(lambda v, _: f"{v:,.0f}")
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(f)
    return ax


def _shade_weekends(ax, dates, is_weekend):
    for d, w in zip(dates, is_weekend):
        if w:
            ax.axvspan(d - pd.Timedelta("12h"), d + pd.Timedelta("12h"),
                       color=M3["surface"], zorder=0, lw=0)
    return ax


def _series_color(name: str) -> str:
    """One rule for the whole paper: the proposed model is coral, nothing else is."""
    n = str(name)
    if n.startswith(NAME_PROPOSED if "NAME_PROPOSED" in globals() else "CRISP-Net"):
        return ACCENT if "w/o" not in n else M3["amber"]
    if n.startswith("TDAG-Net") or n.startswith("TGALSTM"):
        return M3["purple"]
    return M3["blue600"]


log("CRISP design system configured "
    f"(single {IEEE_COL}in / double {IEEE_DBL}in · 600 dpi · Type-42 · "
    "slate-teal primary, coral accent, greyscale-safe ramps).")


07:47:30 │ INFO    │ CRISP design system configured (single 3.5in / double 7.16in · 600 dpi · Type-42 · slate-teal primary, coral accent, greyscale-safe ramps).


## 20 · Figures — corpus, cleaning, scale, bidirectional structure

In [23]:
@figure("F01", "Ingestion audit and data retention across source files", "01_corpus")
def fig_ingestion_audit():
    if not len(_audit):
        return _skip("F01", "no ingestion audit rows")
    d = _audit.sort_values("rows_read", ascending=True).copy()
    d["discarded"] = d["rows_read"] - d["rows_kept"]
    fig, axes = new_fig("double", max(2.5, 0.155 * len(d) + 0.7), 1, 2,
                        gridspec_kw={"width_ratios": [2.3, 1.0]})
    y = np.arange(len(d))
    axes[0].barh(y, d["rows_kept"] / 1e6, color=M3["blue600"], height=0.72, label="retained")
    axes[0].barh(y, d["discarded"] / 1e6, left=d["rows_kept"] / 1e6,
                 color="#d9d9d9", height=0.72, label="discarded")
    axes[0].set_yticks(y)
    axes[0].set_yticklabels(d["file"], fontsize=5.8)
    axes[0].legend(loc="lower right", fontsize=6.2)
    _finish(axes[0], "Records per source file", "records (millions)")

    rate = 100 * d["discarded"] / d["rows_read"].clip(lower=1)
    axes[1].barh(y, rate, color=np.where(rate > 0.1, ACCENT, M3["blue100"]), height=0.72)
    axes[1].set_yticks([])
    axes[1].axvline(0.1, color=MUTED, ls="--", lw=0.6)
    _finish(axes[1], "Discard rate", "% of rows read")
    _bar_labels(axes[1], rate.to_numpy(), fmt="{:.2f}%", fs=5.6, color=ACCENT)
    axes[1].set_xlim(0, max(rate.max() * 1.35, 0.25))
    _panels(axes)
    _safe_tight(fig)
    return fig, d


@figure("F02", "Longitudinal network ridership by direction with flagged event days", "01_corpus")
def fig_daily_ridership():
    D4 = X_total.reshape(N_DAYS, BINS_PER_DAY, N_STATIONS, 2)
    ent = D4[:, :, :, CH_IN].sum(axis=(1, 2))
    ext = D4[:, :, :, CH_OUT].sum(axis=(1, 2))
    ev_set = set(EVENTS["event_days"])
    d = pd.DataFrame({"date": DAY_DATES, "entry": ent, "exit": ext,
                      "total": ent + ext, "weekend": IS_WEEKEND_DAY,
                      "event": [x in ev_set for x in TENS["days"]]})
    fig, axes = new_fig("double", 3.5, 2, 1, sharex=True,
                        gridspec_kw={"height_ratios": [1.55, 1.0]})
    ax = axes[0]
    _shade_weekends(ax, d["date"], d["weekend"])
    ax.plot(d["date"], d["entry"] / 1e6, "-o", ms=2.4, lw=1.05,
            color=CHANNEL_COLORS["Entry"], label="entry", zorder=3)
    ax.plot(d["date"], d["exit"] / 1e6, "-s", ms=2.2, lw=1.05,
            color=CHANNEL_COLORS["Exit"], label="exit", zorder=3)
    ev = d[d["event"]]
    if len(ev):
        ax.scatter(ev["date"], ev["entry"] / 1e6, s=34, facecolor="none",
                   edgecolor=M3["red"], lw=0.9, zorder=5, label="flagged event day")
    med = np.median(ent) / 1e6
    ax.axhline(med, color=MUTED, ls="--", lw=0.7, zorder=2,
               label=f"entry median {med:.2f} M")
    ax.legend(loc="lower left", ncol=4, fontsize=6.2)
    _finish(ax, "Daily network ridership by direction", None, "taps (millions)")

    ax = axes[1]
    imb = 100 * (d["entry"] - d["exit"]) / d["total"].clip(lower=1)
    _shade_weekends(ax, d["date"], d["weekend"])
    ax.bar(d["date"], imb, width=0.72,
           color=np.where(np.abs(imb) > 1.0, ACCENT, M3["blue600"]))
    ax.axhline(0, color=INK, lw=0.6)
    ax.axhspan(-1, 1, color=M3["blue50"], zorder=0, lw=0)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    _finish(ax, "Entry–exit imbalance (closed-system check)", None, "imbalance (%)")
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45)
        lbl.set_ha("right")
    _panels(axes)
    _safe_tight(fig)
    return fig, d


@figure("F03", "Service-hour completeness and data availability", "01_corpus")
def fig_coverage_calendar():
    daily = X_total.reshape(N_DAYS, BINS_PER_DAY, N_STATIONS, 2).sum(axis=(1, 2, 3))
    live_bins = (X_total.sum(axis=(1, 2)).reshape(N_DAYS, BINS_PER_DAY) > 0).sum(axis=1)
    d = pd.DataFrame({"date": DAY_DATES, "records": daily,
                      "live_bins": live_bins, "coverage_pct": 100 * live_bins / BINS_PER_DAY})
    d["week"] = d["date"].dt.isocalendar().week.to_numpy()
    d["dow"] = d["date"].dt.dayofweek
    weeks = sorted(d["week"].unique())
    M = np.full((7, len(weeks)), np.nan)
    for _, r in d.iterrows():
        M[int(r["dow"]), weeks.index(int(r["week"]))] = r["records"] / 1e6

    fig, axes = new_fig("double", 3.9, 2, 1, gridspec_kw={"height_ratios": [1.15, 1.0]})
    im = axes[0].imshow(M, aspect="auto", cmap=SEQ, interpolation="nearest")
    if M.shape[1] <= 22:
        _heatmap_annot(axes[0], np.nan_to_num(M), fmt=".1f", fs=4.9,
                       thresh=np.nanmean(M))
    axes[0].set_yticks(range(7))
    axes[0].set_yticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], fontsize=6.4)
    axes[0].set_xticks(range(len(weeks)))
    axes[0].set_xticklabels([f"W{w}" for w in weeks], fontsize=6.2)
    axes[0].grid(False)
    _cbar(fig, im, axes[0], "M taps")
    _finish(axes[0], "Calendar coverage (ISO week × weekday)")

    axes[1].bar(d["date"], d["coverage_pct"], width=0.72,
                color=np.where(d["coverage_pct"] > 95, M3["blue600"], ACCENT))
    axes[1].axhline(95, color=MUTED, ls="--", lw=0.7)
    axes[1].set_ylim(0, 105)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    _finish(axes[1], "Bin occupancy per day", None, "non-empty bins (%)")
    for lbl in axes[1].get_xticklabels():
        lbl.set_rotation(45)
        lbl.set_ha("right")
    _panels(axes)
    _safe_tight(fig)
    return fig, d


@figure("F04", "Ticket-code volume distribution and cumulative coverage", "01_corpus")
def fig_card_pareto():
    d = CARD_FEATS.sort_values("share_pct", ascending=False).reset_index(drop=True).copy()
    d["cum_pct"] = d["share_pct"].cumsum()
    pmap = PERSONA_ASSIGN.set_index("label")["persona_idx"]
    colors = [PERSONA_COLORS[int(pmap.loc[l])] for l in d["label"]]
    fig, ax = new_fig("double", 2.7)
    ax.bar(range(len(d)), d["share_pct"], color=colors, width=0.76)
    ax.set_xticks(range(len(d)))
    ax.set_xticklabels(d["label"], rotation=55, ha="right", fontsize=6.0)
    _finish(ax, "Ticket-code volume share and Pareto coverage", None, "share of taps (%)")
    ax2 = ax.twinx()
    ax2.plot(range(len(d)), d["cum_pct"], color=INK, lw=1.0, marker="o", ms=2.0, zorder=5)
    ax2.set_ylabel("cumulative (%)", fontsize=7.5, fontweight="bold", color=MUTED)
    ax2.set_ylim(0, 104)
    ax2.grid(False)
    ax2.tick_params(labelsize=6.8, colors=MUTED, length=2.4, width=0.5)
    ax2.axhline(90, color=MUTED, ls=":", lw=0.7)
    ax2.spines["right"].set_visible(True)
    ax2.spines["right"].set_color(M3["outline"])
    ax2.spines["right"].set_linewidth(0.6)
    ax.legend(handles=[Patch(color=PERSONA_COLORS[k], label=_pname(k))
                       for k in range(K_PERSONA)],
              loc="center right", fontsize=6.2, ncol=1)
    _safe_tight(fig)
    return fig, d


@figure("F05", "Diurnal temporal signature of each ticket code", "01_corpus")
def fig_card_signature_heatmap():
    prof = SIG["profile_norm"]
    order = np.argsort(-CARD_FEATS["share_pct"].to_numpy())
    Mtx = prof[:, order].T
    fig, ax = new_fig("double", max(2.8, 0.135 * N_CARD + 0.9))
    im = ax.imshow(Mtx, aspect="auto", cmap=SEQ, interpolation="nearest")
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([f"{CARD_FEATS['label'].iloc[i]}  ({CARD_FEATS['share_pct'].iloc[i]:.1f}%)"
                        for i in order], fontsize=5.8)
    tick = np.arange(0, BINS_PER_DAY, max(1, BINS_PER_DAY // 12))
    ax.set_xticks(tick)
    ax.set_xticklabels([f"{int(HOUR_OF_BIN[t]):02d}" for t in tick], fontsize=6.4)
    ax.grid(False)
    for i, oi in enumerate(order):
        k = int(PERSONA_ASSIGN["persona_idx"].iloc[oi])
        ax.add_patch(Rectangle((-2.4, i - 0.5), 1.7, 1, color=PERSONA_COLORS[k],
                               clip_on=False, lw=0))
    ax.set_xlim(-2.6, BINS_PER_DAY - 0.5)
    _cbar(fig, im, ax, "share of daily volume")
    _finish(ax, "Diurnal profile per ticket code (persona colour-coded at left)",
            "hour of day")
    _safe_tight(fig)
    return fig, pd.DataFrame(Mtx, index=[CARD_FEATS["label"].iloc[i] for i in order])


@figure("F06", "Spatial demand concentration across the network", "01_corpus")
def fig_demand_concentration():
    tot = X_total.sum(axis=(0, 2)).astype(float)
    s = np.sort(tot)[::-1]
    lor = np.cumsum(np.sort(tot)) / tot.sum()
    x = np.arange(1, len(tot) + 1) / len(tot)
    trap = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    gini = 1 - 2 * trap(lor, x)

    fig, axes = new_fig("double", 2.5, 1, 2)
    axes[0].bar(range(len(s)), s / 1e3, color=M3["blue600"], width=1.0)
    axes[0].set_yscale("log")
    _finish(axes[0], "Station volume, rank-ordered", "station rank", "taps (thousands, log)")
    axes[1].plot(x, lor, color=M3["blue600"], lw=1.3, zorder=3)
    axes[1].plot([0, 1], [0, 1], color=MUTED, ls="--", lw=0.7)
    axes[1].fill_between(x, lor, x, color=M3["blue600"], alpha=0.16, lw=0)
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)
    axes[1].text(0.05, 0.88, f"Gini = {gini:.3f}", transform=axes[1].transAxes,
                 fontsize=7.0, fontweight="bold", color=INK)
    _finish(axes[1], "Lorenz curve of demand", "cumulative station share",
            "cumulative demand share")
    _panels(axes)
    _safe_tight(fig)
    return fig, pd.DataFrame({"rank": range(1, len(s) + 1), "volume": s,
                              "cum_share": np.cumsum(s) / s.sum()})


@figure("F07", "Entry-to-exit propagation across the network", "01_corpus")
def fig_delay_evidence():
    net = DELAY["net_corr"]
    jm = PROBE.get("journey_minutes", np.array([]))
    fig, axes = new_fig("double", 2.5, 1, 3,
                        gridspec_kw={"width_ratios": [1.15, 1.25, 1.0]})

    # (a) network lagged cross-correlation
    ax = axes[0]
    cols = [ACCENT if int(r["lag_steps"]) in DELAY_LAGS else M3["blue100"]
            for _, r in net.iterrows()]
    ax.bar(net["lag_steps"], net["corr"], color=cols, width=0.66)
    best = net.loc[net["corr"].idxmax()]
    ax.axvline(best["lag_steps"], color=M3["red"], ls="--", lw=0.8, zorder=1)
    ax.annotate(f"peak δ={int(best['lag_steps'])}",
                xy=(best["lag_steps"], best["corr"]), xytext=(3, -9),
                textcoords="offset points", fontsize=6.0, color=M3["red"],
                fontweight="bold")
    ax.axvspan(min(DELAY_LAGS) - 0.5, max(DELAY_LAGS) + 0.5,
               color=ACCENT, alpha=0.08, zorder=0, lw=0)
    ax.set_xticks(net["lag_steps"])
    _finish(ax, r"corr($\Sigma$entry$[t]$, $\Sigma$exit$[t{+}\delta]$)",
            r"lag $\delta$ (10-min steps)", "Pearson r")

    # (b) journey duration distribution vs the kernel cut-off
    ax = axes[1]
    if len(jm):
        cut = CFG.freq_minutes * max(DELAY_LAGS)        # δ≤3 → 30 min
        bins = np.arange(0, 121, 2.5)
        ax.hist(jm, bins=bins, color=M3["blue600"], edgecolor="none", alpha=0.9)
        ax.axvline(cut, color=M3["red"], ls="--", lw=0.9)
        frac = 100 * (jm <= cut).mean()
        ax.annotate(f"{frac:.0f}% within\nδ ≤ {max(DELAY_LAGS)} ({cut} min)",
                    xy=(cut, ax.get_ylim()[1] * 0.72), xytext=(6, 0),
                    textcoords="offset points", fontsize=6.2, color=M3["red"],
                    fontweight="bold", va="center")
        # stagger the percentile labels vertically — at p50/p90 spacings they would
        # otherwise print on top of one another and on top of the cut-off annotation
        for (q, ls, yf) in [(50, ":", 0.99), (90, "-.", 0.88)]:
            xq = float(np.percentile(jm, q))
            ax.axvline(xq, color=MUTED, ls=ls, lw=0.7)
            ax.annotate(f"p{q} = {xq:.0f}", xy=(xq, ax.get_ylim()[1] * yf),
                        xytext=(2, 0), textcoords="offset points",
                        fontsize=5.8, color=MUTED, ha="left", va="top")
        ax.set_xlim(0, 120)
        ax.set_ylim(0, ax.get_ylim()[1] * 1.12)
        _thousands(ax)
    _finish(ax, "Observed journey duration", "minutes", "trips")

    # (c) per-lag masked edge strength = the empirical kernel
    ax = axes[2]
    ek = DELAY["empirical_kernel"]
    ax.bar(range(len(DELAY_LAGS)), 100 * ek,
           color=[LAG_COLORS[i % len(LAG_COLORS)] for i in range(len(DELAY_LAGS))],
           width=0.62)
    ax.set_xticks(range(len(DELAY_LAGS)))
    ax.set_xticklabels([rf"$A^{{({d})}}$" for d in DELAY_LAGS], fontsize=6.6)
    _bar_labels(ax, 100 * ek, fmt="{:.1f}%", horizontal=False, fs=6.0)
    ax.set_ylim(0, max(100 * ek) * 1.25)
    _finish(ax, "Masked edge strength per lag", r"delay kernel support",
            "share of total (%)")
    _panels(axes)
    _safe_tight(fig)
    return fig, pd.DataFrame({"lag": DELAY_LAGS,
                              "edge_strength": DELAY["edge_strength"],
                              "empirical_kernel": ek})


## 21 · Figures — personas and context vector

In [24]:
@figure("F08", "Passenger personas in behavioural feature space", "02_persona")
def fig_persona_space():
    from sklearn.decomposition import PCA          # local: figure cells stay self-contained
    Z = PERSONA["Z"]
    if Z.shape[0] < 3:
        return _skip("F08", "too few ticket codes to project")
    p = PCA(n_components=2, random_state=CFG.seed)
    XY = p.fit_transform(Z)
    vol = CARD_FEATS["volume"].to_numpy(dtype=float)
    size = 14 + 260 * (vol / max(vol.max(), 1)) ** 0.45
    fig, ax = new_fig("onehalf", 3.0)
    for k in range(K_PERSONA):
        m = PERSONA_ASSIGN["persona_idx"].to_numpy() == k
        ax.scatter(XY[m, 0], XY[m, 1], s=size[m], color=PERSONA_COLORS[k],
                   alpha=0.78, edgecolor="#ffffff", lw=0.6, label=_pname(k), zorder=3)
    for i, lab in enumerate(CARD_FEATS["label"]):
        if CARD_FEATS["share_pct"].iloc[i] > 1.0:
            ax.annotate(lab.replace("card_", ""), (XY[i, 0], XY[i, 1]),
                        fontsize=5.6, ha="center", va="center", color="white",
                        weight="bold", zorder=4)
    ax.legend(loc="best", markerscale=0.45, fontsize=6.2)
    _finish(ax, "Persona clusters (marker area ∝ ridership)",
            f"PC1 ({p.explained_variance_ratio_[0] * 100:.0f}% var.)",
            f"PC2 ({p.explained_variance_ratio_[1] * 100:.0f}% var.)")
    _safe_tight(fig)
    return fig, pd.DataFrame({"label": CARD_FEATS["label"], "pc1": XY[:, 0], "pc2": XY[:, 1],
                              "persona": PERSONA_ASSIGN["persona"], "volume": vol})


@figure("F09", "Diurnal persona profiles, weekday versus weekend", "02_persona")
def fig_persona_profiles():
    prof = X_persona.sum(axis=(1, 2)).reshape(N_DAYS, BINS_PER_DAY, K_PERSONA)
    wd = prof[~IS_WEEKEND_DAY].mean(axis=0) if (~IS_WEEKEND_DAY).any() else np.zeros((BINS_PER_DAY, K_PERSONA))
    we = prof[IS_WEEKEND_DAY].mean(axis=0) if IS_WEEKEND_DAY.any() else np.zeros((BINS_PER_DAY, K_PERSONA))
    hours = HOUR_OF_BIN[:BINS_PER_DAY]
    fig, axes = new_fig("double", 1.9, 1, K_PERSONA, sharex=True)
    for k in range(K_PERSONA):
        ax = axes[k]
        ax.fill_between(hours, wd[:, k] / 1e3, color=PERSONA_COLORS[k], alpha=0.22, lw=0)
        ax.plot(hours, wd[:, k] / 1e3, color=PERSONA_COLORS[k], lw=1.15, label="weekday")
        ax.plot(hours, we[:, k] / 1e3, color=INK, lw=0.95, ls="--", label="weekend")
        wr = we[:, k].sum() / max(wd[:, k].sum(), 1e-9)
        _finish(ax, f"{_pname(k).split('/')[0]}  (we/wd = {wr:.2f})", "hour",
                "k taps / 10 min" if k == 0 else None)
        _hour_axis(ax)
        if k == 0:
            ax.legend(loc="upper left", fontsize=6.0)
    _panels(axes)
    _safe_tight(fig)
    d = pd.DataFrame(np.hstack([wd, we]),
                     columns=[f"weekday_{_pname(k)}" for k in range(K_PERSONA)] +
                             [f"weekend_{_pname(k)}" for k in range(K_PERSONA)])
    d.insert(0, "hour", hours)
    return fig, d


@figure("F10", "Soft-assignment prior matrix P0 inherited from Paper 3", "02_persona")
def fig_prior_matrix():
    order = np.argsort(-CARD_FEATS["share_pct"].to_numpy())
    Mtx = PERSONA_PRIOR[order]
    fig, axes = new_fig("onehalf", max(2.8, 0.135 * N_CARD + 0.9), 1, 2,
                        gridspec_kw={"width_ratios": [1.0, 1.25]})
    im = axes[0].imshow(Mtx, aspect="auto", cmap=SEQ, vmin=0, vmax=1, interpolation="nearest")
    axes[0].set_xticks(range(K_PERSONA))
    axes[0].set_xticklabels([_pname(k).split("/")[0] for k in range(K_PERSONA)],
                            rotation=32, ha="right", fontsize=6.2)
    axes[0].set_yticks(range(len(order)))
    axes[0].set_yticklabels([CARD_FEATS["label"].iloc[i] for i in order], fontsize=5.8)
    axes[0].grid(False)
    _cbar(fig, im, axes[0], r"$P(\mathrm{persona}\mid\mathrm{code})$")
    _finish(axes[0], r"Prior $P_0$")

    conf = PERSONA_PRIOR[order].max(axis=1)
    share = CARD_FEATS["share_pct"].to_numpy()[order]
    cols = [PERSONA_COLORS[int(PERSONA_ASSIGN["persona_idx"].iloc[i])] for i in order]
    axes[1].barh(range(len(order)), conf, color=cols, height=0.74)
    axes[1].axvline(1 / K_PERSONA, color=MUTED, ls="--", lw=0.7)
    axes[1].set_yticks(range(len(order)))
    axes[1].set_yticklabels([f"{s:.1f}%" for s in share], fontsize=5.8)
    axes[1].set_xlim(0, 1.0)
    _finish(axes[1], "Assignment confidence (label = volume share)", r"max$_k P_0$")
    axes[0].invert_yaxis()
    axes[1].invert_yaxis()
    _panels(axes)
    _safe_tight(fig)
    d = pd.DataFrame(PERSONA_PRIOR, columns=[_pname(k) for k in range(K_PERSONA)],
                     index=CARD_FEATS["label"]).reset_index()
    return fig, d


@figure("F11", "Persona mix trajectory", "02_persona")
def fig_pi_trajectory():
    hours = HOUR_OF_BIN[:BINS_PER_DAY]
    pi_day = PI_T.reshape(N_DAYS, BINS_PER_DAY, K_PERSONA)
    wd = pi_day[~IS_WEEKEND_DAY].mean(axis=0) if (~IS_WEEKEND_DAY).any() else pi_day.mean(axis=0)
    we = pi_day[IS_WEEKEND_DAY].mean(axis=0) if IS_WEEKEND_DAY.any() else pi_day.mean(axis=0)
    ev_set = set(EVENTS["event_days"])
    ev_mask = np.array([d in ev_set for d in TENS["days"]])
    evp = pi_day[ev_mask].mean(axis=0) if ev_mask.any() else None

    fig, axes = new_fig("double", 2.4, 1, 3,
                        gridspec_kw={"width_ratios": [1.35, 1.35, 0.9]})
    for ax, mix, ttl in [(axes[0], wd, "Weekday"), (axes[1], we, "Weekend")]:
        bottom = np.zeros(BINS_PER_DAY)
        for k in range(K_PERSONA):
            ax.fill_between(hours, bottom, bottom + 100 * mix[:, k],
                            color=PERSONA_COLORS[k], alpha=0.88, lw=0,
                            label=_pname(k).split("/")[0])
            bottom += 100 * mix[:, k]
        ax.set_ylim(0, 100)
        ax.set_xlim(hours[0], hours[-1])
        _hour_axis(ax)
        _finish(ax, rf"{ttl}: $\pi_t$ composition", "hour",
                "share of flow (%)" if ax is axes[0] else None)
        ax.grid(False)
    axes[0].legend(ncol=2, fontsize=5.9, loc="lower center")

    ax = axes[2]
    w = 0.36
    xs = np.arange(K_PERSONA)
    ax.bar(xs - w / 2, 100 * wd.mean(axis=0), w, color=M3["blue100"],
           edgecolor=M3["blue600"], lw=0.5, label="weekday")
    ax.bar(xs + w / 2, 100 * we.mean(axis=0), w, color=M3["blue600"], label="weekend")
    if evp is not None:
        ax.plot(xs, 100 * evp.mean(axis=0), "D", ms=3.2, color=M3["red"],
                label="event day", zorder=5)
    ax.set_xticks(xs)
    ax.set_xticklabels([_pname(k).split("/")[0] for k in range(K_PERSONA)],
                       rotation=32, ha="right", fontsize=6.2)
    ax.legend(fontsize=6.0)
    _finish(ax, "Regime means", None, "share (%)")
    _panels(axes)
    _safe_tight(fig)
    d = pd.DataFrame(100 * wd, columns=[f"weekday_{_pname(k)}" for k in range(K_PERSONA)])
    for k in range(K_PERSONA):
        d[f"weekend_{_pname(k)}"] = 100 * we[:, k]
    d.insert(0, "hour", hours)
    return fig, d


@figure("F12", "Exogenous context vector z_t and its regime partition", "02_persona")
def fig_context_vector():
    names = EXO["names"]
    dz = Z_GLOBAL.shape[1]
    show_days = min(7, N_DAYS)
    sl = slice(0, show_days * BINS_PER_DAY)
    t = TIME_INDEX[sl]

    fig, axes = new_fig("double", 3.6, 2, 1,
                        gridspec_kw={"height_ratios": [1.5, 1.0]})
    ax = axes[0]
    offs = np.arange(dz)[::-1] * 1.25
    pal = [M3["blue600"], M3["blue800"], M3["teal"], M3["blue900"],
           M3["muted"], M3["purple"], M3["red"], ACCENT, M3["green"], M3["amber"]]
    for i in range(dz):
        v = Z_GLOBAL[sl, i]
        rng = max(v.max() - v.min(), 1e-6)
        ax.plot(t, offs[i] + (v - v.min()) / rng, lw=0.75, color=pal[i % len(pal)])
        ax.text(t[0], offs[i] + 0.52, names[i], fontsize=6.0, color=MUTED,
                va="center", ha="left", fontweight="bold",
                bbox=dict(fc="white", ec="none", pad=0.6, alpha=0.75))
    ax.set_yticks([])
    ax.set_ylim(-0.3, offs[0] + 1.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.grid(False)
    _finish(ax, rf"$z_t \in \mathbb{{R}}^{{{dz}}}$ over the first {show_days} days "
                "(each trace min–max scaled)")

    ax = axes[1]
    ev = Z_GLOBAL[:, IDX_EVENT_FEAT] > 0.5
    pk = Z_GLOBAL[:, IDX_PEAK_FEAT] > 0.5
    cells, labels, counts = [], [], []
    for e in [False, True]:
        for p in [False, True]:
            m = (ev == e) & (pk == p)
            cells.append((e, p))
            labels.append(f"{'event' if e else 'normal'}\n{'peak' if p else 'off-peak'}")
            counts.append(int(m.sum()))
    cmap_c = [M3["blue100"], M3["blue600"], "#fdd9b5", ACCENT]
    ax.bar(range(4), counts, color=cmap_c, width=0.62,
           edgecolor=M3["outline"], lw=0.4)
    ax.set_xticks(range(4))
    ax.set_xticklabels(labels, fontsize=6.2)
    _bar_labels(ax, np.array(counts, dtype=float), fmt="{:,.0f}",
                horizontal=False, fs=6.0)
    ax.set_ylim(0, max(counts) * 1.22)
    _thousands(ax)
    _finish(ax, "Mondrian conformal partition: occupancy of the four bins",
            None, "10-min steps")
    _panels(axes)
    _safe_tight(fig)
    return fig, pd.DataFrame({"bin": labels, "steps": counts})


## 22 · Figures — graph structure and events

In [25]:
import networkx as nx


def _graph_layout(A, seed=None):
    """Deterministic spring layout on the topology, seeded for reproducibility."""
    G = nx.from_numpy_array(np.asarray(A))
    return G, nx.spring_layout(G, seed=CFG.seed if seed is None else seed,
                               k=1.6 / np.sqrt(max(G.number_of_nodes(), 1)),
                               iterations=180)


@figure("F13", "Inferred metro topology coloured by line and sized by ridership", "03_graph")
def fig_network_graph():
    G, pos = _graph_layout(A_TOPO)
    vol = X_total.sum(axis=(0, 2)).astype(float)
    size = 4 + 90 * (vol / max(vol.max(), 1)) ** 0.6
    lines = np.array([STATION_LINE.get(int(s), 0) for s in STATION_IDS])
    uq = sorted(set(lines.tolist()))
    cmap = plt.get_cmap("tab20")
    cmap_line = {l: cmap(i % 20) for i, l in enumerate(uq)}
    node_c = [cmap_line[l] for l in lines]

    fig, ax = new_fig("onehalf", 3.6)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=M3["outline"], width=0.45, alpha=0.85)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=size, node_color=node_c,
                           linewidths=0.35, edgecolors="#ffffff")
    for i in np.argsort(-vol)[:8]:
        ax.annotate(str(int(STATION_IDS[i])), pos[i], fontsize=5.6, fontweight="bold",
                    color=INK, ha="center", va="center",
                    path_effects=[pe.withStroke(linewidth=1.6, foreground="white")])
    ax.set_axis_off()
    handles = [Line2D([], [], marker="o", ls="", ms=3.4, color=cmap_line[l],
                      label=f"L{int(l)}") for l in uq if l]
    ax.legend(handles=handles, loc="upper left", ncol=4, fontsize=5.8,
              title="line", title_fontsize=6.2)
    ax.set_title(f"Topology: {N_STATIONS} stations, {TOPO['n_edges']} edges "
                 f"(marker area ∝ ridership)", loc="left",
                 fontsize=8.0, fontweight="bold", color=INK, pad=4)
    _safe_tight(fig)
    return fig, pd.DataFrame({"station_id": STATION_IDS, "line": lines, "volume": vol})


@figure("F14", "Correlation topology and the sparsity mask M", "03_graph")
def fig_mask_panel():
    order = np.argsort([STATION_LINE.get(int(s), 99) for s in STATION_IDS])
    ix = np.ix_(order, order)
    panels = [
        (GRAPH["C_all"][ix], "Aggregate correlation", DIV, -1, 1),
        (GRAPH["C_origin"][ix], "Entry-channel correlation", DIV, -1, 1),
        (GRAPH["C_dest"][ix], "Exit-channel correlation", DIV, -1, 1),
        (MASK_M[ix], r"Sparsity mask $M$", SEQ, 0, 1),
    ]
    fig, axes = new_fig("double", 2.15, 1, 4)
    for i, (Mx, ttl, cm, lo, hi) in enumerate(panels):
        im = axes[i].imshow(Mx, cmap=cm, vmin=lo, vmax=hi, interpolation="nearest")
        axes[i].set_xticks([])
        axes[i].set_yticks([])
        axes[i].grid(False)
        axes[i].set_title(ttl, loc="left", fontsize=7.2, fontweight="bold", color=INK, pad=3.5)
        _cbar(fig, im, axes[i], None, shrink=0.86)
    axes[0].set_ylabel("stations (ordered by line)", fontsize=6.8,
                       fontweight="bold", color=MUTED)
    fig.suptitle(f"Live edges in M: {int(MASK_M.sum() // 2):,} "
                 f"({100 * MASK_M.sum() / (N_STATIONS ** 2 - N_STATIONS):.2f}% density) · "
                 f"entry-vs-exit graph cosine {GRAPH['od_cosine']:.4f}",
                 fontsize=7.2, fontweight="bold", color=MUTED)
    _panels(axes)
    _safe_tight(fig, has_suptitle=True)
    return fig, pd.DataFrame({"stat": ["live_edges", "density_pct", "od_cosine",
                                       "mean_degree"],
                              "value": [MASK_M.sum() // 2,
                                        100 * MASK_M.sum() / (N_STATIONS ** 2 - N_STATIONS),
                                        GRAPH["od_cosine"], MASK_M.sum(axis=1).mean()]})


@figure("F15", "Empirical delay-lagged adjacency A(delta)", "03_graph")
def fig_delta_adjacency():
    order = np.argsort([STATION_LINE.get(int(s), 99) for s in STATION_IDS])
    ix = np.ix_(order, order)
    vmax = float(np.percentile(A_DELTA[A_DELTA > 0], 99)) if (A_DELTA > 0).any() else 1.0
    fig, axes = new_fig("double", 2.25, 1, N_LAGS + 1,
                        gridspec_kw={"width_ratios": [1] * N_LAGS + [0.85]})
    for i, d in enumerate(DELAY_LAGS):
        im = axes[i].imshow(A_DELTA[i][ix], cmap=SEQ, vmin=0, vmax=vmax,
                            interpolation="nearest")
        axes[i].set_xticks([])
        axes[i].set_yticks([])
        axes[i].grid(False)
        axes[i].set_title(rf"$A^{{({d})}}$  ({d * CFG.freq_minutes} min)",
                          loc="left", fontsize=7.2, fontweight="bold", color=INK, pad=3.5)
        if i == N_LAGS - 1:
            _cbar(fig, im, axes[i], "lagged corr.", shrink=0.86)
    ax = axes[-1]
    nz = [(A_DELTA[i] > 1e-3).sum() / max(MASK_M.sum(), 1) for i in range(N_LAGS)]
    ax.bar(range(N_LAGS), 100 * np.array(nz),
           color=[LAG_COLORS[i % len(LAG_COLORS)] for i in range(N_LAGS)], width=0.6)
    ax.set_xticks(range(N_LAGS))
    ax.set_xticklabels([f"δ={d}" for d in DELAY_LAGS], fontsize=6.2)
    _bar_labels(ax, 100 * np.array(nz), fmt="{:.0f}%", horizontal=False, fs=6.0)
    ax.set_ylim(0, 118)
    _finish(ax, "Active mask edges", None, "% of M")
    _panels(axes)
    _safe_tight(fig)
    return fig, pd.DataFrame({"lag": DELAY_LAGS, "active_edge_frac": nz,
                              "edge_strength": DELAY["edge_strength"]})


@figure("F16", "Spatio-temporal flow heatmap by direction", "03_graph")
def fig_flow_heatmap():
    order = np.argsort(-X_total.sum(axis=(0, 2)))[:60]
    show_days = min(5, N_DAYS)
    sl = slice(0, show_days * BINS_PER_DAY)
    fig, axes = new_fig("double", 3.0, 2, 1, sharex=True)
    for ci, (ch, nm) in enumerate([(CH_IN, "Entry"), (CH_OUT, "Exit")]):
        Mx = X_total[sl][:, order, ch].T
        im = axes[ci].imshow(Mx, aspect="auto", cmap=SEQ, interpolation="nearest",
                             norm=LogNorm(vmin=max(Mx.min(), 1), vmax=max(Mx.max(), 2)))
        axes[ci].set_yticks(range(0, len(order), 10))
        axes[ci].set_yticklabels([str(int(STATION_IDS[order[i]]))
                                  for i in range(0, len(order), 10)], fontsize=6.0)
        axes[ci].grid(False)
        _cbar(fig, im, axes[ci], "taps / 10 min")
        _finish(axes[ci], f"{nm} flow — 60 busiest stations", None, "station id")
    tick = np.arange(0, show_days * BINS_PER_DAY, BINS_PER_DAY // 2)
    axes[1].set_xticks(tick)
    axes[1].set_xticklabels([TIME_INDEX[t].strftime("%b%d %H:%M") for t in tick],
                            fontsize=5.8, rotation=35, ha="right")
    axes[1].set_xlabel("time", fontsize=7.5, fontweight="bold", color=MUTED)
    _panels(axes)
    _safe_tight(fig)
    return fig, pd.DataFrame({"station_id": STATION_IDS[order]})


# ── GROUP 04 — EVENTS ────────────────────────────────────────────────────────

@figure("F17", "Station-level robust z-score surge scan", "04_event")
def fig_event_scan():
    z = EVENTS["robust_z"]
    fig, axes = new_fig("double", 2.5, 1, 2, gridspec_kw={"width_ratios": [1.75, 1.0]})
    ax = axes[0]
    for di in range(N_DAYS):
        ax.scatter([DAY_DATES[di]] * N_STATIONS, z[di],
                   s=np.clip(1.5 + z[di] * 1.6, 1.5, 34),
                   c=np.where(z[di] >= 5, M3["red"],
                              np.where(z[di] >= 4, ACCENT, M3["blue100"])),
                   alpha=0.75, lw=0)
    ax.axhline(4, color=ACCENT, ls="--", lw=0.8, label="z = 4 (candidate)")
    ax.axhline(5, color=M3["red"], ls=":", lw=0.8, label="z = 5 (event day)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.legend(loc="upper left", fontsize=6.2)
    _finish(ax, "Station-day peak z-scores", None, "robust z-score")
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(45)
        lbl.set_ha("right")

    cand = EVENTS["candidates"]
    ax = axes[1]
    if len(cand):
        top = cand.head(15).iloc[::-1]
        ax.barh(range(len(top)), top["surge_ratio"], color=ACCENT, height=0.72)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels([f"{r['date'][5:]}  st{r['station_id']}"
                            for _, r in top.iterrows()], fontsize=5.8)
        ax.axvline(1.0, color=MUTED, ls="--", lw=0.7)
        _bar_labels(ax, top["surge_ratio"].to_numpy(), fmt="{:.1f}×", fs=5.8)
        ax.set_xlim(0, top["surge_ratio"].max() * 1.2)
        _finish(ax, "Largest surges", "peak / baseline")
    else:
        ax.set_axis_off()
    _panels(axes)
    _safe_tight(fig)
    return fig, cand


@figure("F18", "Anatomy of the largest detected surge", "04_event")
def fig_event_anatomy():
    cand = EVENTS["candidates"]
    if not len(cand):
        return _skip("F18", "no surge candidates detected")
    top = cand.iloc[0]
    di = TENS["days"].index(top["date"])
    ni = ST_POS[int(top["station_id"])]
    sl = slice(di * BINS_PER_DAY, (di + 1) * BINS_PER_DAY)
    hours = HOUR_OF_BIN[:BINS_PER_DAY]

    same_dow = np.where((DAY_DATES.dayofweek == DAY_DATES[di].dayofweek) &
                        (np.arange(N_DAYS) != di))[0]
    if len(same_dow) == 0:
        same_dow = np.array([i for i in range(N_DAYS) if i != di])
    X4 = X_total.reshape(N_DAYS, BINS_PER_DAY, N_STATIONS, 2)
    base_in = X4[same_dow][:, :, ni, CH_IN]
    base_out = X4[same_dow][:, :, ni, CH_OUT]
    act_in = X_total[sl, ni, CH_IN]
    act_out = X_total[sl, ni, CH_OUT]

    pers_day = X_persona[sl, ni, :, :].sum(axis=1)
    pers_base = (X_persona.reshape(N_DAYS, BINS_PER_DAY, N_STATIONS, 2, K_PERSONA)
                 [same_dow][:, :, ni, :, :].sum(axis=2).mean(axis=0))
    excess = np.maximum(pers_day - pers_base, 0)
    pb = int(top["peak_bin"])
    peak_win = slice(max(pb - 6, 0), min(pb + 7, BINS_PER_DAY))
    attrib = excess[peak_win].sum(axis=0)
    attrib_pct = 100 * attrib / max(attrib.sum(), 1e-9)

    fig, axes = new_fig("double", 2.4, 1, 3,
                        gridspec_kw={"width_ratios": [1.9, 1.15, 0.85]})
    ax = axes[0]
    ax.fill_between(hours, *np.percentile(base_in, [10, 90], axis=0),
                    color=M3["blue600"], alpha=0.16, lw=0, label="entry 10–90 pct")
    ax.plot(hours, np.median(base_in, axis=0), color=M3["blue600"], lw=0.85, ls="--")
    ax.plot(hours, act_in, color=M3["blue800"], lw=1.35, label="entry (event day)")
    ax.plot(hours, np.median(base_out, axis=0), color=ACCENT, lw=0.85, ls="--")
    ax.plot(hours, act_out, color=ACCENT, lw=1.35, label="exit (event day)")
    ax.axvspan(hours[peak_win.start], hours[peak_win.stop - 1], color=M3["red"],
               alpha=0.07, lw=0)
    ax.legend(loc="upper left", fontsize=6.0)
    _hour_axis(ax)
    _finish(ax, f"Station {int(top['station_id'])} on {top['date']} "
                f"(surge ×{top['surge_ratio']:.1f})", "hour", "taps / 10 min")

    ax = axes[1]
    bottom = np.zeros(BINS_PER_DAY)
    for k in range(K_PERSONA):
        ax.fill_between(hours, bottom, bottom + excess[:, k], color=PERSONA_COLORS[k],
                        alpha=0.85, lw=0, label=_pname(k).split("/")[0])
        bottom += excess[:, k]
    ax.legend(fontsize=5.9, loc="upper left")
    _hour_axis(ax)
    _finish(ax, "Excess over baseline, by persona", "hour", "excess taps / 10 min")

    ax = axes[2]
    ax.barh(range(K_PERSONA), attrib_pct, color=PERSONA_COLORS, height=0.62)
    ax.set_yticks(range(K_PERSONA))
    ax.set_yticklabels([_pname(k).split("/")[0] for k in range(K_PERSONA)], fontsize=6.2)
    ax.invert_yaxis()
    _bar_labels(ax, attrib_pct, fmt="{:.1f}%", fs=6.0)
    ax.set_xlim(0, max(attrib_pct) * 1.3 if attrib_pct.max() > 0 else 1)
    _finish(ax, "Peak-window attribution", "% of excess")
    _panels(axes)
    _safe_tight(fig)
    return fig, pd.DataFrame({"hour": hours, "entry": act_in, "exit": act_out})


@figure("F19", "Event days versus normal days: demand and mix", "04_event")
def fig_event_vs_normal():
    ev_set = set(EVENTS["event_days"])
    ev_mask = np.array([d in ev_set for d in TENS["days"]])
    if not ev_mask.any() or ev_mask.all():
        return _skip("F19", "need both event and normal days")
    X4 = X_total.reshape(N_DAYS, BINS_PER_DAY, N_STATIONS, 2)
    hours = HOUR_OF_BIN[:BINS_PER_DAY]

    fig, axes = new_fig("double", 2.3, 1, 3)
    ax = axes[0]
    for m, c, nm in [(~ev_mask, M3["blue600"], "normal"), (ev_mask, ACCENT, "event")]:
        prof = X4[m].sum(axis=(2, 3))
        ax.fill_between(hours, *np.percentile(prof, [25, 75], axis=0),
                        color=c, alpha=0.18, lw=0)
        ax.plot(hours, np.median(prof, axis=0) / 1e3, color=c, lw=1.25, label=nm)
    _hour_axis(ax)
    _finish(ax, "Network diurnal profile", "hour", "k taps / 10 min", legend=True)

    ax = axes[1]
    pk = X4.max(axis=1).sum(axis=(1, 2))
    parts = [pk[~ev_mask], pk[ev_mask]]
    bp = ax.boxplot(parts, widths=0.5, patch_artist=True, showfliers=True,
                    flierprops=dict(marker="o", ms=1.8, mfc=MUTED, mec="none"),
                    medianprops=dict(color=INK, lw=1.0))
    for patch, c in zip(bp["boxes"], [M3["blue100"], "#fdd9b5"]):
        patch.set_facecolor(c)
        patch.set_edgecolor(M3["outline"])
        patch.set_linewidth(0.5)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["normal", "event"], fontsize=6.6)
    _thousands(ax)
    _finish(ax, "Daily network peak", None, "peak taps / 10 min")

    ax = axes[2]
    pmix = X_persona.reshape(N_DAYS, BINS_PER_DAY, N_STATIONS, 2, K_PERSONA).sum(axis=(1, 2, 3))
    pmix = pmix / np.maximum(pmix.sum(axis=1, keepdims=True), 1e-9)
    w = 0.36
    xs = np.arange(K_PERSONA)
    ax.bar(xs - w / 2, 100 * pmix[~ev_mask].mean(axis=0), w,
           color=M3["blue100"], edgecolor=M3["blue600"], lw=0.5, label="normal")
    ax.bar(xs + w / 2, 100 * pmix[ev_mask].mean(axis=0), w, color=ACCENT, label="event")
    ax.set_xticks(xs)
    ax.set_xticklabels([_pname(k).split("/")[0] for k in range(K_PERSONA)],
                       rotation=32, ha="right", fontsize=6.2)
    _finish(ax, "Persona mix shift", None, "share (%)", legend=True)
    _panels(axes)
    _safe_tight(fig)
    return fig, pd.DataFrame({"persona": [_pname(k) for k in range(K_PERSONA)],
                              "normal_pct": 100 * pmix[~ev_mask].mean(axis=0),
                              "event_pct": 100 * pmix[ev_mask].mean(axis=0)})


## 23 · Render the descriptive figure set

In [26]:
EDA_FIGURES = [
    fig_ingestion_audit, fig_daily_ridership, fig_coverage_calendar, fig_card_pareto,
    fig_card_signature_heatmap, fig_demand_concentration, fig_delay_evidence,
    fig_persona_space, fig_persona_profiles, fig_prior_matrix, fig_pi_trajectory,
    fig_context_vector,
    fig_network_graph, fig_mask_panel, fig_delta_adjacency, fig_flow_heatmap,
    fig_event_scan, fig_event_anatomy, fig_event_vs_normal,
]


def render_eda_figures(force: bool = False):
    ok = 0
    for fn in tqdm(EDA_FIGURES, desc="EDA figures"):
        if fn(force=force):
            ok += 1
    log(f"Descriptive figures done: {ok}/{len(EDA_FIGURES)}")
    return ok


render_eda_figures()
figure_status_report()
state_set("eda_figures_ready", True)


EDA figures:   0%|          | 0/19 [00:00<?, ?it/s]

07:47:30 │ INFO    │ ✓ figure  [F01] done — loaded from disk (force=True to redraw)
07:47:30 │ INFO    │ ✓ figure  [F02] done — loaded from disk (force=True to redraw)
07:47:30 │ INFO    │ ✓ figure  [F03] done — loaded from disk (force=True to redraw)
07:47:30 │ INFO    │ ✓ figure  [F04] done — loaded from disk (force=True to redraw)
07:47:30 │ INFO    │ ✓ figure  [F05] done — loaded from disk (force=True to redraw)
07:47:30 │ INFO    │ ✓ figure  [F06] done — loaded from disk (force=True to redraw)
07:47:30 │ INFO    │ ✓ figure  [F07] done — loaded from disk (force=True to redraw)
07:47:30 │ INFO    │ ✓ figure  [F08] done — loaded from disk (force=True to redraw)
07:47:30 │ INFO    │ ✓ figure  [F09] done — loaded from disk (force=True to redraw)
07:47:30 │ INFO    │ ✓ figure  [F10] done — loaded from disk (force=True to redraw)
07:47:30 │ INFO    │ ✓ figure  [F11] done — loaded from disk (force=True to redraw)
07:47:30 │ INFO    │ ✓ figure  [F12] done — loaded from disk (force=True to 

## 24 · Windowing, splits, scalers and batcher

In [27]:
import torch.nn as nn
import torch.nn.functional as Fn

L, H = CFG.seq_len, CFG.horizon


@stage("windows", depends_on=["dataset", "seq_len", "horizon", "train_frac", "val_frac",
                              "calib_frac", "freq_minutes", "service_start", "service_end"],
       ext="pkl", subdir="tensors")
def build_windows() -> dict:
    seg = TENS["day_index"]                              # segment id == day id
    starts = []
    for s in range(T_STEPS - L - H + 1):
        if seg[s] == seg[s + L + H - 1]:
            starts.append(s)
    starts = np.array(starts, dtype=np.int64)

    day_of_start = seg[starts]
    n_days = N_DAYS
    d_tr = int(round(CFG.train_frac * n_days))
    d_va = int(round((CFG.train_frac + CFG.val_frac) * n_days))
    split = np.where(day_of_start < d_tr, 0, np.where(day_of_start < d_va, 1, 2))

    # ── the one protocol change: halve the validation block BY DAY ───────────
    val_days = np.arange(d_tr, d_va)
    n_cal_days = max(1, int(round(CFG.calib_frac * len(val_days)))) if len(val_days) else 0
    calib_days = set(val_days[len(val_days) - n_cal_days:].tolist())
    is_val = split == 1
    is_calib = is_val & np.isin(day_of_start, list(calib_days))
    split = np.where(is_calib, 3, split)                 # 3 == calibration

    return {"starts": starts, "split": split, "day_of_start": day_of_start,
            "day_bounds": (d_tr, d_va), "calib_days": sorted(calib_days)}


WIN = build_windows()
STARTS, SPLIT = WIN["starts"], WIN["split"]
IDX_TR = STARTS[SPLIT == 0]
IDX_VA = STARTS[SPLIT == 1]          # tuning half — early stopping / model selection
IDX_CAL = STARTS[SPLIT == 3]         # calibration half — conformal only, never trained on
IDX_TE = STARTS[SPLIT == 2]
d_tr, d_va = WIN["day_bounds"]

print(f"Windows            : {len(STARTS):,} (L={L} → H={H}; "
      f"{len(STARTS) * 100 / max(T_STEPS, 1):.0f}% of steps are valid starts)")
print(f"  train        {len(IDX_TR):>6,} days 0–{d_tr - 1}      "
      f"{TENS['days'][0]} → {TENS['days'][max(d_tr - 1, 0)]}")
_cal0 = WIN["calib_days"][0] if WIN["calib_days"] else d_va
print(f"  val-tune     {len(IDX_VA):>6,} "
      f"{TENS['days'][min(d_tr, N_DAYS - 1)]} → "
      f"{TENS['days'][min(max(_cal0 - 1, 0), N_DAYS - 1)]}")
print(f"  val-calib    {len(IDX_CAL):>6,} "
      f"{TENS['days'][WIN['calib_days'][0]] if WIN['calib_days'] else '—'} → "
      f"{TENS['days'][WIN['calib_days'][-1]] if WIN['calib_days'] else '—'}"
      f"   ← conformal only")
print(f"  test         {len(IDX_TE):>6,} days {d_va}–{N_DAYS - 1}   "
      f"{TENS['days'][min(d_va, N_DAYS - 1)]} → {TENS['days'][-1]}")
_test_ev = [d for d in TENS["days"][d_va:] if d in set(EVENTS["event_days"])]
print(f"  event days in the test period : {len(_test_ev)} {_test_ev[:8]}")
_cal_ev = [TENS["days"][i] for i in WIN["calib_days"]
           if TENS["days"][i] in set(EVENTS["event_days"])]
print(f"  event days in the calibration set : {len(_cal_ev)} {_cal_ev}")
if len(_cal_ev) == 0:
    print("  ⚠ No event day in the calibration half — the Mondrian event bin will be "
          "empty and ECQ will use the cross-conformal fallback for it.")

# ── scalers (train statistics only) ──────────────────────────────────────────
_tr_mask = np.zeros(T_STEPS, dtype=bool)
_tr_mask[: d_tr * BINS_PER_DAY] = True
MU = X_total[_tr_mask].mean(axis=0).astype(np.float32)                     # [N, 2]
SD = np.maximum(X_total[_tr_mask].std(axis=0).astype(np.float32), 1.0)
SCALE_CARD = SD.mean(axis=1)[:, None, None]                               # [N,1,1]

print(f"\nScalers fitted on training days only.")
for ci, nm in enumerate(CHANNEL_NAMES):
    print(f"  {nm:<6s} μ mean {MU[:, ci].mean():8.2f}    σ mean {SD[:, ci].mean():8.2f}")

# ── torch tensors (kept whole; windows are gathered by index) ───────────────
_dev = DEVICE if CFG.preload_to_gpu and torch.cuda.is_available() else torch.device("cpu")
Xc_t = torch.from_numpy(
    (X_card.astype(np.float32) / SCALE_CARD[None]).reshape(T_STEPS, N_STATIONS, 2 * N_CARD)
).to(_dev)
Xt_t = torch.from_numpy(((X_total.astype(np.float32) - MU[None]) / SD[None])).to(_dev)
Y_t = Xt_t
Zg_t = torch.from_numpy(Z_GLOBAL).to(_dev)
Zn_t = torch.from_numpy(Z_NODE).to(_dev)
Pi_t = torch.from_numpy(PI_T).to(_dev)

# persona-resolved flow — v2: z-scored PER PERSONA CHANNEL with its own
_Xp = X_persona / np.maximum(X_persona.sum(axis=3, keepdims=True), 1e-6) * \
      X_total[:, :, :, None].astype(np.float32)
_MUp = _Xp[_tr_mask].mean(axis=0).astype(np.float32)              # [N, 2, K]
_SDp = np.maximum(_Xp[_tr_mask].std(axis=0).astype(np.float32), 1.0)
Xp_t = torch.from_numpy(((_Xp - _MUp[None]) / _SDp[None]).astype(np.float32)).to(_dev)
del _Xp

MU_t = torch.from_numpy(MU).to(DEVICE)
SD_t = torch.from_numpy(SD).to(DEVICE)
A_TOPO_t = torch.from_numpy(A_TOPO).float().to(DEVICE)
MASK_t = torch.from_numpy(MASK_M).float().to(DEVICE)
A_DELTA_t = torch.from_numpy(A_DELTA).float().to(DEVICE)
PI_PRIOR_t = torch.from_numpy(PI_PRIOR).to(DEVICE)

# train-mean context — reference point for the delay-kernel entropy regulariser
ZG_REF = torch.from_numpy(Z_GLOBAL[_tr_mask].mean(axis=0).astype(np.float32)).to(DEVICE)

print(f"Tensors on {_dev}: X_card {tuple(Xc_t.shape)} "
      f"({Xc_t.element_size() * Xc_t.nelement() / 1e6:.0f} MB), X_total {tuple(Xt_t.shape)}")


class Batcher:
    """Deterministic, index-based mini-batcher."""

    def __init__(self, idx, batch_size, shuffle=True, seed=0):
        self.idx = np.asarray(idx)
        self.bs = batch_size
        self.shuffle = shuffle
        self.seed = seed

    def __len__(self):
        return int(np.ceil(len(self.idx) / self.bs))

    def order(self, epoch):
        if not self.shuffle:
            return self.idx
        return np.random.default_rng(self.seed + 9973 * epoch).permutation(self.idx)

    def batches(self, epoch, skip=0):
        o = self.order(epoch)
        for b in range(skip, len(self)):
            yield b, o[b * self.bs:(b + 1) * self.bs]


def gather(starts_np, need_card=True, need_persona=False):
    """Materialise one batch. Returns a dict of tensors on DEVICE."""
    s = torch.as_tensor(starts_np, dtype=torch.long, device=Xt_t.device)
    ti = s[:, None] + torch.arange(L, device=Xt_t.device)[None, :]          # [B, L]
    to = s[:, None] + L + torch.arange(H, device=Xt_t.device)[None, :]      # [B, H]
    out = {
        "x_total": Xt_t[ti].to(DEVICE, non_blocking=True),                  # [B,L,N,2]
        "y": Y_t[to].to(DEVICE, non_blocking=True),                         # [B,H,N,2]
        "z": Zg_t[ti].to(DEVICE, non_blocking=True),                        # [B,L,dz]
        "znode": Zn_t[ti].to(DEVICE, non_blocking=True),                    # [B,L,N,dn]
        "pi": Pi_t[ti].to(DEVICE, non_blocking=True),                       # [B,L,K]
    }
    if need_card:
        b = Xc_t[ti]                                                        # [B,L,N,2*C]
        out["x_card"] = b.reshape(*b.shape[:3], 2, N_CARD).to(DEVICE, non_blocking=True)
    if need_persona:
        out["x_persona"] = Xp_t[ti].to(DEVICE, non_blocking=True)           # [B,L,N,2,K]
    out["event"] = out["z"][:, :, IDX_EVENT_FEAT]                           # [B,L]
    out["peak"] = out["z"][:, :, IDX_PEAK_FEAT]                             # [B,L]
    return out


DZ = Z_GLOBAL.shape[1]
DN = Z_NODE.shape[2]
_b = gather(IDX_TR[:4], need_persona=True)
print("Batch shapes:", {k: tuple(v.shape) for k, v in _b.items()})
del _b


# ── window-level regime labels (target-time), used by ECQ and by event metrics ─
def window_regime(idx_array):
    """For each window, the regime of its FORECAST TARGET (not its history): whether the target block falls on a flagged event day, and whether...."""
    ev_days = set(EVENTS["event_days"])
    tgt0 = idx_array + L
    is_ev = np.array([TENS["days"][TENS["day_index"][t]] in ev_days for t in tgt0])
    is_pk = IS_PEAK[tgt0] > 0.5
    return is_ev, is_pk


TEST_EVENT_MASK, TEST_PEAK_MASK = window_regime(IDX_TE)
CAL_EVENT_MASK, CAL_PEAK_MASK = window_regime(IDX_CAL)
print(f"\nTest windows on event days : {int(TEST_EVENT_MASK.sum()):,} "
      f"({100 * TEST_EVENT_MASK.mean():.1f}%)   in peak: {int(TEST_PEAK_MASK.sum()):,}")
print(f"Calib windows on event days: {int(CAL_EVENT_MASK.sum()):,} "
      f"({100 * CAL_EVENT_MASK.mean():.1f}%)   in peak: {int(CAL_PEAK_MASK.sum()):,}")


07:47:31 │ INFO    │ ✓ cached  [windows] loaded in 0.3s  (windows__2023__5b7a3a5529.pkl)
Windows            : 5,187 (L=12 → H=6; 84% of steps are valid starts)
  train         3,640 days 0–39      2023-03-01 → 2023-05-14
  val-tune        364 2023-05-15 → 2023-05-18
  val-calib       364 2023-05-19 → 2023-05-22   ← conformal only
  test            819 days 48–56   2023-05-23 → 2023-05-31
  event days in the test period : 2 ['2023-05-27', '2023-05-28']
  event days in the calibration set : 2 ['2023-05-20', '2023-05-21']

Scalers fitted on training days only.
  Entry  μ mean    83.25    σ mean    75.16
  Exit   μ mean    83.53    σ mean    74.03
Tensors on cuda: X_card (6156, 191, 50) (235 MB), X_total (6156, 191, 2)
Batch shapes: {'x_total': (4, 12, 191, 2), 'y': (4, 6, 191, 2), 'z': (4, 12, 8), 'znode': (4, 12, 191, 3), 'pi': (4, 12, 4), 'x_card': (4, 12, 191, 2, 25), 'x_persona': (4, 12, 191, 2, 4), 'event': (4, 12), 'peak': (4, 12)}

Test windows on event days : 182 (22.2%)   in peak

## 25 · CRISP-Net components

In [28]:
class ChannelAttention(nn.Module):
    """avg-pool ⊕ max-pool → shared MLP → sigmoid → per-channel gain."""

    def __init__(self, ch, r=4):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(ch, max(ch // r, 4)), nn.ReLU(),
                                 nn.Linear(max(ch // r, 4), ch))

    def forward(self, x):                              # x: [B, C, L]
        a = self.mlp(x.mean(dim=-1))
        m = self.mlp(x.amax(dim=-1))
        return x * torch.sigmoid(a + m).unsqueeze(-1)


class EnhancedTCN(nn.Module):
    """Multi-scale dilated causal TCN → gate → channel attention → 1×1 residual."""

    def __init__(self, c_in, c_out, dilations=(1, 2, 4, 8), k=3, dropout=0.1):
        super().__init__()
        self.dils = list(dilations)[:3] if len(dilations) >= 3 else list(dilations)
        self.k = k
        self.convs = nn.ModuleList([nn.Conv1d(c_in, c_out, k, dilation=d) for d in self.dils])
        self.gate = nn.Conv1d(c_in, c_out, k, dilation=self.dils[min(1, len(self.dils) - 1)])
        self.catt = ChannelAttention(c_out)
        self.res = nn.Conv1d(c_in, c_out, 1)
        self.norm = nn.BatchNorm1d(c_out)
        self.drop = nn.Dropout(dropout)

    def _causal(self, conv, x):
        pad = (self.k - 1) * conv.dilation[0]
        return conv(Fn.pad(x, (pad, 0)))

    def forward(self, x):                       # x: [B, C_in, L]
        outs = [self._causal(c, x) for c in self.convs]
        g = torch.sigmoid(self._causal(self.gate, x))
        outs[min(1, len(outs) - 1)] = outs[min(1, len(outs) - 1)] * g
        h = Fn.relu(sum(outs))
        h = self.catt(h)
        h = self.norm(h)
        return self.drop(h) + self.res(x)


class DenseGAT(nn.Module):
    """Multi-head graph attention over a dense (masked) adjacency — N≈191, dense is cheap."""

    def __init__(self, d_in, d_out, heads=4, dropout=0.1, concat=True):
        super().__init__()
        self.h, self.dk = heads, d_out // heads if concat else d_out
        self.concat = concat
        self.W = nn.Linear(d_in, self.h * self.dk, bias=False)
        self.a_src = nn.Parameter(torch.empty(1, self.h, 1, self.dk))
        self.a_dst = nn.Parameter(torch.empty(1, self.h, 1, self.dk))
        nn.init.xavier_uniform_(self.a_src)
        nn.init.xavier_uniform_(self.a_dst)
        self.drop = nn.Dropout(dropout)

    def attention(self, h_pool, adj):
        B, N, _ = h_pool.shape
        Wh = self.W(h_pool).view(B, N, self.h, self.dk).permute(0, 2, 1, 3)
        e = ((Wh * self.a_src).sum(-1).unsqueeze(-1) +
             (Wh * self.a_dst).sum(-1).unsqueeze(-2))
        e = Fn.leaky_relu(e, 0.2)
        if adj.dim() == 2:
            adj = adj.unsqueeze(0).expand(B, -1, -1)
        eye = torch.eye(N, device=adj.device, dtype=torch.bool).unsqueeze(0)
        mask = ((adj.abs() > 0) | eye).unsqueeze(1)
        e = e.masked_fill(~mask, -9e15)
        att = torch.softmax(e, dim=-1)
        att = torch.nan_to_num(att, nan=0.0)
        return self.drop(att), Wh

    def forward(self, h_seq, h_pool, adj):
        att, _ = self.attention(h_pool, adj)
        B, Lx, N, _ = h_seq.shape
        Wh = self.W(h_seq).view(B, Lx, N, self.h, self.dk)
        out = torch.einsum("bhij,bljhd->blihd", att, Wh)
        out = out.reshape(B, Lx, N, self.h * self.dk) if self.concat else out.mean(2)
        return Fn.elu(out), att


class AttentionLSTMDecoder(nn.Module):
    """Self-attention LSTM decoder, emitting H × n_out × n_quant per node."""

    def __init__(self, d_in, hidden, horizon, n_out=2, n_quant=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(d_in, hidden, batch_first=True)
        self.attn = nn.Linear(hidden, 1)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout),
                                  nn.Linear(hidden, horizon * n_out * n_quant))
        self.horizon, self.n_out, self.n_quant = horizon, n_out, n_quant

    def forward(self, x):                       # x: [B, N, L, d_in]
        B, N, Lx, D = x.shape
        h, _ = self.lstm(x.reshape(B * N, Lx, D))
        w = torch.softmax(self.attn(h), dim=1)
        ctx = (w * h).sum(dim=1)
        y = self.head(ctx).view(B, N, self.horizon, self.n_out, self.n_quant)
        return y.permute(0, 2, 1, 3, 4).contiguous(), w.view(B, N, Lx)


import math

# ── shared model interface ───────────────────────────────────────────────────
class _Base(nn.Module):
    needs_card = False
    needs_persona = False
    n_quant = 1
    quantiles = (0.5,)
    q_median = 0

    def point(self, y):
        return y[..., self.q_median]

    def interval(self, y):
        return (None, None) if y.shape[-1] < 3 else (y[..., 0], y[..., -1])

    def aux_loss(self):
        dev = next(self.parameters()).device
        return torch.zeros((), device=dev), torch.zeros((), device=dev)

    def forward(self, batch, return_aux=False):
        raise NotImplementedError

# ── PRI — persona channel gates ──────────────────────────────────────────────
class PersonaChannelGate(nn.Module):
    """A learned scalar gate per (direction, persona) input channel, applied to the per-channel z-scored persona-resolved flow tensor before...."""

    def __init__(self, k_persona: int, n_dir: int = 2, init: float = None,
                 enabled: bool = True):
        super().__init__()
        self.enabled = enabled
        self.k, self.d = k_persona, n_dir
        init = CFG.gate_init if init is None else init
        # softplus(raw) = init  →  raw = log(exp(init) - 1); stable for init ~ 1
        raw = math.log(math.expm1(max(init, 1e-3)))
        self.raw = nn.Parameter(torch.full((n_dir, k_persona), float(raw)))

    def gates(self) -> torch.Tensor:
        """→ [n_dir, K], strictly positive."""
        return Fn.softplus(self.raw)

    def forward(self, xp: torch.Tensor) -> torch.Tensor:
        """xp: [B, L, N, D, K] → same shape."""
        if not self.enabled:
            return xp
        return xp * self.gates().view(1, 1, 1, self.d, self.k)

    def l1(self) -> torch.Tensor:
        """Mild sparsity: a persona that carries nothing is free to switch off."""
        if not self.enabled:
            return torch.zeros((), device=self.raw.device)
        return self.gates().abs().mean()


# ── quantile head support ────────────────────────────────────────────────────
def pinball_loss(pred, target, quantiles):
    """pred: [..., n_quant] · target: [...] · quantiles: sequence of tau."""
    tq = torch.as_tensor(quantiles, device=pred.device, dtype=pred.dtype)
    err = target.unsqueeze(-1) - pred
    return torch.maximum(tq * err, (tq - 1.0) * err).mean()


log("CRISP-Net components defined: TDAG-Net backbone (verbatim) · "
    "persona channel gates · tail-pinball quantile heads.")


07:47:32 │ INFO    │ CRISP-Net components defined: TDAG-Net backbone (verbatim) · persona channel gates · tail-pinball quantile heads.


## 26 · CRISP-Net model

In [29]:
class CRISPNet(_Base):
    """Conformal Risk-Interval Station Prediction network."""

    needs_persona = True
    needs_card = False

    def __init__(self, n_nodes, horizon, topo=None, dz=8, dn=3, k_persona=4,
                 ch=None, hid=None, emb=None,
                 use_pri=True, use_gates=True, use_ecq=True, **kw):
        super().__init__()
        ch = CFG.tcn_channels if ch is None else ch
        hid = CFG.hidden_dim if hid is None else hid
        emb = CFG.node_emb_dim if emb is None else emb

        self.use_pri, self.use_gates, self.use_ecq = use_pri, use_gates, use_ecq
        self.k_persona = k_persona
        self.needs_persona = bool(use_pri)

        # ── quantile configuration ───────────────────────────────────────────
        if use_ecq:
            self.quantiles = tuple(CFG.quantiles)
            self.n_quant = len(self.quantiles)
            self.q_median = int(np.argmin([abs(q - 0.5) for q in self.quantiles]))
        else:
            self.quantiles, self.n_quant, self.q_median = (0.5,), 1, 0

        # ── (PRI) input: persona-resolved and gated, or unified flow ─────────
        d_in = (2 * k_persona if use_pri else 2) + dn
        self.gate = PersonaChannelGate(k_persona, 2, enabled=(use_pri and use_gates))
        self.in_proj = nn.Linear(d_in, ch)

        # ── TDAG-Net encoder, unchanged ──────────────────────────────────────
        self.tcn = EnhancedTCN(ch, ch, CFG.tcn_dilations, dropout=CFG.dropout)
        self.E1 = nn.Parameter(torch.randn(n_nodes, emb) * 0.05)
        self.E2 = nn.Parameter(torch.randn(n_nodes, emb) * 0.05)
        self.mix = nn.Parameter(torch.tensor(0.5))
        self.gat_adapt = DenseGAT(ch, hid, CFG.gat_heads, CFG.gat_dropout)
        self.gat_topo = DenseGAT(ch, hid, CFG.gat_heads, CFG.gat_dropout)
        self.register_buffer("A_topo", topo if topo is not None else torch.eye(n_nodes))
        self.fuse_w = nn.Parameter(torch.ones(3))

        # ── joint entry/exit decoder with quantile heads ─────────────────────
        self.decoder = AttentionLSTMDecoder(ch + hid + hid, CFG.lstm_hidden, horizon,
                                            n_out=2, n_quant=self.n_quant,
                                            dropout=CFG.dropout)
        self.horizon = horizon
        self._last = {}

    # ── graph (learned once, frozen at inference — the pilot's verdict) ──────
    def adaptive_adj(self):
        A0 = self.A_topo.abs()
        A0 = A0 / A0.sum(-1, keepdim=True).clamp(min=1e-6)
        Aad = Fn.softmax(Fn.relu(self.E1 @ self.E2.t()), dim=-1)
        w = torch.sigmoid(self.mix)
        return w * Aad + (1 - w) * A0

    def forward(self, batch, return_aux=False):
        if self.use_pri:
            xp = self.gate(batch["x_persona"])               # [B,L,N,2,K]
            B, Lx, N = xp.shape[:3]
            x = xp.reshape(B, Lx, N, -1)                     # [B,L,N,2K]
        else:
            x = batch["x_total"]                             # [B,L,N,2]
            B, Lx, N = x.shape[:3]

        h = self.in_proj(torch.cat([x, batch["znode"]], dim=-1))
        hh = h.permute(0, 2, 3, 1).reshape(B * N, -1, Lx)
        hh = self.tcn(hh)
        h = hh.view(B, N, -1, Lx).permute(0, 3, 1, 2)        # [B,L,N,ch]

        A = self.adaptive_adj()
        g_a, _ = self.gat_adapt(h, h.mean(1), A)
        g_t, _ = self.gat_topo(h, h.mean(1), self.A_topo)
        w = torch.softmax(self.fuse_w, 0)
        fused = torch.cat([w[0] * h, w[1] * g_a, w[2] * g_t], dim=-1)

        y, attn = self.decoder(fused.permute(0, 2, 1, 3))    # [B,H,N,2,Q]

        # quantile monotonicity: enforce lo <= median <= hi by construction, so a
        # crossing can never reach the conformal step and inflate a score.
        if self.n_quant == 3:
            lo, md, hi = y[..., 0], y[..., 1], y[..., 2]
            lo = torch.minimum(lo, md)
            hi = torch.maximum(hi, md)
            y = torch.stack([lo, md, hi], dim=-1)

        if return_aux:
            return y, {"A_adapt": A.detach(), "fusion_w": w.detach(),
                       "gates": self.gate.gates().detach(), "attn": attn.detach()}
        return y

    # ── auxiliary objective ──────────────────────────────────────────────────
    def aux_loss(self):
        """Returns (reg, ent) to match the Trainer's interface."""
        dev = next(self.parameters()).device
        reg = CFG.gate_l1 * self.gate.l1() if self.use_gates else torch.zeros((), device=dev)
        return reg, torch.zeros((), device=dev)

    # ── interpretability ─────────────────────────────────────────────────────
    @torch.no_grad()
    def persona_gates(self) -> pd.DataFrame:
        """The persona attribution table quoted in the Discussion."""
        g = self.gate.gates().cpu().numpy()                   # [2, K]
        rows = []
        for d, dn_ in enumerate(CHANNEL_NAMES):
            for k in range(self.k_persona):
                rows.append({"direction": dn_, "persona": CFG.persona_names[k],
                             "gate": float(g[d, k]),
                             "share_of_direction": float(g[d, k] / max(g[d].sum(), 1e-9))})
        return pd.DataFrame(rows)


log("CRISP-Net defined — TDAG-Net encoder + gated persona-resolved input + "
    f"quantile heads at tau={list(CFG.quantiles)}.")


07:47:32 │ INFO    │ CRISP-Net defined — TDAG-Net encoder + gated persona-resolved input + quantile heads at tau=[0.05, 0.5, 0.95].


## 27 · Baselines — programme predecessors

In [30]:
class TGALSTM(_Base):
    """Paper 1's predecessor: fixed-topology GAT + enhanced TCN + attention-LSTM."""

    def __init__(self, n_nodes, horizon, topo=None, ch=64, hid=64, **kw):
        super().__init__()
        self.in_proj = nn.Linear(2, ch)
        self.tcn = EnhancedTCN(ch, ch, CFG.tcn_dilations, dropout=CFG.dropout)
        self.gat = DenseGAT(ch, hid, CFG.gat_heads, CFG.gat_dropout)
        self.register_buffer("A", topo if topo is not None else torch.eye(n_nodes))
        self.fuse_w = nn.Parameter(torch.ones(2))
        self.decoder = AttentionLSTMDecoder(ch + hid, CFG.lstm_hidden, horizon,
                                            n_out=2, n_quant=1, dropout=CFG.dropout)
        self.horizon = horizon

    def forward(self, batch, return_aux=False):
        x = batch["x_total"]
        B, Lx, N, _ = x.shape
        h = self.in_proj(x)
        hh = h.permute(0, 2, 3, 1).reshape(B * N, -1, Lx)
        hh = self.tcn(hh)
        h = hh.view(B, N, -1, Lx).permute(0, 3, 1, 2)                 # [B,L,N,ch]
        g, _ = self.gat(h, h.mean(1), self.A)
        w = torch.softmax(self.fuse_w, 0)
        fused = torch.cat([w[0] * h, w[1] * g], dim=-1)
        y, _ = self.decoder(fused.permute(0, 2, 1, 3))                # [B,H,N,2,1]
        return (y, {}) if return_aux else y


class TDAGNetRef(_Base):
    """Paper 3 reference implementation, in the configuration its results table reports (adaptive adjacency + fixed-topology branch + gated...."""

    def __init__(self, n_nodes, horizon, topo=None, dz=8, dn=3, ch=64, hid=64,
                 emb=16, **kw):
        super().__init__()
        self.in_proj = nn.Linear(2 + dn, ch)
        self.tcn = EnhancedTCN(ch, ch, CFG.tcn_dilations, dropout=CFG.dropout)
        self.E1 = nn.Parameter(torch.randn(n_nodes, emb) * 0.05)
        self.E2 = nn.Parameter(torch.randn(n_nodes, emb) * 0.05)
        self.mix = nn.Parameter(torch.tensor(0.5))
        self.gat_adapt = DenseGAT(ch, hid, CFG.gat_heads, CFG.gat_dropout)
        self.gat_topo = DenseGAT(ch, hid, CFG.gat_heads, CFG.gat_dropout)
        A0 = topo if topo is not None else torch.eye(n_nodes)
        self.register_buffer("A_topo", A0)
        self.fuse_w = nn.Parameter(torch.ones(3))
        self.decoder = AttentionLSTMDecoder(ch + hid + hid, CFG.lstm_hidden, horizon,
                                            n_out=2, n_quant=1, dropout=CFG.dropout)
        self.horizon = horizon

    def adaptive_adj(self):
        A0 = self.A_topo.abs()
        A0 = A0 / A0.sum(-1, keepdim=True).clamp(min=1e-6)
        Aad = Fn.softmax(Fn.relu(self.E1 @ self.E2.t()), dim=-1)
        w = torch.sigmoid(self.mix)
        return w * Aad + (1 - w) * A0

    def forward(self, batch, return_aux=False):
        x = batch["x_total"]
        znode = batch["znode"]
        B, Lx, N, _ = x.shape
        h = self.in_proj(torch.cat([x, znode], dim=-1))
        hh = h.permute(0, 2, 3, 1).reshape(B * N, -1, Lx)
        hh = self.tcn(hh)
        h = hh.view(B, N, -1, Lx).permute(0, 3, 1, 2)
        A = self.adaptive_adj()
        g_a, _ = self.gat_adapt(h, h.mean(1), A)
        g_t, _ = self.gat_topo(h, h.mean(1), self.A_topo)
        w = torch.softmax(self.fuse_w, 0)
        fused = torch.cat([w[0] * h, w[1] * g_a, w[2] * g_t], dim=-1)
        y, _ = self.decoder(fused.permute(0, 2, 1, 3))
        if return_aux:
            return y, {"A_adapt": A.detach(), "fusion_w": w.detach()}
        return y


class TDAGNetSplitter(_Base):
    """TDAG-Net (Input-Splitter) — the direct comparator for PRI."""

    needs_persona = True

    def __init__(self, n_nodes, horizon, topo=None, dz=8, dn=3, k_persona=4,
                 ch=64, hid=64, emb=16, **kw):
        super().__init__()
        self.k_persona = k_persona
        self.in_proj = nn.Linear(2 * k_persona + dn, ch)     # ← the input split
        self.tcn = EnhancedTCN(ch, ch, CFG.tcn_dilations, dropout=CFG.dropout)
        self.E1 = nn.Parameter(torch.randn(n_nodes, emb) * 0.05)
        self.E2 = nn.Parameter(torch.randn(n_nodes, emb) * 0.05)
        self.mix = nn.Parameter(torch.tensor(0.5))
        self.gat_adapt = DenseGAT(ch, hid, CFG.gat_heads, CFG.gat_dropout)
        self.gat_topo = DenseGAT(ch, hid, CFG.gat_heads, CFG.gat_dropout)
        self.register_buffer("A_topo", topo if topo is not None else torch.eye(n_nodes))
        self.fuse_w = nn.Parameter(torch.ones(3))
        self.decoder = AttentionLSTMDecoder(ch + hid + hid, CFG.lstm_hidden, horizon,
                                            n_out=2, n_quant=1, dropout=CFG.dropout)
        self.horizon = horizon

    def adaptive_adj(self):
        A0 = self.A_topo.abs()
        A0 = A0 / A0.sum(-1, keepdim=True).clamp(min=1e-6)
        Aad = Fn.softmax(Fn.relu(self.E1 @ self.E2.t()), dim=-1)
        w = torch.sigmoid(self.mix)
        return w * Aad + (1 - w) * A0

    def forward(self, batch, return_aux=False):
        xp = batch["x_persona"]                              # [B,L,N,2,K]
        B, Lx, N = xp.shape[:3]
        x = xp.reshape(B, Lx, N, -1)                         # [B,L,N,2K]
        h = self.in_proj(torch.cat([x, batch["znode"]], dim=-1))
        hh = h.permute(0, 2, 3, 1).reshape(B * N, -1, Lx)
        hh = self.tcn(hh)
        h = hh.view(B, N, -1, Lx).permute(0, 3, 1, 2)
        A = self.adaptive_adj()
        g_a, _ = self.gat_adapt(h, h.mean(1), A)
        g_t, _ = self.gat_topo(h, h.mean(1), self.A_topo)
        w = torch.softmax(self.fuse_w, 0)
        fused = torch.cat([w[0] * h, w[1] * g_a, w[2] * g_t], dim=-1)
        y, _ = self.decoder(fused.permute(0, 2, 1, 3))
        return (y, {"A_adapt": A.detach()}) if return_aux else y


log("Baselines defined (v2): TGALSTM · TDAG-Net · TDAG-Net Input-Splitter. "
    "LSTM/TCN/ConvLSTM/SVR-LSTM removed — cited from Paper 3, not rerun.")


07:47:32 │ INFO    │ Baselines defined (v2): TGALSTM · TDAG-Net · TDAG-Net Input-Splitter. LSTM/TCN/ConvLSTM/SVR-LSTM removed — cited from Paper 3, not rerun.


## 28 · Baselines — modern graph networks

In [31]:
def _sym_norm(A):
    A = A + torch.eye(A.shape[0], device=A.device)
    d = A.sum(-1).clamp(min=1e-6).pow(-0.5)
    return d.unsqueeze(-1) * A * d.unsqueeze(-2)


class GraphWaveNet(_Base):
    """Wu et al., IJCAI 2019 — gated dilated TCN + GCN with a learned adaptive adjacency."""

    def __init__(self, n_nodes, horizon, adj, ch=48, layers=3, emb=16, **kw):
        super().__init__()
        self.register_buffer("A", _sym_norm(adj))
        self.E1 = nn.Parameter(torch.randn(n_nodes, emb) * 0.05)
        self.E2 = nn.Parameter(torch.randn(n_nodes, emb) * 0.05)
        self.start = nn.Conv2d(2, ch, 1)
        self.dils = [2 ** i for i in range(layers)]
        self.filt = nn.ModuleList([nn.Conv2d(ch, ch, (1, 2), dilation=(1, d)) for d in self.dils])
        self.gate = nn.ModuleList([nn.Conv2d(ch, ch, (1, 2), dilation=(1, d)) for d in self.dils])
        self.gconv = nn.ModuleList([nn.Conv2d(ch * 3, ch, 1) for _ in self.dils])
        self.skip = nn.ModuleList([nn.Conv2d(ch, ch, 1) for _ in self.dils])
        self.end = nn.Sequential(nn.ReLU(), nn.Conv2d(ch, ch, 1), nn.ReLU(),
                                 nn.Conv2d(ch, horizon * 2, 1))
        self.horizon = horizon

    def forward(self, batch, return_aux=False):
        x = batch["x_total"].permute(0, 3, 2, 1)                      # [B,2,N,L]
        B = x.shape[0]
        Aad = Fn.softmax(Fn.relu(self.E1 @ self.E2.t()), dim=-1)
        h = self.start(x)
        skip = 0
        for i, d in enumerate(self.dils):
            hp = Fn.pad(h, (d, 0))                                    # causal left-pad
            z = torch.tanh(self.filt[i](hp)) * torch.sigmoid(self.gate[i](hp))
            zs = torch.cat([z,
                            torch.einsum("ij,bcjl->bcil", self.A, z),
                            torch.einsum("ij,bcjl->bcil", Aad, z)], dim=1)
            h = self.gconv[i](zs) + h
            skip = skip + self.skip[i](h)
        y = self.end(skip[..., -1:])                                  # [B,H*2,N,1]
        y = y.squeeze(-1).view(B, self.horizon, 2, -1).permute(0, 1, 3, 2).unsqueeze(-1)
        return (y, {}) if return_aux else y


class AGCRN(_Base):
    """Bai et al., NeurIPS 2020 — node-adaptive parameters + data-adaptive graph in a GRU."""

    def __init__(self, n_nodes, horizon, hidden=64, emb=10, cheb_k=2, **kw):
        super().__init__()
        self.E = nn.Parameter(torch.randn(n_nodes, emb) * 0.05)
        self.k = cheb_k
        self.Wz = nn.Parameter(torch.empty(emb, cheb_k, 2 + hidden, hidden))
        self.Wr = nn.Parameter(torch.empty(emb, cheb_k, 2 + hidden, hidden))
        self.Wh = nn.Parameter(torch.empty(emb, cheb_k, 2 + hidden, hidden))
        self.bz = nn.Parameter(torch.zeros(emb, hidden))
        self.br = nn.Parameter(torch.zeros(emb, hidden))
        self.bh = nn.Parameter(torch.zeros(emb, hidden))
        for p in (self.Wz, self.Wr, self.Wh):
            nn.init.xavier_uniform_(p)
        self.head = nn.Linear(hidden, horizon * 2)
        self.hidden, self.horizon = hidden, horizon

    def _agc(self, x, W, b):
        A = Fn.softmax(Fn.relu(self.E @ self.E.t()), dim=1)
        supports = [torch.eye(A.shape[0], device=x.device, dtype=x.dtype), A]
        for _ in range(2, self.k):
            supports.append(2 * A @ supports[-1] - supports[-2])
        S = torch.stack(supports[: self.k])                          # [k,N,N]
        xg = torch.einsum("knm,bmf->bknf", S, x)
        Wn = torch.einsum("ne,ekfh->nkfh", self.E, W)
        bn = self.E @ b
        return torch.einsum("bknf,nkfh->bnh", xg, Wn) + bn

    def forward(self, batch, return_aux=False):
        x = batch["x_total"]
        B, Lx, N, _ = x.shape
        h = torch.zeros(B, N, self.hidden, device=x.device, dtype=x.dtype)
        for t in range(Lx):
            z = torch.sigmoid(self._agc(torch.cat([x[:, t], h], dim=-1), self.Wz, self.bz))
            r = torch.sigmoid(self._agc(torch.cat([x[:, t], h], dim=-1), self.Wr, self.br))
            hc = torch.tanh(self._agc(torch.cat([x[:, t], r * h], dim=-1), self.Wh, self.bh))
            h = z * h + (1 - z) * hc
        y = self.head(h).view(B, N, self.horizon, 2).permute(0, 2, 1, 3).unsqueeze(-1)
        return (y, {}) if return_aux else y


log("Baselines defined (v2): Graph WaveNet · AGCRN. "
    "STGCN/DCRNN removed — cited from Paper 3, not rerun.")


07:47:32 │ INFO    │ Baselines defined (v2): Graph WaveNet · AGCRN. STGCN/DCRNN removed — cited from Paper 3, not rerun.


## 29 · Uncertainty baselines

In [32]:
_Z_ALPHA = 1.6448536269514722      # Φ⁻¹(0.95) → symmetric 90% Gaussian interval


class MCDropoutWrapper(nn.Module):
    """Wraps any point model."""

    needs_card = False
    n_quant = 1
    quantiles = (0.5,)
    q_median = 0

    def __init__(self, base: nn.Module, passes: int = None):
        super().__init__()
        self.base = base
        self.passes = passes or CFG.mc_dropout_passes
        # the wrapper is transparent to the batcher: if the wrapped model wants
        self.needs_persona = getattr(base, "needs_persona", False)
        self.needs_card = getattr(base, "needs_card", False)

    def forward(self, batch, return_aux=False):
        return self.base(batch, return_aux=return_aux)

    def point(self, y):
        return y[..., 0]

    def interval(self, y):
        return None, None

    def aux_loss(self):
        return self.base.aux_loss()

    @torch.no_grad()
    def predict_interval(self, batch):
        """→ (lower, median, upper) each [B,H,N,2], from `passes` stochastic passes."""
        was_training = self.base.training
        self.base.eval()
        for m in self.base.modules():                 # dropout ON, batch-norm OFF
            if isinstance(m, (nn.Dropout, nn.Dropout1d, nn.Dropout2d)):
                m.train()
        samples = []
        for _ in range(self.passes):
            out = self.base(batch)
            samples.append(out[..., 0])
        S = torch.stack(samples)                      # [P,B,H,N,2]
        mu, sd = S.mean(0), S.std(0)
        self.base.train(was_training)
        return mu - _Z_ALPHA * sd, mu, mu + _Z_ALPHA * sd


class DeepEnsemble(nn.Module):
    """M independently seeded members."""

    needs_card = False

    def __init__(self, members: List[nn.Module]):
        super().__init__()
        self.members = nn.ModuleList(members)
        self.needs_persona = any(getattr(m, "needs_persona", False) for m in members)
        self.needs_card = any(getattr(m, "needs_card", False) for m in members)

    @torch.no_grad()
    def predict_interval(self, batch):
        S = torch.stack([m(batch)[..., 0] for m in self.members])     # [M,B,H,N,2]
        mu, sd = S.mean(0), S.std(0, unbiased=len(self.members) > 1)
        return mu - _Z_ALPHA * sd, mu, mu + _Z_ALPHA * sd


log(f"Uncertainty baselines defined — MC-Dropout ({CFG.mc_dropout_passes} passes) · "
    f"Deep Ensemble (M = {CFG.ensemble_members}). "
    "Raw quantile heads come from CRISP-Net itself, not a duplicate run.")


07:47:32 │ INFO    │ Uncertainty baselines defined — MC-Dropout (30 passes) · Deep Ensemble (M = 5). Raw quantile heads come from CRISP-Net itself, not a duplicate run.


## 30 · Model registry and smoke test

In [33]:
def build_model(name: str) -> nn.Module:
    n, hz = N_STATIONS, H
    common = dict(n_nodes=n, horizon=hz)
    crisp = dict(n_nodes=n, horizon=hz, topo=A_TOPO_t, dz=DZ, dn=DN,
                 k_persona=K_PERSONA)
    table = {
        # ── proposed ────────────────────────────────────────────────────────
        "CRISP-Net":            lambda: CRISPNet(**crisp),
        # ── ablations (same code path, one flag each) ───────────────────────
        "CRISP-Net w/o gates":  lambda: CRISPNet(**crisp, use_gates=False),
        "CRISP-Net w/o PRI":    lambda: CRISPNet(**crisp, use_pri=False),
        # ── reused baselines (constructed only if a cache miss forces it) ───
        "Graph WaveNet": lambda: GraphWaveNet(adj=A_TOPO_t, **common),
        "AGCRN":         lambda: AGCRN(**common),
        "TGALSTM":       lambda: TGALSTM(topo=A_TOPO_t, **common),
        "TDAG-Net":      lambda: TDAGNetRef(topo=A_TOPO_t, dz=DZ, dn=DN, **common),
        "TDAG-Net (Input-Splitter)":
                         lambda: TDAGNetSplitter(topo=A_TOPO_t, dz=DZ, dn=DN,
                                                 k_persona=K_PERSONA, **common),
        # ── uncertainty baselines ───────────────────────────────────────────
        "MC-Dropout":    lambda: MCDropoutWrapper(CRISPNet(**crisp, use_ecq=False)),
    }
    for _m in range(CFG.ensemble_members):
        table[f"Ensemble member {_m}"] = lambda: CRISPNet(**crisp, use_ecq=False)
    if name not in table:
        raise KeyError(f"Unknown model '{name}'. Available: {list(table)}")
    return table[name]().to(DEVICE)


NAME_PROPOSED = "CRISP-Net"
ABLATIONS = ["CRISP-Net w/o gates", "CRISP-Net w/o PRI"]
BASELINE_ORDER = ["Graph WaveNet", "AGCRN", "TGALSTM", "TDAG-Net",
                  "TDAG-Net (Input-Splitter)"]
MODEL_ORDER = BASELINE_ORDER + [NAME_PROPOSED]
UNCERTAINTY_MODELS = ["MC-Dropout"]
ENSEMBLE_MEMBERS = [f"Ensemble member {m}" for m in range(CFG.ensemble_members)]
# Quoted from the v2 metrics file if present; never retrained here.
LEGACY_QUOTED = ["Encoder control (no modules)", "CALM-Net"]


def role(name: str) -> str:
    if name == NAME_PROPOSED:
        return "PROPOSED"
    if name in ABLATIONS:
        return "ABLATION"
    if name in UNCERTAINTY_MODELS or name in ENSEMBLE_MEMBERS or name == "Deep Ensemble":
        return "UNCERTAINTY"
    if name in LEGACY_QUOTED:
        return "LEGACY"
    if name in ("TDAG-Net", "TGALSTM", "TDAG-Net (Input-Splitter)"):
        return "PREDECESSOR"
    return "BASELINE"


ROLE_COLOR = {"PROPOSED": ACCENT, "ABLATION": M3["amber"], "PREDECESSOR": M3["purple"],
              "BASELINE": M3["blue600"], "UNCERTAINTY": M3["teal"],
              "LEGACY": M3["outline"]}


def bar_color(name: str) -> str:
    return ROLE_COLOR.get(role(name), M3["blue600"])


def dname(name: str) -> str:
    return (name.replace("TDAG-Net (Input-Splitter)", "TDAG-Net (split)")
                .replace("Encoder control (no modules)", "Encoder ctrl")
                .replace("CRISP-Net w/o ", "− "))


def slug(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_")


def _run_dir_candidates(name: str):
    """Checkpoint directories, newest naming first, then the v2 multi-seed form."""
    base = f"{P['runs']}/{CFG.dataset}__{CFG.exp_version}__"
    cands = [f"{base}{slug(name)}"]
    for sd in (42, 1337, 7):
        cands.append(f"{base}{slug(f'{name} [seed {sd}]')}")
    return cands


def restore_model(name: str = None):
    """Rebuild a trained model from the live Trainer if one survives, else from the newest checkpoint on disk."""
    name = NAME_PROPOSED if name is None else name
    tr = TRAINERS.get(name) if "TRAINERS" in globals() else None
    if tr is not None and getattr(tr, "model", None) is not None:
        with contextlib.suppress(Exception):
            return tr.load_best()
    for d in _run_dir_candidates(name):
        for tag in ("best", "last"):
            ckp = f"{d}/checkpoints/{tag}.pt"
            if not os.path.exists(ckp):
                continue
            try:
                m = build_model(name)
                ck = torch.load(ckp, map_location=DEVICE, weights_only=False)
                m.load_state_dict(ck["model"])
                m.eval()
                log(f"{name} restored from {os.path.basename(d)}/{tag}.pt")
                return m
            except Exception as exc:
                log(f"  could not restore {name} from {tag}.pt: {exc}", "warning")
    return None


# backward-compatible alias used by a few figure functions
_restore_crispnet = restore_model


# ── shape + parameter smoke test ─────────────────────────────────────────────
print(f"{'model':<28s} {'params':>9s} {'Q':>2s}  {'persona':>8s}   forward output")
print("─" * 78)
_probe = gather(IDX_TR[:2], need_persona=True)
_rows = []
for _name in [NAME_PROPOSED] + ABLATIONS + BASELINE_ORDER + UNCERTAINTY_MODELS:
    try:
        _m = build_model(_name)
        _m.eval()
        with torch.no_grad():
            _y = _m(_probe)
        _p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
        _q = int(_y.shape[-1])
        _pe = "yes" if getattr(_m, "needs_persona", False) else "no"
        print(f"{dname(_name):<28s} {_p:>9,d} {_q:>2d}  {_pe:>8s}   {tuple(_y.shape)}")
        _rows.append({"model": _name, "params": _p, "n_quant": _q,
                      "needs_persona": _pe, "out_shape": str(tuple(_y.shape))})
        assert _y.shape[:4] == (2, H, N_STATIONS, 2), f"{_name} shape {_y.shape}"
        del _m, _y
    except Exception as _e:
        print(f"{dname(_name):<28s}  ✗ {type(_e).__name__}: {_e}")
        _rows.append({"model": _name, "params": None, "n_quant": None,
                      "needs_persona": None, "out_shape": f"FAILED: {_e}"})
del _probe
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

save_df(pd.DataFrame(_rows),
        f"{P['results']}/model_registry_{CFG.dataset}_{CFG.exp_version}.csv")
print()
log(f"Registry ready — {len([NAME_PROPOSED] + ABLATIONS + UNCERTAINTY_MODELS)} models to "
    f"train (1 seed) + {CFG.ensemble_members} ensemble members; "
    f"{len(BASELINE_ORDER)} baselines reused from the v2 cache.")


model                           params  Q   persona   forward output
──────────────────────────────────────────────────────────────────────────────
CRISP-Net                      257,313  3       yes   (2, 6, 191, 2, 3)
− gates                        257,313  3       yes   (2, 6, 191, 2, 3)
− PRI                          256,929  3        no   (2, 6, 191, 2, 3)
Graph WaveNet                   65,068  1        no   (2, 6, 191, 2, 1)
AGCRN                          258,050  1        no   (2, 6, 191, 2, 1)
TGALSTM                        210,527  1        no   (2, 6, 191, 2, 1)
TDAG-Net                       253,825  1        no   (2, 6, 191, 2, 1)
TDAG-Net (split)               254,209  1       yes   (2, 6, 191, 2, 1)
MC-Dropout                     254,217  1       yes   (2, 6, 191, 2, 1)

07:47:34 │ INFO    │ Registry ready — 4 models to train (1 seed) + 5 ensemble members; 5 baselines reused from the v2 cache.


## 31 · Metrics

In [34]:
def inverse_transform(arr):
    """[..., N, 2] z-scored → original passenger counts."""
    a = np.asarray(arr)
    if a.ndim == 3:
        return a * SD[None] + MU[None]
    if a.ndim == 4:
        return a * SD[None, None] + MU[None, None]
    raise ValueError(f"unexpected ndim {a.ndim}")


def _safe(x):
    return np.asarray(x, dtype=np.float64)


def metric_suite(y_true, y_pred, event_mask=None, peak_q=None):
    """y_true / y_pred : [S, H, N, 2] in ORIGINAL units."""
    yt, yp = _safe(y_true).ravel(), _safe(y_pred).ravel()
    res = yt - yp
    ss_tot = ((yt - yt.mean()) ** 2).sum()
    out = {
        "RMSE": float(np.sqrt((res ** 2).mean())),
        "MAE": float(np.abs(res).mean()),
        "WMAPE": float(100 * np.abs(res).sum() / max(np.abs(yt).sum(), 1e-9)),
        "R2": float(1 - (res ** 2).sum() / max(ss_tot, 1e-9)),
        "NDE": float(100 * np.sqrt((res ** 2).sum() / max((yt ** 2).sum(), 1e-9))),
    }
    # EPE — accuracy where it matters: top-decile cells inside event windows
    q = CFG.event_peak_quantile if peak_q is None else peak_q
    yt4, yp4 = _safe(y_true), _safe(y_pred)
    if event_mask is not None and np.any(event_mask):
        yt_e, yp_e = yt4[event_mask], yp4[event_mask]
    else:
        yt_e, yp_e = yt4, yp4
    thr = np.quantile(yt_e, q) if yt_e.size else 0.0
    m = yt_e >= max(thr, 1.0)
    out["EPE"] = float(100 * np.abs(yt_e[m] - yp_e[m]).sum() / max(np.abs(yt_e[m]).sum(), 1e-9)) \
        if m.any() else float("nan")
    out["n_peak_cells"] = int(m.sum())
    return out


def metric_suite_by_channel(y_true, y_pred, event_mask=None):
    """The six metrics computed for entry, for exit, and pooled."""
    out = {"overall": metric_suite(y_true, y_pred, event_mask)}
    for ci, nm in enumerate(CHANNEL_NAMES):
        out[nm.lower()] = metric_suite(y_true[..., ci:ci + 1], y_pred[..., ci:ci + 1],
                                       event_mask)
    return out


def per_horizon_metrics(y_true, y_pred):
    """Genuine per-step metrics — never a broadcast of the pooled figure."""
    rows = []
    for h in range(y_true.shape[1]):
        r = metric_suite(y_true[:, h:h + 1], y_pred[:, h:h + 1])
        r["step"] = h + 1
        r["minutes_ahead"] = (h + 1) * CFG.freq_minutes
        for ci, nm in enumerate(CHANNEL_NAMES):
            rc = metric_suite(y_true[:, h:h + 1, :, ci:ci + 1],
                              y_pred[:, h:h + 1, :, ci:ci + 1])
            r[f"RMSE_{nm.lower()}"] = rc["RMSE"]
        rows.append(r)
    return pd.DataFrame(rows)


# ── Uncertainty metrics ──────────────────────────────────────────────────────
def picp(y_true, lo, hi):
    """Prediction Interval Coverage Probability — empirical coverage."""
    y, lo, hi = _safe(y_true), _safe(lo), _safe(hi)
    return float(100 * ((y >= lo) & (y <= hi)).mean())


def mpiw(lo, hi):
    """Mean Prediction Interval Width — sharpness, in passengers per 10 min."""
    return float(np.mean(_safe(hi) - _safe(lo)))


def winkler_score(y_true, lo, hi, alpha=None):
    """Winkler / interval score at level α: width, plus a 2/α penalty for each unit of exceedance."""
    a = CFG.alpha if alpha is None else alpha
    y, lo, hi = _safe(y_true), _safe(lo), _safe(hi)
    w = hi - lo
    w = w + (2.0 / a) * np.clip(lo - y, 0, None)
    w = w + (2.0 / a) * np.clip(y - hi, 0, None)
    return float(w.mean())


def peak_containment_rate(y_true, hi, event_mask=None, q=None):
    """Peak Containment Rate (PCR) — the fraction of true peaks that fall at or below the predicted upper bound."""
    qq = CFG.event_peak_quantile if q is None else q
    yt, hh = _safe(y_true), _safe(hi)
    if event_mask is not None and np.any(event_mask):
        yt, hh = yt[event_mask], hh[event_mask]
    if yt.size == 0:
        return float("nan"), 0
    thr = np.quantile(yt, qq)
    m = yt >= max(thr, 1.0)
    if not m.any():
        return float("nan"), 0
    return float(100 * (yt[m] <= hh[m]).mean()), int(m.sum())


def uncertainty_suite(y_true, lo, hi, event_mask=None, alpha=None):
    a = CFG.alpha if alpha is None else alpha
    pcr, n_pk = peak_containment_rate(y_true, hi, event_mask)
    out = {
        "PICP": picp(y_true, lo, hi),
        "MPIW": mpiw(lo, hi),
        "Winkler": winkler_score(y_true, lo, hi, a),
        "PCR": pcr,
        "n_peak_cells": n_pk,
        "target_PICP": 100 * (1 - a),
    }
    out["coverage_gap"] = out["PICP"] - out["target_PICP"]
    return out


def uncertainty_by_bin(y_true, lo, hi, ev_mask, pk_mask):
    """Coverage and width inside each Mondrian cell — the conditional-validity check."""
    rows = []
    for e in [False, True]:
        for p in [False, True]:
            m = (ev_mask == e) & (pk_mask == p)
            if not m.any():
                rows.append({"event": e, "peak": p, "n_windows": 0, "PICP": np.nan,
                             "MPIW": np.nan, "Winkler": np.nan})
                continue
            rows.append({"event": e, "peak": p, "n_windows": int(m.sum()),
                         "PICP": picp(y_true[m], lo[m], hi[m]),
                         "MPIW": mpiw(lo[m], hi[m]),
                         "Winkler": winkler_score(y_true[m], lo[m], hi[m])})
    return pd.DataFrame(rows)


METRIC_COLS = ["RMSE", "MAE", "WMAPE", "R2", "EPE", "NDE"]
UNC_COLS = ["PICP", "MPIW", "Winkler", "PCR"]
LOWER_IS_BETTER = {"RMSE", "MAE", "WMAPE", "EPE", "NDE", "MPIW", "Winkler"}

print("Metric suite ready.")
print(f"  accuracy    : {METRIC_COLS}  × {{overall, entry, exit}} × {{all, event, normal}}")
print(f"  uncertainty : {UNC_COLS}  at α = {CFG.alpha} "
      f"(target PICP = {100 * (1 - CFG.alpha):.0f}%)")


Metric suite ready.
  accuracy    : ['RMSE', 'MAE', 'WMAPE', 'R2', 'EPE', 'NDE']  × {overall, entry, exit} × {all, event, normal}
  uncertainty : ['PICP', 'MPIW', 'Winkler', 'PCR']  at α = 0.1 (target PICP = 90%)


## 32 · ECQ — event-conditional conformal quantiles

In [35]:
BIN_LABELS = {(False, False): "normal · off-peak", (False, True): "normal · peak",
              (True, False): "event · off-peak", (True, True): "event · peak"}
BIN_ORDER = ["normal · off-peak", "normal · peak", "event · off-peak", "event · peak"]


def conformal_scores(y_true, lo, hi):
    """CQR nonconformity. Negative ⇒ the point sits comfortably inside the interval."""
    return np.maximum(_safe(lo) - _safe(y_true), _safe(y_true) - _safe(hi))


def conformal_quantile(scores, alpha=None):
    """(1−α) conformal quantile with the finite-sample (n+1) correction."""
    a = CFG.alpha if alpha is None else alpha
    s = np.asarray(scores, dtype=np.float64).ravel()
    s = s[np.isfinite(s)]
    n = len(s)
    if n == 0:
        return float("inf"), 0
    k = int(np.ceil((n + 1) * (1 - a)))
    if k > n:
        return float("inf"), n
    return float(np.partition(s, k - 1)[k - 1]), n


class MondrianConformal:
    """Event-conditional conformal quantiles with window-scale bin accounting and optional per-channel calibration."""

    def __init__(self, alpha=None, min_bin_windows=None, per_channel=None):
        self.alpha = CFG.alpha if alpha is None else alpha
        self.min_bin_windows = (CFG.min_bin_windows if min_bin_windows is None
                                else min_bin_windows)
        self.per_channel = (CFG.per_channel_conformal if per_channel is None
                            else per_channel)
        self.n_chan = 1
        self.q_, self.n_, self.n_win_, self.method_ = {}, {}, {}, {}
        self.q_global_ = None
        self.fitted = False

    # ── internal: quantile per channel for a score block [S,H,N,C] ───────────
    def _q_by_channel(self, s):
        if not self.per_channel:
            q, n = conformal_quantile(s, self.alpha)
            return np.full(self.n_chan, q), n
        qs, ns = [], 0
        for c in range(self.n_chan):
            q, n = conformal_quantile(s[..., c], self.alpha)
            qs.append(q)
            ns += n
        return np.asarray(qs, dtype=np.float64), ns

    def fit(self, y_true, lo, hi, ev_mask, pk_mask, fallback_scores=None):
        """y_true, lo, hi : [S, H, N, C] on the CALIBRATION windows, original units ev_mask/pk_mask: [S] bool fallback_scores: pooled score array...."""
        s = conformal_scores(y_true, lo, hi)                 # [S,H,N,C]
        self.n_chan = s.shape[-1] if self.per_channel else 1
        self.q_global_, n_glob = self._q_by_channel(s)

        fb = None
        if fallback_scores is not None:
            f = np.asarray(fallback_scores)
            if self.per_channel and f.ndim >= 1 and f.shape[-1] == self.n_chan:
                fb, _ = self._q_by_channel(f)
            else:
                qf, _ = conformal_quantile(f, self.alpha)
                fb = np.full(self.n_chan, qf)
            if not np.all(np.isfinite(fb)):
                fb = None

        for e in (False, True):
            for p in (False, True):
                key = (bool(e), bool(p))
                m = (ev_mask == e) & (pk_mask == p)
                n_win = int(m.sum())
                self.n_win_[key] = n_win
                q, n = (self._q_by_channel(s[m]) if n_win else
                        (np.full(self.n_chan, np.inf), 0))
                if n_win < self.min_bin_windows or not np.all(np.isfinite(q)):
                    if fb is not None:
                        self.q_[key], self.n_[key] = fb, int(np.size(fallback_scores))
                        self.method_[key] = (f"cross-conformal fallback "
                                             f"(n_win={n_win} < {self.min_bin_windows})")
                        continue
                    self.q_[key], self.n_[key] = self.q_global_, n
                    self.method_[key] = (f"pooled fallback "
                                         f"(n_win={n_win} < {self.min_bin_windows})")
                else:
                    self.q_[key], self.n_[key] = q, n
                    self.method_[key] = f"Mondrian bin (n_win={n_win})"
        self.fitted = True
        return self

    def q_for(self, ev_mask, pk_mask):
        """→ [S, C] correction per window per channel."""
        out = np.tile(np.asarray(self.q_global_, dtype=np.float64),
                      (len(ev_mask), 1))
        for e in (False, True):
            for p in (False, True):
                m = (ev_mask == e) & (pk_mask == p)
                if m.any():
                    out[m] = self.q_[(bool(e), bool(p))]
        return out

    def apply(self, lo, hi, ev_mask, pk_mask):
        if not self.fitted:
            raise RuntimeError("MondrianConformal.apply() before fit()")
        q = self.q_for(ev_mask, pk_mask)                     # [S,C]
        C = _safe(lo).shape[-1]
        q = q if q.shape[-1] == C else np.repeat(q[:, :1], C, axis=1)
        q = q[:, None, None, :]                              # [S,1,1,C]
        return _safe(lo) - q, _safe(hi) + q

    def report(self) -> pd.DataFrame:
        rows = []
        for (e, p), q in self.q_.items():
            r = {"event": e, "peak": p, "bin": BIN_LABELS[(e, p)],
                 "n_windows": self.n_win_[(e, p)], "n_scores": self.n_[(e, p)],
                 "method": self.method_[(e, p)]}
            qa = np.atleast_1d(q)
            if self.per_channel and len(qa) == len(CHANNEL_NAMES):
                for c, nm in enumerate(CHANNEL_NAMES):
                    r[f"q_hat[{nm}]"] = float(qa[c])
            r["q_hat"] = float(np.mean(qa))
            rows.append(r)
        return pd.DataFrame(rows).sort_values(["event", "peak"]).reset_index(drop=True)


def needs_cross_conformal(conf: MondrianConformal) -> bool:
    """True when any Mondrian bin fell back — i.e. the principal risk materialised."""
    return conf.fitted and any("fallback" in m for m in conf.method_.values())


def cross_conformal_scores(model_name: str, k: int = None, force: bool = False):
    """K-fold cross-conformal over the training windows, partitioned BY DAY — the proposal's principal-risk mitigation for a thin event bin."""
    k = CFG.cross_conformal_k if k is None else k
    cache = f"{P['conformal']}/{CFG.dataset}__{CFG.exp_version}__{slug(model_name)}_crossconf.npz"
    if os.path.exists(cache) and not force:
        d = np.load(cache)
        log(f"✓ cached [cross-conformal {model_name}] {d['scores'].shape} scores")
        return d["scores"]

    idx = np.asarray(IDX_TR)
    days = TENS["day_index"][idx + L]
    uniq = np.unique(days)
    if len(uniq) < k:
        log(f"  cross-conformal needs ≥{k} training days, have {len(uniq)} — skipping",
            "warning")
        return None
    day_folds = np.array_split(uniq, k)

    all_scores = []
    for f, fold_days in enumerate(day_folds):
        held = np.isin(days, fold_days)
        fit_idx, out_idx = idx[~held], idx[held]
        if len(fit_idx) < 10 or len(out_idx) < 10:
            continue
        fname = f"{model_name} [xconf fold {f}]"
        log(f"  ▶ cross-conformal fold {f + 1}/{k} — "
            f"fit {len(fit_idx):,} / held-out {len(out_idx):,} windows")
        set_seed(CFG.seed + 1000 + f)
        mdl = build_model(model_name)
        tr = Trainer(fname, mdl)
        tr.fit(fit_idx, IDX_VA)
        tr.load_best()
        pr = tr.predict(out_idx)
        if pr.get("lo") is None:
            log(f"  fold {f} produced no interval — aborting cross-conformal", "warning")
            return None
        sc = conformal_scores(pr["y_true"], pr["lo"], pr["hi"])       # [S,H,N,C]
        all_scores.append(sc.reshape(-1, sc.shape[-1]))
        del mdl, tr
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    if not all_scores:
        return None
    scores = np.concatenate(all_scores, axis=0)
    np.savez_compressed(cache, scores=scores.astype(np.float32))
    log(f"✔ cross-conformal [{model_name}]: {scores.shape[0]:,} pooled held-out "
        f"scores × {scores.shape[1]} channels")
    return scores


# ── the decisive comparison: raw vs pooled vs Mondrian, per bin ──────────────
def _cov_by_bin(y, lo, hi, ev, pk):
    rows = []
    for e in (False, True):
        for p in (False, True):
            m = (ev == e) & (pk == p)
            if not m.any():
                continue
            inside = (_safe(y[m]) >= _safe(lo[m])) & (_safe(y[m]) <= _safe(hi[m]))
            rows.append({"bin": BIN_LABELS[(bool(e), bool(p))],
                         "n_windows": int(m.sum()),
                         "PICP": 100.0 * float(inside.mean()),
                         "MPIW": float(np.mean(_safe(hi[m]) - _safe(lo[m]))),
                         "Winkler": float(winkler_score(y[m], lo[m], hi[m]))})
    return rows


def coverage_comparison(name: str = None) -> pd.DataFrame:
    """The paper's pivotal table."""
    name = NAME_PROPOSED if name is None else name
    cal = CALIB_PRED.get(name)
    te = PREDICTIONS.get(name)
    if cal is None or te is None or te.get("lo_raw") is None:
        log(f"coverage_comparison({name}): raw intervals unavailable", "warning")
        return pd.DataFrame()

    y, lo_raw, hi_raw = te["y_true"], te["lo_raw"], te["hi_raw"]
    ev, pk = TEST_EVENT_MASK, TEST_PEAK_MASK

    out = []
    for r in _cov_by_bin(y, lo_raw, hi_raw, ev, pk):
        out.append({"method": "raw quantile heads", **r})

    pooled = MondrianConformal(min_bin_windows=10 ** 9, per_channel=False).fit(
        cal["y_true"], cal["lo"], cal["hi"], CAL_EVENT_MASK, CAL_PEAK_MASK)
    lo_p, hi_p = pooled.apply(lo_raw, hi_raw, ev, pk)
    for r in _cov_by_bin(y, lo_p, hi_p, ev, pk):
        out.append({"method": "pooled conformal", **r})

    mond = CONFORMAL.get(name)
    if mond is not None:
        lo_m, hi_m = mond.apply(lo_raw, hi_raw, ev, pk)
        for r in _cov_by_bin(y, lo_m, hi_m, ev, pk):
            out.append({"method": "Mondrian ECQ", **r})

    df = pd.DataFrame(out)
    df["method"] = pd.Categorical(
        df["method"], ["raw quantile heads", "pooled conformal", "Mondrian ECQ"], True)
    df["bin"] = pd.Categorical(df["bin"], BIN_ORDER, True)
    df = df.sort_values(["method", "bin"]).reset_index(drop=True)
    df["|PICP − 90|"] = (df["PICP"] - 100 * (1 - CFG.alpha)).abs()
    save_df(df, f"{P['conformal']}/{CFG.dataset}__{CFG.exp_version}__"
                f"{slug(name)}_raw_pooled_mondrian.csv")
    return df


# ── cross-architecture transfer: absolute-residual conformal, no retraining ──
def residual_conformal(name: str, force: bool = False) -> Optional[dict]:
    """Give a POINT forecaster calibrated intervals using |y − ŷ| as the nonconformity score, calibrated Mondrian-style on the same held-out...."""
    cache = f"{P['conformal']}/{CFG.dataset}__{CFG.exp_version}__{slug(name)}_transfer.npz"
    if os.path.exists(cache) and not force:
        d = np.load(cache)
        return {k: d[k] for k in d.files}

    m = restore_model(name)
    if m is None:
        log(f"transfer [{name}]: no checkpoint on disk — skipped", "warning")
        return None
    tr = Trainer(f"{name} [transfer]", m)
    tr.model = m
    pc = tr.predict(IDX_CAL, want_interval=False)
    pt = tr.predict(IDX_TE, want_interval=False)

    # |residual| scores, calibrated per bin (and per channel when enabled)
    s_cal = np.abs(_safe(pc["y_true"]) - _safe(pc["y_pred"]))
    conf = MondrianConformal().fit(pc["y_true"], pc["y_pred"], pc["y_pred"],
                                   CAL_EVENT_MASK, CAL_PEAK_MASK)
    lo, hi = conf.apply(pt["y_pred"], pt["y_pred"], TEST_EVENT_MASK, TEST_PEAK_MASK)

    out = {"y_true": pt["y_true"].astype(np.float32),
           "y_pred": pt["y_pred"].astype(np.float32),
           "lo": lo.astype(np.float32), "hi": hi.astype(np.float32)}
    np.savez_compressed(cache, **out)
    CONFORMAL[f"{name} + ECQ"] = conf
    save_df(conf.report(), f"{P['conformal']}/{CFG.dataset}__{CFG.exp_version}__"
                           f"{slug(name)}_transfer_bins.csv")
    log(f"✔ transfer [{name} + ECQ] — mean |residual| score "
        f"{float(s_cal.mean()):.2f}, mean interval width {float((hi - lo).mean()):.2f}")
    del m, tr
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return out


CALIB_PRED: Dict[str, dict] = {}      # per-model calibration predictions
TRANSFER: Dict[str, dict] = {}        # per-model residual-conformal outputs

print("ECQ ready.")
print(f"  α = {CFG.alpha} → target coverage {100 * (1 - CFG.alpha):.0f}%")
print(f"  Mondrian partition : event_flag × peak/off-peak ({len(BIN_LABELS)} bins)")
print(f"  per-channel        : {CFG.per_channel_conformal} "
      f"({', '.join(CHANNEL_NAMES)} calibrated separately)")
print(f"  calibration set    : {len(IDX_CAL):,} windows "
      f"({int(CAL_EVENT_MASK.sum())} on event days)")
print(f"  min bin size       : {CFG.min_bin_windows} WINDOWS "
      f"(below → {CFG.cross_conformal_k}-fold cross-conformal, then pooled)")
print(f"  transfer targets   : {list(CFG.transfer_models) if CFG.enable_conformal_transfer else '—'}")


ECQ ready.
  α = 0.1 → target coverage 90%
  Mondrian partition : event_flag × peak/off-peak (4 bins)
  per-channel        : True (Entry, Exit calibrated separately)
  calibration set    : 364 windows (182 on event days)
  min bin size       : 40 WINDOWS (below → 5-fold cross-conformal, then pooled)
  transfer targets   : ['AGCRN', 'Graph WaveNet', 'TDAG-Net', 'TDAG-Net (Input-Splitter)']


## 33 · Trainer

In [36]:
def _run_dir(name):
    d = f"{P['runs']}/{CFG.dataset}__{CFG.exp_version}__{slug(name)}"
    Path(f"{d}/checkpoints").mkdir(parents=True, exist_ok=True)
    return d


def _rng_state():
    s = {"python": random.getstate(), "numpy": np.random.get_state(),
         "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        s["cuda"] = torch.cuda.get_rng_state_all()
    return s


def _rng_restore(s):
    with contextlib.suppress(Exception):
        random.setstate(s["python"])
        np.random.set_state(s["numpy"])
        torch.set_rng_state(s["torch"].cpu() if hasattr(s["torch"], "cpu") else s["torch"])
        if torch.cuda.is_available() and "cuda" in s:
            torch.cuda.set_rng_state_all([t.cpu() for t in s["cuda"]])


class Trainer:
    def __init__(self, name, model, cfg=CFG):
        self.name = name
        self.model = model
        self.cfg = cfg
        self.dir = _run_dir(name)
        self.ckpt_last = f"{self.dir}/checkpoints/last.pt"
        self.ckpt_best = f"{self.dir}/checkpoints/best.pt"
        self.done_flag = f"{self.dir}/DONE"
        self.hist_path = f"{self.dir}/history.csv"
        self.state_path = f"{self.dir}/state.json"

        self.opt = torch.optim.Adam(model.parameters(), lr=cfg.lr,
                                    weight_decay=cfg.weight_decay)
        self.sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.opt, mode="min", factor=0.5, patience=cfg.lr_patience, min_lr=1e-6)
        _amp_on = cfg.amp and torch.cuda.is_available()
        try:
            self.scaler = torch.amp.GradScaler("cuda", enabled=_amp_on)
        except (AttributeError, TypeError):             # torch < 2.3
            self.scaler = torch.cuda.amp.GradScaler(enabled=_amp_on)
        self.loss_fn = nn.HuberLoss(delta=cfg.huber_delta)

        self.epoch = 0
        self.batch_in_epoch = 0
        self.best_val = float("inf")
        self.best_epoch = -1
        self.patience = 0
        self.history = []
        self.resume_points = []
        self.needs_card = getattr(model, "needs_card", False)
        self.needs_persona = getattr(model, "needs_persona", False)

        save_json({**asdict(cfg), "model": name}, f"{self.dir}/config.json")

    # ── checkpoint I/O ───────────────────────────────────────────────────────
    def _payload(self):
        return {
            "model": self.model.state_dict(),
            "optimizer": self.opt.state_dict(),
            "scheduler": self.sched.state_dict(),
            "scaler": self.scaler.state_dict(),
            "epoch": self.epoch, "batch_in_epoch": self.batch_in_epoch,
            "best_val": self.best_val, "best_epoch": self.best_epoch,
            "patience": self.patience, "history": self.history,
            "resume_points": self.resume_points,
            "rng": _rng_state(), "config": asdict(self.cfg), "name": self.name,
            "saved_at": datetime.now().isoformat(timespec="seconds"),
        }

    def save(self, path=None, tag="last"):
        path = path or self.ckpt_last
        with atomic_path(path) as tmp:
            torch.save(self._payload(), tmp)
        save_json({"epoch": self.epoch, "batch_in_epoch": self.batch_in_epoch,
                   "best_val": self.best_val, "best_epoch": self.best_epoch,
                   "tag": tag, "updated": datetime.now().isoformat(timespec="seconds")},
                  self.state_path)

    def load(self):
        if not os.path.exists(self.ckpt_last):
            return False
        try:
            ck = torch.load(self.ckpt_last, map_location=DEVICE, weights_only=False)
        except Exception as exc:
            log(f"  checkpoint unreadable ({exc}) — starting fresh", "warning")
            return False
        self.model.load_state_dict(ck["model"])
        self.opt.load_state_dict(ck["optimizer"])
        with contextlib.suppress(Exception):
            self.sched.load_state_dict(ck["scheduler"])
            self.scaler.load_state_dict(ck["scaler"])
        self.epoch = ck["epoch"]
        self.batch_in_epoch = ck.get("batch_in_epoch", 0)
        self.best_val = ck["best_val"]
        self.best_epoch = ck["best_epoch"]
        self.patience = ck["patience"]
        self.history = ck.get("history", [])
        self.resume_points = ck.get("resume_points", [])
        _rng_restore(ck.get("rng", {}))
        log(f"  ↻ resumed [{self.name}] at epoch {self.epoch}, batch {self.batch_in_epoch}, "
            f"best_val={self.best_val:.5f} (epoch {self.best_epoch})")
        self.resume_points.append({"epoch": self.epoch, "batch": self.batch_in_epoch,
                                   "at": datetime.now().isoformat(timespec="seconds")})
        return True

    def _append_history(self, row):
        self.history.append(row)
        df = pd.DataFrame([row])
        header = not os.path.exists(self.hist_path)
        with open(self.hist_path, "a", encoding="utf-8") as f:
            df.to_csv(f, header=header, index=False)

    # ── one forward/backward ─────────────────────────────────────────────────
    def _compute_loss(self, batch):
        out = self.model(batch)
        y = batch["y"]
        point = self.model.point(out)
        loss = self.loss_fn(point, y)                       # Huber on the median
        parts = {"huber": float(loss.detach())}
        if out.shape[-1] > 1:
            # v2: pinball on the TAIL quantiles only. The old objective
            qs = list(self.model.quantiles)
            tail = [i for i, q in enumerate(qs) if abs(q - 0.5) > 1e-6]
            if tail:
                pin = pinball_loss(out[..., tail], y, [qs[i] for i in tail])
                loss = loss + self.cfg.pinball_weight * pin
                parts["pinball"] = float(pin.detach())
        reg, ent = self.model.aux_loss()
        loss = (loss + self.cfg.aux_persona_weight * reg
                     + self.cfg.aux_entropy_weight * ent)
        parts["film_reg"] = float(reg.detach())
        parts["kernel_ent"] = float(ent.detach())
        return loss, parts

    def _step(self, batch, train=True):
        with torch.autocast("cuda", enabled=self.cfg.amp and torch.cuda.is_available(),
                            dtype=torch.float16):
            loss, parts = self._compute_loss(batch)
        if train:
            self.opt.zero_grad(set_to_none=True)
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            gn = torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
            self.scaler.step(self.opt)
            self.scaler.update()
            return float(loss.detach()), float(gn), parts
        return float(loss.detach()), 0.0, parts

    @torch.no_grad()
    def evaluate(self, idx, bs=None):
        self.model.eval()
        bs = bs or self.cfg.batch_size
        tot, n = 0.0, 0
        for i in range(0, len(idx), bs):
            b = gather(idx[i:i + bs], need_card=self.needs_card,
                       need_persona=self.needs_persona)
            l, _, _ = self._step(b, train=False)
            tot += l * len(idx[i:i + bs])
            n += len(idx[i:i + bs])
        return tot / max(n, 1)

    # ── main loop ────────────────────────────────────────────────────────────
    def fit(self, idx_tr, idx_va):
        if os.path.exists(self.done_flag):
            log(f"✓ trained [{self.name}] — DONE marker present, skipping")
            self.load()
            return self
        self.load()

        bat = Batcher(idx_tr, self.cfg.batch_size, shuffle=True, seed=self.cfg.seed)
        last_save = time.time()
        log(f"▶ training [{self.name}] — {len(idx_tr):,} train / {len(idx_va):,} tune "
            f"windows, {len(bat)} batches/epoch, from epoch {self.epoch}")

        try:
            while self.epoch < self.cfg.epochs:
                self.model.train()
                t0 = time.time()
                run_loss, run_n, gnorm = 0.0, 0, 0.0
                skip = self.batch_in_epoch
                if skip:
                    log(f"  ↻ skipping {skip} already-completed batches of epoch {self.epoch}")
                for bidx, sel in bat.batches(self.epoch, skip=skip):
                    b = gather(sel, need_card=self.needs_card,
                               need_persona=self.needs_persona)
                    l, gn, _ = self._step(b, train=True)
                    run_loss += l * len(sel)
                    run_n += len(sel)
                    gnorm = max(gnorm, gn)
                    self.batch_in_epoch = bidx + 1
                    if (time.time() - last_save) > self.cfg.ckpt_every_minutes * 60:
                        self.save(tag="mid-epoch")
                        last_save = time.time()
                        log(f"    ⯑ autosave @ epoch {self.epoch} batch {self.batch_in_epoch}")
                tr_loss = run_loss / max(run_n, 1)
                va_loss = self.evaluate(idx_va)
                self.sched.step(va_loss)
                lr_now = self.opt.param_groups[0]["lr"]

                improved = va_loss < self.best_val - 1e-6
                if improved:
                    self.best_val, self.best_epoch, self.patience = va_loss, self.epoch, 0
                    with atomic_path(self.ckpt_best) as tmp:
                        torch.save(self._payload(), tmp)
                else:
                    self.patience += 1

                self._append_history({
                    "epoch": self.epoch, "train_loss": tr_loss, "val_loss": va_loss,
                    "lr": lr_now, "grad_norm_max": gnorm, "secs": round(time.time() - t0, 1),
                    "improved": improved, "patience": self.patience,
                    "timestamp": datetime.now().isoformat(timespec="seconds"),
                })
                log(f"  e{self.epoch:>3d} train {tr_loss:.5f} val {va_loss:.5f} "
                    f"{' ★' if improved else '   '} lr {lr_now:.2e} "
                    f"{time.time() - t0:.0f}s patience {self.patience}/"
                    f"{self.cfg.early_stop_patience}")

                self.epoch += 1
                self.batch_in_epoch = 0
                if self.epoch % self.cfg.ckpt_every_epochs == 0:
                    self.save(tag="epoch")
                    last_save = time.time()

                if self.patience >= self.cfg.early_stop_patience:
                    log(f" ⯑ early stop at epoch {self.epoch} "
                        f"(best {self.best_val:.5f} @ epoch {self.best_epoch})")
                    break
        except KeyboardInterrupt:
            self.save(tag="interrupt")
            log(f" ⯑ interrupted — checkpoint written at epoch {self.epoch} "
                f"batch {self.batch_in_epoch}. Re-run this cell to continue.")
            raise

        self.save(tag="final")
        Path(self.done_flag).write_text(
            json.dumps({"finished": datetime.now().isoformat(timespec="seconds"),
                        "best_val": self.best_val, "best_epoch": self.best_epoch,
                        "epochs_run": self.epoch}, indent=2))
        log(f"✔ finished [{self.name}] best_val={self.best_val:.5f} @ epoch {self.best_epoch}")
        return self

    def load_best(self):
        p = self.ckpt_best if os.path.exists(self.ckpt_best) else self.ckpt_last
        if os.path.exists(p):
            ck = torch.load(p, map_location=DEVICE, weights_only=False)
            self.model.load_state_dict(ck["model"])
            return self.model

    # ── inference ────────────────────────────────────────────────────────────
    @torch.no_grad()
    def predict(self, idx, bs=None, want_interval=True):
        """→ dict of arrays in ORIGINAL units: y_true [S,H,N,2] · y_pred [S,H,N,2] · lo/hi [S,H,N,2] (None if unavailable)."""
        self.model.eval()
        bs = bs or self.cfg.batch_size
        yt, yp, ylo, yhi = [], [], [], []
        has_int = True
        for i in range(0, len(idx), bs):
            b = gather(idx[i:i + bs], need_card=self.needs_card,
                       need_persona=self.needs_persona)
            if want_interval and hasattr(self.model, "predict_interval"):
                lo, mu, hi = self.model.predict_interval(b)
            else:
                out = self.model(b)
                mu = self.model.point(out)
                lo, hi = self.model.interval(out)
            yt.append(b["y"].float().cpu().numpy())
            yp.append(mu.float().cpu().numpy())
            if lo is None or hi is None:
                has_int = False
            else:
                ylo.append(lo.float().cpu().numpy())
                yhi.append(hi.float().cpu().numpy())
        out = {
            "y_true": inverse_transform(np.concatenate(yt)),
            "y_pred": inverse_transform(np.concatenate(yp)),
        }
        out["lo"] = inverse_transform(np.concatenate(ylo)) if has_int and ylo else None
        out["hi"] = inverse_transform(np.concatenate(yhi)) if has_int and yhi else None
        return out


log("Trainer ready (v2) — resumable to the batch, Huber + tail-pinball objective, "
    "real kernel-entropy penalty.")


07:47:34 │ INFO    │ Trainer ready (v2) — resumable to the batch, Huber + tail-pinball objective, real kernel-entropy penalty.


## 34 · Experiment runner

In [37]:
EXPERIMENT_QUEUE = [NAME_PROPOSED] + ABLATIONS + BASELINE_ORDER + UNCERTAINTY_MODELS
if CFG.quick_test:
    EXPERIMENT_QUEUE = ["TDAG-Net", NAME_PROPOSED]

RESULTS: Dict[str, dict] = {}
TRAINERS: Dict[str, Trainer] = {}
PREDICTIONS: Dict[str, dict] = {}
CONFORMAL: Dict[str, MondrianConformal] = {}
# CALIB_PRED / TRANSFER are created in cell [31]; do not reset them here.


def _pred_path(name):
    return f"{P['preds']}/{CFG.dataset}__{CFG.exp_version}__{slug(name)}.npz"


def _metric_path(name):
    return f"{P['metrics']}/{CFG.dataset}__{CFG.exp_version}__{slug(name)}.json"


def _calib_path(name):
    return f"{P['conformal']}/{CFG.dataset}__{CFG.exp_version}__{slug(name)}_calib.npz"


def _as_df(obj):
    if obj is None:
        return pd.DataFrame()
    if isinstance(obj, pd.DataFrame):
        return obj
    try:
        return pd.DataFrame(obj)
    except Exception:
        return pd.DataFrame()


def score_model(name, pred_te, conf: Optional[MondrianConformal] = None) -> dict:
    """Assemble the full metric bundle for one model."""
    yt, yp = pred_te["y_true"], pred_te["y_pred"]
    ev, pk = TEST_EVENT_MASK, TEST_PEAK_MASK
    res = {
        "model": name,
        "role": role(name),
        "n_seeds": 1,
        "seeds": [int(CFG.seed)],
        "overall": metric_suite(yt, yp, ev),
        "event": metric_suite(yt[ev], yp[ev], np.ones(int(ev.sum()), dtype=bool)),
        "normal": metric_suite(yt[~ev], yp[~ev]) if (~ev).any() else {},
        "per_horizon": per_horizon_metrics(yt, yp).to_dict("records"),
    }
    for ci, nm in enumerate(CHANNEL_NAMES):
        res[f"overall_{nm.lower()}"] = metric_suite(yt[..., ci:ci + 1], yp[..., ci:ci + 1])
        if ev.any():
            res[f"event_{nm.lower()}"] = metric_suite(
                yt[ev][..., ci:ci + 1], yp[ev][..., ci:ci + 1],
                np.ones(int(ev.sum()), dtype=bool))

    lo, hi = pred_te.get("lo_raw", pred_te.get("lo")), pred_te.get("hi_raw", pred_te.get("hi"))
    if lo is not None and hi is not None:
        res["uncertainty_raw"] = uncertainty_suite(yt, lo, hi, ev)
        res["uncertainty_raw_bins"] = uncertainty_by_bin(yt, lo, hi, ev, pk).to_dict("records")
        if conf is not None and conf.fitted:
            clo, chi = conf.apply(lo, hi, ev, pk)
            res["uncertainty"] = uncertainty_suite(yt, clo, chi, ev)
            res["uncertainty_bins"] = uncertainty_by_bin(yt, clo, chi, ev, pk).to_dict("records")
            res["conformal_report"] = conf.report().to_dict("records")
        else:
            res["uncertainty"] = res["uncertainty_raw"]
            res["uncertainty_bins"] = res["uncertainty_raw_bins"]
    return res


def _fit_conformal(name, pred_cal):
    """Mondrian fit + the cross-conformal fallback when a bin is under-sized."""
    conf = MondrianConformal().fit(pred_cal["y_true"], pred_cal["lo"], pred_cal["hi"],
                                   CAL_EVENT_MASK, CAL_PEAK_MASK)
    if needs_cross_conformal(conf) and CFG.enable_cross_conformal:
        thin = [BIN_LABELS[k] for k, m in conf.method_.items() if "fallback" in m]
        log(f" ⚠ under-sized Mondrian bin(s) for [{name}] {thin} — "
            f"invoking {CFG.cross_conformal_k}-fold cross-conformal")
        xs = cross_conformal_scores(name)
        if xs is not None:
            conf = MondrianConformal().fit(
                pred_cal["y_true"], pred_cal["lo"], pred_cal["hi"],
                CAL_EVENT_MASK, CAL_PEAK_MASK, fallback_scores=xs)
    return conf


def run_experiment(name, force=False) -> Optional[dict]:
    pred_f, met_f = _pred_path(name), _metric_path(name)

    # ── fast path: cached (this is how the four v2 baselines come back) ──────
    if os.path.exists(pred_f) and os.path.exists(met_f) and not force:
        d = np.load(pred_f, allow_pickle=False)
        PREDICTIONS[name] = {k: (d[k] if k in d.files else None)
                             for k in ["y_true", "y_pred", "lo", "hi", "lo_raw", "hi_raw"]}
        RESULTS[name] = load_json(met_f)
        cf = _calib_path(name)
        if os.path.exists(cf):
            c = np.load(cf)
            CALIB_PRED[name] = {k: c[k] for k in c.files}
            with contextlib.suppress(Exception):
                CONFORMAL[name] = _fit_conformal(name, CALIB_PRED[name])
        ns = RESULTS[name].get("n_seeds", 1)
        log(f"✓ cached [{name}] predictions + metrics loaded"
            f"{f' ({ns} seeds, from v2)' if ns > 1 else ''}")
        return RESULTS[name]

    set_seed(CFG.seed)
    model = build_model(name)
    tr = Trainer(name, model)
    TRAINERS[name] = tr
    tr.fit(IDX_TR, IDX_VA)
    tr.load_best()
    pred_te = tr.predict(IDX_TE)

    # raw = pre-conformal heads; kept for the decisive comparison in cell [34]
    if pred_te.get("lo") is not None:
        pred_te["lo_raw"] = pred_te["lo"].copy()
        pred_te["hi_raw"] = pred_te["hi"].copy()

    conf = None
    if pred_te.get("lo") is not None and len(IDX_CAL):
        pred_cal = tr.predict(IDX_CAL)
        CALIB_PRED[name] = pred_cal
        conf = _fit_conformal(name, pred_cal)
        CONFORMAL[name] = conf
        save_df(conf.report(),
                f"{P['conformal']}/{CFG.dataset}__{CFG.exp_version}__{slug(name)}_bins.csv")
        np.savez_compressed(_calib_path(name), y_true=pred_cal["y_true"],
                            lo=pred_cal["lo"], hi=pred_cal["hi"],
                            y_pred=pred_cal["y_pred"])
        # the calibrated intervals become the model's reported intervals
        pred_te["lo"], pred_te["hi"] = conf.apply(
            pred_te["lo_raw"], pred_te["hi_raw"], TEST_EVENT_MASK, TEST_PEAK_MASK)

    res = score_model(name, pred_te, conf)
    res["params"] = int(sum(p.numel() for p in model.parameters() if p.requires_grad))
    res["epochs"] = int(tr.epoch)
    res["best_val"] = float(tr.best_val)

    np.savez_compressed(pred_f, **{k: v.astype(np.float32)
                                   for k, v in pred_te.items() if v is not None})
    save_json(res, met_f)
    PREDICTIONS[name] = pred_te
    RESULTS[name] = res

    TRAINERS.pop(name, None)
    del model, tr
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return res


# ── pre-flight ────────────────────────────────────────────────────────────────
_queue_full = EXPERIMENT_QUEUE + ENSEMBLE_MEMBERS
_cached = [n for n in _queue_full if os.path.exists(_pred_path(n)) or
           os.path.exists(f"{P['runs']}/{CFG.dataset}__{CFG.exp_version}__{slug(n)}/DONE")]
_pending = [n for n in _queue_full if n not in _cached]
print(f"Test windows: {len(IDX_TE):,} │ on event days: {int(TEST_EVENT_MASK.sum())} "
      f"({100 * TEST_EVENT_MASK.mean():.1f}%) │ in peak: {int(TEST_PEAK_MASK.sum())}")
print(f"Queue: {len(EXPERIMENT_QUEUE)} experiments + {CFG.ensemble_members} ensemble "
      f"members, {len(CFG.report_seeds)} seed each")
print(f"  already on disk : {len(_cached)}  {[dname(n) for n in _cached]}")
print(f"  to train        : {len(_pending)}  {[dname(n) for n in _pending] or 'none'}\n")


for _name in EXPERIMENT_QUEUE:
    try:
        _r = run_experiment(_name)
        if _r:
            _m = _r["overall"]
            _std = (_r.get("overall_std") or {}).get("RMSE")
            _line = f"  {dname(_name):<26s} RMSE {_m.get('RMSE', np.nan):8.3f}"
            _line += f" ± {_std:5.3f}" if _std is not None else " " * 8
            _line += (f"  MAE {_m.get('MAE', np.nan):7.3f} "
                      f"WMAPE {_m.get('WMAPE', np.nan):6.2f}% "
                      f"R² {_m.get('R2', np.nan):.4f}")
            if "uncertainty" in _r:
                _u = _r["uncertainty"]
                _line += (f" │ PICP {_u.get('PICP', np.nan):5.1f}% "
                          f"MPIW {_u.get('MPIW', np.nan):6.2f}")
            print(_line)
    except Exception as _exc:
        log(f"✗ experiment [{_name}] failed: {type(_exc).__name__}: {_exc}", "error")
        log(traceback.format_exc(limit=5), "error")


# ── deep ensemble: train the members, then combine ──────────────────────────
def run_deep_ensemble(force=False):
    name = "Deep Ensemble"
    pred_f, met_f = _pred_path(name), _metric_path(name)
    if os.path.exists(pred_f) and os.path.exists(met_f) and not force:
        d = np.load(pred_f)
        PREDICTIONS[name] = {k: (d[k] if k in d.files else None)
                             for k in ["y_true", "y_pred", "lo", "hi", "lo_raw", "hi_raw"]}
        RESULTS[name] = load_json(met_f)
        log(f"✓ cached [{name}] (M={RESULTS[name].get('n_members', '?')})")
        return RESULTS[name]

    members = []
    for m in range(CFG.ensemble_members):
        mname = f"Ensemble member {m}"
        set_seed(CFG.seed + 100 + m)
        mdl = build_model(mname)
        t = Trainer(mname, mdl)
        t.fit(IDX_TR, IDX_VA)
        t.load_best()
        members.append(t.model.eval())
    ens = DeepEnsemble(members).to(DEVICE)
    log(f"Deep Ensemble assembled from M={len(members)} members")

    def _predict(idx):
        yt, lo_, mu_, hi_ = [], [], [], []
        with torch.no_grad():
            for i in range(0, len(idx), CFG.batch_size):
                b = gather(idx[i:i + CFG.batch_size], need_card=False, need_persona=True)
                lo, mu, hi = ens.predict_interval(b)
                yt.append(b["y"].float().cpu().numpy())
                lo_.append(lo.float().cpu().numpy())
                mu_.append(mu.float().cpu().numpy())
                hi_.append(hi.float().cpu().numpy())
        return {"y_true": inverse_transform(np.concatenate(yt)),
                "y_pred": inverse_transform(np.concatenate(mu_)),
                "lo": inverse_transform(np.concatenate(lo_)),
                "hi": inverse_transform(np.concatenate(hi_))}

    pred_te = _predict(IDX_TE)
    pred_te["lo_raw"], pred_te["hi_raw"] = pred_te["lo"].copy(), pred_te["hi"].copy()
    pred_cal = _predict(IDX_CAL)
    CALIB_PRED[name] = pred_cal
    conf = _fit_conformal(name, pred_cal)
    CONFORMAL[name] = conf
    pred_te["lo"], pred_te["hi"] = conf.apply(pred_te["lo_raw"], pred_te["hi_raw"],
                                              TEST_EVENT_MASK, TEST_PEAK_MASK)
    res = score_model(name, pred_te, conf)
    res["params"] = int(sum(p.numel() for p in ens.parameters() if p.requires_grad))
    res["epochs"] = CFG.epochs
    res["best_val"] = float("nan")
    res["n_members"] = int(len(members))
    np.savez_compressed(pred_f, **{k: v.astype(np.float32)
                                   for k, v in pred_te.items() if v is not None})
    save_json(res, met_f)
    PREDICTIONS[name] = pred_te
    RESULTS[name] = res
    del ens, members
    gc.collect()
    return res


if not CFG.quick_test:
    try:
        _r = run_deep_ensemble()
        if _r:
            print(f"  {'Deep Ensemble':<26s} RMSE {_r['overall']['RMSE']:8.3f}          "
                  f"│ PICP {_r['uncertainty']['PICP']:5.1f}% "
                  f"MPIW {_r['uncertainty']['MPIW']:6.2f}   "
                  f"(M={_r.get('n_members', CFG.ensemble_members)})")
    except Exception as _exc:
        log(f"✗ deep ensemble failed: {type(_exc).__name__}: {_exc}", "error")


# ── contribution 2: conformal transfer across architectures (no training) ────
if CFG.enable_conformal_transfer:
    print("\nConformal transfer — absolute-residual ECQ on point baselines "
          "(inference only):")
    for _name in CFG.transfer_models:
        if _name not in RESULTS:
            continue
        try:
            _t = residual_conformal(_name)
            if _t is None:
                continue
            TRANSFER[_name] = _t
            _u = uncertainty_suite(_t["y_true"], _t["lo"], _t["hi"], TEST_EVENT_MASK)
            print(f"  {dname(_name) + ' + ECQ':<26s} PICP {_u['PICP']:5.1f}% "
                  f"MPIW {_u['MPIW']:6.2f}  Winkler {_u['Winkler']:7.2f}  "
                  f"PCR {_u['PCR']:5.1f}%")
        except Exception as _exc:
            log(f"✗ transfer [{_name}] failed: {type(_exc).__name__}: {_exc}", "error")

state_set("experiments_done", list(RESULTS.keys()))
print(f"\n{len(RESULTS)} experiments scored · {len(TRANSFER)} baselines transferred.")


Test windows: 819 │ on event days: 182 (22.2%) │ in peak: 216
Queue: 9 experiments + 5 ensemble members, 1 seed each
  already on disk : 14  ['CRISP-Net', '− gates', '− PRI', 'Graph WaveNet', 'AGCRN', 'TGALSTM', 'TDAG-Net', 'TDAG-Net (split)', 'MC-Dropout', 'Ensemble member 0', 'Ensemble member 1', 'Ensemble member 2', 'Ensemble member 3', 'Ensemble member 4']
  to train        : 0  none

07:47:41 │ INFO    │ ✓ cached [CRISP-Net] predictions + metrics loaded
  CRISP-Net                  RMSE   30.896          MAE  13.645 WMAPE  18.59% R² 0.9522 │ PICP  90.0% MPIW  61.45
07:47:45 │ INFO    │ ✓ cached [CRISP-Net w/o gates] predictions + metrics loaded
  − gates                    RMSE   30.652          MAE  13.661 WMAPE  18.61% R² 0.9530 │ PICP  90.1% MPIW  62.87
07:47:50 │ INFO    │ ✓ cached [CRISP-Net w/o PRI] predictions + metrics loaded
  − PRI                      RMSE   32.383          MAE  14.136 WMAPE  19.26% R² 0.9475 │ PICP  89.9% MPIW  65.54
07:47:51 │ INFO    │ ✓ cached [Grap

## 35 · Deep ensemble

In [52]:
def run_deep_ensemble(force=False):
    name = "Deep Ensemble"
    pred_f, met_f = _pred_path(name), _metric_path(name)
    if os.path.exists(pred_f) and os.path.exists(met_f) and not force:
        d = np.load(pred_f)
        PREDICTIONS[name] = {k: (d[k] if k in d.files else None)
                             for k in ["y_true", "y_pred", "lo", "hi", "lo_raw", "hi_raw"]}
        RESULTS[name] = load_json(met_f)
        log(f"✓ cached [{name}] (M={RESULTS[name].get('n_members', '?')})")
        return RESULTS[name]

    members = []
    for m in range(CFG.ensemble_members):
        mname = f"Ensemble member {m}"
        set_seed(CFG.seed + 100 + m)
        mdl = build_model(mname)
        t = Trainer(mname, mdl)
        t.fit(IDX_TR, IDX_VA)          # cached → resumes and returns immediately
        t.load_best()
        members.append(t.model.eval())
    ens = DeepEnsemble(members).to(DEVICE)
    log(f"Deep Ensemble assembled from M={len(members)} members")

    def _predict(idx):
        yt, lo_, mu_, hi_ = [], [], [], []
        with torch.no_grad():
            for i in range(0, len(idx), CFG.batch_size):
                b = gather(idx[i:i + CFG.batch_size], need_card=False, need_persona=True)
                lo, mu, hi = ens.predict_interval(b)
                yt.append(b["y"].float().cpu().numpy())
                lo_.append(lo.float().cpu().numpy())
                mu_.append(mu.float().cpu().numpy())
                hi_.append(hi.float().cpu().numpy())
        return {"y_true": inverse_transform(np.concatenate(yt)),
                "y_pred": inverse_transform(np.concatenate(mu_)),
                "lo": inverse_transform(np.concatenate(lo_)),
                "hi": inverse_transform(np.concatenate(hi_))}

    pred_te = _predict(IDX_TE)
    pred_te["lo_raw"], pred_te["hi_raw"] = pred_te["lo"].copy(), pred_te["hi"].copy()
    pred_cal = _predict(IDX_CAL)
    CALIB_PRED[name] = pred_cal

    # pooled fallback only — no cross-conformal for an ensemble (see header)
    conf = MondrianConformal().fit(pred_cal["y_true"], pred_cal["lo"], pred_cal["hi"],
                                   CAL_EVENT_MASK, CAL_PEAK_MASK)
    CONFORMAL[name] = conf
    if needs_cross_conformal(conf):
        thin = [BIN_LABELS[k] for k, m in conf.method_.items() if "fallback" in m]
        log(f"  ℹ {name}: {thin} calibrated by POOLED fallback, not cross-conformal "
            f"— report this asymmetry against CRISP-Net in the paper.")

    pred_te["lo"], pred_te["hi"] = conf.apply(pred_te["lo_raw"], pred_te["hi_raw"],
                                              TEST_EVENT_MASK, TEST_PEAK_MASK)
    res = score_model(name, pred_te, conf)
    res["params"] = int(sum(p.numel() for p in ens.parameters() if p.requires_grad))
    res["epochs"] = CFG.epochs
    res["best_val"] = float("nan")
    res["n_members"] = int(len(members))
    np.savez_compressed(pred_f, **{k: v.astype(np.float32)
                                   for k, v in pred_te.items() if v is not None})
    save_json(res, met_f)
    save_df(conf.report(),
            f"{P['conformal']}/{CFG.dataset}__{CFG.exp_version}__{slug(name)}_bins.csv")
    PREDICTIONS[name] = pred_te
    RESULTS[name] = res
    del ens, members
    gc.collect()
    return res


_r = run_deep_ensemble()
print(f"  {'Deep Ensemble':<26s} RMSE {_r['overall']['RMSE']:8.3f}  "
      f"│ PICP {_r['uncertainty']['PICP']:5.1f}%  MPIW {_r['uncertainty']['MPIW']:6.2f}  "
      f"Winkler {_r['uncertainty']['Winkler']:7.2f}  PCR {_r['uncertainty']['PCR']:5.1f}%  "
      f"(M={_r['n_members']})")

# ── ensemble mean as a free point-accuracy row (no extra training) ──────────
_ens = RESULTS.get("Deep Ensemble", {})
RESULTS["CRISP-Net (M=5)"] = score_model("CRISP-Net (M=5)",
                                         PREDICTIONS["Deep Ensemble"],
                                         CONFORMAL.get("Deep Ensemble"))
RESULTS["CRISP-Net (M=5)"].update({"params": _ens.get("params"),
                                   "epochs": _ens.get("epochs"),
                                   "n_members": _ens.get("n_members")})
save_json(RESULTS["CRISP-Net (M=5)"], _metric_path("CRISP-Net (M=5)"))
print(f"  {'CRISP-Net (M=5) ensemble':<26s} RMSE "
      f"{RESULTS['CRISP-Net (M=5)']['overall']['RMSE']:8.3f}  "
      f"(single model {RESULTS[NAME_PROPOSED]['overall']['RMSE']:.3f})")


08:42:57 │ INFO    │ ✓ cached [Deep Ensemble] (M=5)
  Deep Ensemble              RMSE   30.088  │ PICP  90.3%  MPIW  54.55  Winkler  109.75  PCR  70.4%  (M=5)
  CRISP-Net (M=5) ensemble   RMSE   30.088  (single model 30.896)


## 36 · Re-score at the current bin threshold

In [53]:
_rescored, _skipped = [], []
print(f"Re-scoring at min_bin_windows = {CFG.min_bin_windows}, "
      f"per_channel = {CFG.per_channel_conformal}\n")

for _name in list(RESULTS.keys()):
    _pred = PREDICTIONS.get(_name)
    if _pred is None or _pred.get("lo_raw") is None:
        _skipped.append((_name, "point model — no intervals"))
        continue

    # calibration predictions: from memory, else from the cached npz
    _cal = CALIB_PRED.get(_name)
    if _cal is None:
        _cf = _calib_path(_name)
        if not os.path.exists(_cf):
            _skipped.append((_name, "no calibration file"))
            continue
        _c = np.load(_cf)
        _cal = {k: _c[k] for k in _c.files}
        CALIB_PRED[_name] = _cal

    # Deep Ensemble cannot use cross-conformal (it would retrain 5 members per
    # fold); it takes the pooled fallback, exactly as in [33b].
    if _name == "Deep Ensemble":
        _conf = MondrianConformal().fit(_cal["y_true"], _cal["lo"], _cal["hi"],
                                        CAL_EVENT_MASK, CAL_PEAK_MASK)
    else:
        _conf = _fit_conformal(_name, _cal)
    CONFORMAL[_name] = _conf

    _lo, _hi = _conf.apply(_pred["lo_raw"], _pred["hi_raw"],
                           TEST_EVENT_MASK, TEST_PEAK_MASK)
    _pred["lo"], _pred["hi"] = _lo, _hi

    _old = (RESULTS[_name].get("uncertainty") or {}).get("PICP", float("nan"))
    _res = score_model(_name, _pred, _conf)
    for _k in ("params", "epochs", "best_val", "n_members", "n_seeds", "seeds"):
        if _k in RESULTS[_name]:
            _res[_k] = RESULTS[_name][_k]
    _res["min_bin_windows"] = int(CFG.min_bin_windows)
    RESULTS[_name] = _res

    np.savez_compressed(_pred_path(_name),
                        **{k: v.astype(np.float32) for k, v in _pred.items()
                           if v is not None})
    save_json(_res, _metric_path(_name))
    save_df(_conf.report(), f"{P['conformal']}/{CFG.dataset}__{CFG.exp_version}__"
                            f"{slug(_name)}_bins.csv")

    _u = _res["uncertainty"]
    _fb = sum("fallback" in m for m in _conf.method_.values())
    _rescored.append(_name)
    print(f"  {dname(_name):<26s} PICP {_old:5.1f}% → {_u['PICP']:5.1f}%   "
          f"MPIW {_u['MPIW']:6.2f}  Winkler {_u['Winkler']:7.2f}  "
          f"PCR {_u['PCR']:5.1f}%   bins falling back: {_fb}/4")

print(f"\n{len(_rescored)} models re-scored.")
for _n, _why in _skipped:
    print(f"  skipped {dname(_n):<24s} ({_why})")

# ── did lowering the threshold actually activate the peak bins? ──────────────
if NAME_PROPOSED in CONFORMAL:
    _rep = CONFORMAL[NAME_PROPOSED].report()
    print(f"\n{NAME_PROPOSED} calibration provenance at "
          f"min_bin_windows = {CFG.min_bin_windows}:")
    print(_rep[["bin", "n_windows", "method"]].to_string(index=False))
    _still = [r for r in _rep["method"] if "fallback" in str(r)]
    if _still:
        print(f"\n  ⚠ {len(_still)} bin(s) STILL falling back. The peak dimension of "
              f"the\n    partition remains partly inactive — drop to 40 and re-run "
              f"this cell,\n    or report the collapse honestly in the paper.")
    else:
        print("\n  ✓ All four bins now use their own Mondrian quantile. "
              "Table VI is a real test.")


Re-scoring at min_bin_windows = 40, per_channel = True

  CRISP-Net                  PICP  90.0% →  90.0%   MPIW  61.45  Winkler   80.19  PCR  91.0%   bins falling back: 0/4
  − gates                    PICP  90.1% →  90.1%   MPIW  62.87  Winkler   80.06  PCR  90.9%   bins falling back: 0/4
  − PRI                      PICP  89.9% →  89.9%   MPIW  65.54  Winkler   82.98  PCR  91.5%   bins falling back: 0/4
  MC-Dropout                 PICP  91.0% →  91.0%   MPIW  58.20  Winkler  110.92  PCR  72.0%   bins falling back: 0/4

4 models re-scored.
  skipped Graph WaveNet            (point model — no intervals)
  skipped AGCRN                    (point model — no intervals)
  skipped TGALSTM                  (point model — no intervals)
  skipped TDAG-Net                 (point model — no intervals)
  skipped TDAG-Net (split)         (point model — no intervals)
  skipped Deep Ensemble            (no calibration file)
  skipped CRISP-Net (M=5)          (point model — no intervals)

CRISP-Net

## 36b · Training history — all epochs, replayed from disk

In [48]:
TRAIN_HISTORY = {}


def _run_dirs_for(name):
    """Every run directory that belongs to a model, newest naming first."""
    import glob as _g
    base = f"{P['runs']}/{CFG.dataset}__{CFG.exp_version}__"
    pats = [f"{base}{slug(name)}", f"{base}{slug(name)}_seed_*",
            f"{base}{slug(name)}_xconf_fold_*"]
    out = []
    for pat in pats:
        out.extend(sorted(d for d in _g.glob(pat) if os.path.isdir(d)))
    return list(dict.fromkeys(out))


def load_history(name):
    """Read history.csv written during training; falls back to the checkpoint."""
    frames = []
    for d in _run_dirs_for(name):
        hp = f"{d}/history.csv"
        if os.path.exists(hp):
            try:
                h = pd.read_csv(hp)
                h["run"] = os.path.basename(d)
                frames.append(h)
                continue
            except Exception:
                pass
        ckp = f"{d}/checkpoints/last.pt"
        if os.path.exists(ckp):
            try:
                ck = torch.load(ckp, map_location="cpu", weights_only=False)
                h = pd.DataFrame(ck.get("history", []))
                if len(h):
                    h["run"] = os.path.basename(d)
                    frames.append(h)
            except Exception as exc:
                log(f"  {name}: could not read {os.path.basename(d)}: {exc}", "warning")
    if not frames:
        return None
    return pd.concat(frames, ignore_index=True)


def show_training(name, every=1, tail=None):
    """Print the epoch-by-epoch record of a completed run."""
    h = load_history(name)
    if h is None or not len(h):
        print(f"  {dname(name):<26s} no history on disk")
        return None
    TRAIN_HISTORY[name] = h
    for run, g in h.groupby("run", sort=True):
        g = g.sort_values("epoch")
        best_i = g["val_loss"].idxmin() if "val_loss" in g else None
        best_e = int(g.loc[best_i, "epoch"]) if best_i is not None else -1
        print(f"\n  ── {run} — {len(g)} epochs, best val "
              f"{g['val_loss'].min():.5f} @ epoch {best_e}")
        rows = g if tail is None else g.tail(tail)
        rows = rows.iloc[::every] if every > 1 else rows
        for _, r in rows.iterrows():
            star = " *" if int(r["epoch"]) == best_e else "  "
            lr = f" lr {r['lr']:.2e}" if "lr" in r and pd.notna(r.get("lr")) else ""
            secs = f" {r['seconds']:.0f}s" if "seconds" in r and pd.notna(r.get("seconds")) else ""
            print(f"     e{int(r['epoch']):>3d}  train {r['train_loss']:.5f}  "
                  f"val {r['val_loss']:.5f}{star}{lr}{secs}")
    return h


print("TRAINING HISTORY ")
print("=" * 76)
_summary = []
for _name in list(RESULTS.keys()):
    _h = show_training(_name)
    if _h is None:
        continue
    for _run, _g in _h.groupby("run"):
        _summary.append({"model": _name, "run": _run, "epochs": int(len(_g)),
                         "best_val": float(_g["val_loss"].min()),
                         "best_epoch": int(_g.loc[_g["val_loss"].idxmin(), "epoch"]),
                         "final_train": float(_g.sort_values("epoch")["train_loss"].iloc[-1])})

if _summary:
    _df = pd.DataFrame(_summary)
    print("\n" + "=" * 76)
    print(f"SUMMARY — {len(_df)} runs, {_df['epochs'].sum():,} epochs total\n")
    print(_df.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    save_df(_df, f"{P['results']}/training_summary_{CFG.dataset}_{CFG.exp_version}.csv")
else:
    print("No run directories found — check that P['runs'] points at the Drive folder.")


TRAINING HISTORY 

  ── 2023__v2__CRISP-Net — 80 epochs, best val 0.08279 @ epoch 78
     e  0  train 0.25601  val 0.15182   lr 3.00e-04
     e  1  train 0.15000  val 0.12090   lr 3.00e-04
     e  2  train 0.13589  val 0.11470   lr 3.00e-04
     e  3  train 0.13093  val 0.11011   lr 3.00e-04
     e  4  train 0.12745  val 0.10709   lr 3.00e-04
     e  5  train 0.12464  val 0.10416   lr 3.00e-04
     e  6  train 0.12343  val 0.10455   lr 3.00e-04
     e  7  train 0.12137  val 0.09966   lr 3.00e-04
     e  8  train 0.11908  val 0.09792   lr 3.00e-04
     e  9  train 0.11786  val 0.09803   lr 3.00e-04
     e 10  train 0.11675  val 0.09679   lr 3.00e-04
     e 11  train 0.11626  val 0.09567   lr 3.00e-04
     e 12  train 0.11508  val 0.09527   lr 3.00e-04
     e 13  train 0.11438  val 0.09395   lr 3.00e-04
     e 14  train 0.11316  val 0.09324   lr 3.00e-04
     e 15  train 0.11372  val 0.09247   lr 3.00e-04
     e 16  train 0.11261  val 0.09314   lr 3.00e-04
     e 17  train 0.11187  val 0

## 37 · Results tables

In [54]:
NAME_ENSEMBLE = "CRISP-Net (M=5)"
ACCURACY_QUEUE = [m for m in MODEL_ORDER + ABLATIONS + [NAME_ENSEMBLE] + LEGACY_QUOTED
                  if m in RESULTS]
UNC_QUEUE = [m for m in [NAME_PROPOSED] + ABLATIONS + ["MC-Dropout", "Deep Ensemble"]
             if m in RESULTS and "uncertainty" in RESULTS[m]]


def results_frame(scope="overall", queue=None) -> pd.DataFrame:
    rows = []
    for name in (queue or ACCURACY_QUEUE):
        r = RESULTS.get(name, {})
        block = r.get(scope)
        if not block:
            continue
        row = {"Model": name, **{k: block.get(k, np.nan) for k in METRIC_COLS}}
        std_block = r.get(f"{scope}_std")
        if std_block:
            row["RMSE_std"] = std_block.get("RMSE", np.nan)
        row["Params"] = r.get("params", np.nan)
        row["Epochs"] = r.get("epochs", np.nan)
        rows.append(row)
    return pd.DataFrame(rows)


def uncertainty_frame(queue=None) -> pd.DataFrame:
    rows = []
    for name in (queue or UNC_QUEUE):
        r = RESULTS.get(name, {})
        u, ur = r.get("uncertainty"), r.get("uncertainty_raw")
        if not u:
            continue
        row = {"Model": name, **{k: u.get(k, np.nan) for k in UNC_COLS}}
        row["PICP_uncal"] = (ur or {}).get("PICP", np.nan)
        row["MPIW_uncal"] = (ur or {}).get("MPIW", np.nan)
        row["Coverage gap"] = u.get("coverage_gap", np.nan)
        rows.append(row)
    return pd.DataFrame(rows)


def to_latex(df, caption, label, bold_best=True, fmt=None):
    """IEEE table: [!t] float, footnotesize, booktabs-free so it drops into any template."""
    d = df.copy()
    fmt = fmt or {"RMSE": "{:.3f}", "RMSE_std": "{:.3f}", "MAE": "{:.3f}",
                  "WMAPE": "{:.2f}", "R2": "{:.4f}", "EPE": "{:.2f}", "NDE": "{:.2f}",
                  "Params": "{:,.0f}", "Epochs": "{:.0f}",
                  "PICP": "{:.2f}", "MPIW": "{:.2f}", "Winkler": "{:.2f}", "PCR": "{:.2f}",
                  "PICP_uncal": "{:.2f}", "MPIW_uncal": "{:.2f}", "Coverage gap": "{:+.2f}"}
    best = {}
    if bold_best:
        for c in d.columns:
            if c in ("Model", "Params", "Epochs", "bin", "n_windows", "RMSE_std"):
                continue
            if not pd.api.types.is_numeric_dtype(d[c]) or not d[c].notna().any():
                continue
            if c == "PICP":                       # closest to nominal wins
                best[c] = (d[c] - 100 * (1 - CFG.alpha)).abs().idxmin()
            elif c in LOWER_IS_BETTER:
                best[c] = d[c].idxmin()
            elif c in ("R2", "PCR"):
                best[c] = d[c].idxmax()
    for c, f in fmt.items():
        if c in d.columns:
            d[c] = d[c].map(lambda v: f.format(v) if pd.notna(v) else "--")
    # Not every table is per-model: the conditional-coverage table is keyed by Mondrian
    # bin and has no Model column at all, so this must be conditional.
    if "Model" in d.columns:
        d["Model"] = d["Model"].map(
            lambda m: (r"\textbf{" + str(m) + "}") if m == NAME_PROPOSED else str(m))
    if bold_best:
        for c, i in best.items():
            if c in d.columns and i in d.index:
                d.loc[i, c] = r"\textbf{" + str(d.loc[i, c]) + "}"
    body = d.to_latex(index=False, escape=False,
                      column_format="l" + "r" * (len(d.columns) - 1))
    return (f"\\begin{{table}}[!t]\n\\caption{{{caption}}}\n\\label{{{label}}}\n"
            f"\\centering\n\\footnotesize\n{body}\\end{{table}}\n")


if NAME_ENSEMBLE in RESULTS:
    _e = RESULTS.get("Deep Ensemble", {})
    RESULTS[NAME_ENSEMBLE].setdefault("params", _e.get("params"))
    RESULTS[NAME_ENSEMBLE].setdefault("epochs", _e.get("epochs"))

TBL_MAIN = results_frame("overall")
TBL_EVENT = results_frame("event")
TBL_NORMAL = results_frame("normal")
TBL_ENTRY = results_frame("overall_entry")
TBL_EXIT = results_frame("overall_exit")
TBL_UNC = uncertainty_frame()

_tex = []
if len(TBL_MAIN):
    print("╔═ TABLE I — MAIN RESULTS (full test set, both channels) ══════════════════")
    print(TBL_MAIN.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
    save_df(TBL_MAIN, f"{P['tables']}/main_results_{CFG.dataset}_{CFG.exp_version}.csv")
    save_df(TBL_EVENT, f"{P['tables']}/event_results_{CFG.dataset}_{CFG.exp_version}.csv")
    save_df(TBL_NORMAL, f"{P['tables']}/normal_results_{CFG.dataset}_{CFG.exp_version}.csv")
    _tex.append(to_latex(
        TBL_MAIN.drop(columns=["Epochs"]),
        f"Network-wide prediction performance on the Nanjing {CFG.dataset} AFC test set "
        f"({N_STATIONS} stations, {CFG.freq_minutes}-min intervals, horizon {H}, "
        f"entry and exit channels jointly). RMSE\\_std is the seed standard deviation. "
        f"Huber $\\delta={CFG.huber_delta}$.",
        "tab:main_results"))

if len(TBL_ENTRY) and len(TBL_EXIT):
    print("\n╔═ TABLE II — PER-CHANNEL BREAKDOWN ═════════════════════════════════════")
    _pc = TBL_ENTRY[["Model", "RMSE", "MAE", "WMAPE", "R2"]].merge(
        TBL_EXIT[["Model", "RMSE", "MAE", "WMAPE", "R2"]],
        on="Model", suffixes=(" (entry)", " (exit)"))
    print(_pc.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))
    save_df(_pc, f"{P['tables']}/per_channel_results_{CFG.dataset}_{CFG.exp_version}.csv")
    _tex.append(to_latex(_pc, "Accuracy decomposed by flow direction. Paper 3 reported "
                              "the pooled figure only; the exit channel is forecast "
                              "explicitly here.", "tab:per_channel", bold_best=False))

if len(TBL_EVENT):
    print("\n╔═ TABLE III — EVENT-DAY RESULTS ════════════════════════════════════════")
    print(TBL_EVENT.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
    _tex.append(to_latex(TBL_EVENT.drop(columns=["Epochs", "Params"]),
                         "Performance restricted to flagged event days "
                         f"({int(TEST_EVENT_MASK.sum())} test windows).",
                         "tab:event_results"))

if len(TBL_UNC):
    print("\n╔═ TABLE IV — UNCERTAINTY QUANTIFICATION ════════════════════════════════")
    print(f"   target PICP = {100 * (1 - CFG.alpha):.0f}%   "
          f"(PICP_uncal = before conformal correction)")
    print(TBL_UNC.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
    save_df(TBL_UNC, f"{P['tables']}/uncertainty_results_{CFG.dataset}_{CFG.exp_version}.csv")
    _tex.append(to_latex(TBL_UNC,
                         f"Distribution-free {100 * (1 - CFG.alpha):.0f}\\% prediction "
                         "intervals. PICP closest to nominal is best; MPIW and Winkler "
                         "lower is better; PCR higher is better.",
                         "tab:uncertainty"))

# ── per-bin conditional coverage, the claim that "coverage holds on event days" ──
if NAME_PROPOSED in RESULTS and "uncertainty_bins" in RESULTS[NAME_PROPOSED]:
    _bins = pd.DataFrame(RESULTS[NAME_PROPOSED]["uncertainty_bins"])
    _bins["bin"] = [BIN_LABELS[(bool(e), bool(p))]
                    for e, p in zip(_bins["event"], _bins["peak"])]
    print(f"\n╔═ TABLE V — CONDITIONAL COVERAGE BY MONDRIAN BIN ({NAME_PROPOSED}) ══════")
    print(_bins[["bin", "n_windows", "PICP", "MPIW", "Winkler"]]
          .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
    save_df(_bins, f"{P['tables']}/conditional_coverage_{CFG.dataset}_{CFG.exp_version}.csv")
    _tex.append(to_latex(_bins[["bin", "n_windows", "PICP", "MPIW", "Winkler"]],
                         "Conditional coverage inside each Mondrian bin. Marginal "
                         "coverage can hold while a bin fails; this table is the check.",
                         "tab:conditional_coverage", bold_best=False))
    if NAME_PROPOSED in CONFORMAL:
        print("\n calibration provenance (n_windows is WINDOW-scale; q_hat per channel):")
        print(CONFORMAL[NAME_PROPOSED].report().to_string(index=False))


# ── gate verdict: does the persona gate earn its place on interval quality? ──
if NAME_PROPOSED in RESULTS and "CRISP-Net w/o gates" in RESULTS:
    _f = RESULTS[NAME_PROPOSED].get("uncertainty", {})
    _n = RESULTS["CRISP-Net w/o gates"].get("uncertainty", {})
    if _f and _n:
        print("\n╔═ GATE VERDICT — full vs ungated ═══════════════════════════════════════")
        print(f"   {'metric':<10s} {'CRISP-Net':>12s} {'− gates':>12s} {'Δ':>10s}")
        _rows = [("RMSE", RESULTS[NAME_PROPOSED]["overall"]["RMSE"],
                  RESULTS["CRISP-Net w/o gates"]["overall"]["RMSE"], "lower"),
                 ("Winkler", _f.get("Winkler"), _n.get("Winkler"), "lower"),
                 ("PCR", _f.get("PCR"), _n.get("PCR"), "higher"),
                 ("MPIW", _f.get("MPIW"), _n.get("MPIW"), "lower"),
                 ("PICP", _f.get("PICP"), _n.get("PICP"), "nominal")]
        _gate_wins = 0
        for _m, _a, _b, _dir in _rows:
            if _a is None or _b is None:
                continue
            _d = _a - _b
            _win = (_d < 0) if _dir == "lower" else ((_d > 0) if _dir == "higher" else None)
            if _m in ("Winkler", "PCR") and _win:
                _gate_wins += 1
            _tag = "" if _win is None else ("  gates better" if _win else "  gates worse")
            print(f"   {_m:<10s} {_a:>12.3f} {_b:>12.3f} {_d:>+10.3f}{_tag}")
        print(f"   → gates win {_gate_wins}/2 interval metrics; all deltas are "
              f"inside the seed-noise floor.")
        print("     Learned shares are near-uniform and entry/exit agree: no persona "
              "is selected.")

# ── TABLE VI — the decisive experiment ───────────────────────────────────────
TBL_DECISIVE = coverage_comparison(NAME_PROPOSED)
if len(TBL_DECISIVE):
    print("\n╔═ TABLE VI — RAW vs POOLED vs MONDRIAN, PER BIN  (the decisive test) ════")
    _piv = TBL_DECISIVE.pivot(index="bin", columns="method", values="PICP")
    _piv = _piv.reindex(BIN_ORDER)
    _n = (TBL_DECISIVE.drop_duplicates("bin").set_index("bin")["n_windows"]
          .reindex(BIN_ORDER))
    _piv.insert(0, "n_win", _n)
    print(f"   coverage (%) against a {100 * (1 - CFG.alpha):.0f}% target")
    print(_piv.to_string(float_format=lambda v: f"{v:,.2f}"))
    _worst = TBL_DECISIVE.groupby("method", observed=True)["|PICP − 90|"].max()
    print("\n   worst-bin deviation from nominal:")
    for _m, _v in _worst.items():
        print(f"     {str(_m):<22s} {_v:5.2f} pp")
    _gain = _worst.get("raw quantile heads", np.nan) - _worst.get("Mondrian ECQ", np.nan)
    if np.isfinite(_gain):
        print(f"\n   → Mondrian ECQ improves worst-bin coverage by {_gain:+.2f} pp. "
              f"{'CLAIM SUPPORTED.' if _gain > 1.0 else 'Claim NOT supported — report it and lead with Table VII.'}")
    save_df(TBL_DECISIVE,
            f"{P['tables']}/decisive_coverage_{CFG.dataset}_{CFG.exp_version}.csv")
    _tex.append(to_latex(
        TBL_DECISIVE[["method", "bin", "n_windows", "PICP", "MPIW", "Winkler"]],
        "Conditional coverage under three calibration regimes. Pooled coverage is "
        "attainable without conformal correction; the event-conditional bins are "
        "where the guarantee is actually needed.", "tab:decisive", bold_best=False))

# ── TABLE VII — conformal transfer across architectures ──────────────────────
if TRANSFER:
    _rows = []
    for _n, _t in TRANSFER.items():
        _u = uncertainty_suite(_t["y_true"], _t["lo"], _t["hi"], TEST_EVENT_MASK)
        _rows.append({"Model": f"{_n} + ECQ", **{k: _u.get(k, np.nan) for k in UNC_COLS},
                      "retrained": "no"})
    if NAME_PROPOSED in RESULTS:
        _u = RESULTS[NAME_PROPOSED]["uncertainty"]
        _rows.append({"Model": NAME_PROPOSED,
                      **{k: _u.get(k, np.nan) for k in UNC_COLS}, "retrained": "—"})
    TBL_TRANSFER = pd.DataFrame(_rows)
    print("\n╔═ TABLE VII — CONFORMAL TRANSFER (inference only, no retraining) ═══════")
    print(TBL_TRANSFER.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
    save_df(TBL_TRANSFER,
            f"{P['tables']}/conformal_transfer_{CFG.dataset}_{CFG.exp_version}.csv")
    _tex.append(to_latex(TBL_TRANSFER,
                         "Absolute-residual event-conditional conformal applied to "
                         "point forecasters without retraining. The calibration layer "
                         "is a property of the method, not of one architecture.",
                         "tab:transfer"))

if _tex:
    with open(f"{P['tables']}/results_{CFG.dataset}_{CFG.exp_version}.tex", "w",
              encoding="utf-8") as f:
        f.write("\n".join(_tex))
    print(f"\nLaTeX written → {P['tables']}/results_{CFG.dataset}_{CFG.exp_version}.tex")

# ── headline comparison against the best non-CALM baseline ──────────────────
if len(TBL_MAIN) and NAME_PROPOSED in TBL_MAIN["Model"].values:
    base = [m for m in TBL_MAIN["Model"]
            if role(m) in ("BASELINE", "PREDECESSOR")
            and not str(m).startswith(NAME_PROPOSED)]   # our own ensemble is not a baseline
    if base:
        b = TBL_MAIN.set_index("Model").loc[base]
        best_base = b["RMSE"].idxmin()
        ours_r = RESULTS[NAME_PROPOSED]
        ours = TBL_MAIN.set_index("Model").loc[NAME_PROPOSED]
        o_std = (ours_r.get("overall_std") or {}).get("RMSE", np.nan)
        b_std = ((RESULTS.get(best_base, {}).get("overall_std") or {}).get("RMSE", np.nan))
        print(f"\n{NAME_PROPOSED} vs best baseline ({best_base}):")
        print(f"   RMSE {b.loc[best_base, 'RMSE']:.3f} ± {b_std:.3f}  →  "
              f"{ours['RMSE']:.3f} ± {o_std:.3f}   (mean ± seed-std)")
        diff = b.loc[best_base, "RMSE"] - ours["RMSE"]
        _stds = [v for v in (o_std, b_std) if np.isfinite(v)]
        pooled = np.sqrt(np.mean(np.square(_stds))) if _stds else np.nan
        if np.isfinite(pooled) and pooled > 0:
            verdict = ("clears 2σ" if abs(diff) > 2 * pooled
                       else "does NOT clear 2σ — inside the seed-noise floor")
            print(f"   ΔRMSE = {diff:+.3f}  →  {verdict}")
        else:
            print(f"   ΔRMSE = {diff:+.3f}  →  single seed on at least one side; "
                  f"no σ is claimed. Treat as provisional until seeded.")
        for c in METRIC_COLS:
            if pd.isna(ours[c]) or pd.isna(b.loc[best_base, c]):
                continue
            d = b.loc[best_base, c] - ours[c]
            pct = 100 * d / max(abs(b.loc[best_base, c]), 1e-9)
            better = (d > 0) if c != "R2" else (d < 0)
            print(f"    {c:<6s} {b.loc[best_base, c]:10.4f} → {ours[c]:10.4f}   "
                  f"{'better' if better else 'worse '} {abs(pct):5.2f}%")

# ── headline summary ────────────────────────────────────────────────────────
print("\n╔═ SUMMARY ══════════════════════════════════════════════════════════════")
if len(TBL_MAIN):
    _t = TBL_MAIN.set_index("Model")
    _best = _t["RMSE"].idxmin()
    print(f"   accuracy   best RMSE {_t.loc[_best, 'RMSE']:.3f} ({_best})")
if len(TBL_EVENT):
    _te = TBL_EVENT.set_index("Model")
    _eb = _te["RMSE"].idxmin()
    print(f"   event days best RMSE {_te.loc[_eb, 'RMSE']:.3f} ({_eb})")
if len(TBL_UNC):
    _u = TBL_UNC.set_index("Model")
    print(f"   intervals  best Winkler {_u['Winkler'].min():.2f} "
          f"({_u['Winkler'].idxmin()}) · best PCR {_u['PCR'].max():.2f}% "
          f"({_u['PCR'].idxmax()})")
if "TBL_DECISIVE" in globals() and len(TBL_DECISIVE):
    _w = TBL_DECISIVE.groupby("method", observed=True)["|PICP − 90|"].max()
    print(f"   calibration worst-bin deviation: "
          + " · ".join(f"{str(k)} {v:.2f}pp" for k, v in _w.items()))
if TRANSFER:
    print(f"   transfer   {len(TRANSFER)} baselines calibrated without retraining")
print("╚════════════════════════════════════════════════════════════════════════")

╔═ TABLE I — MAIN RESULTS (full test set, both channels) ══════════════════
                    Model    RMSE     MAE   WMAPE     R2     EPE     NDE  Params  Epochs  RMSE_std
            Graph WaveNet 36.7400 15.6184 21.2814 0.9324 17.2456 23.0672   65068      80       NaN
                    AGCRN 30.8610 13.5311 18.4372 0.9523 16.4926 19.3760  258050      80    0.2977
                  TGALSTM 35.3384 15.3205 20.8754 0.9375 17.3508 22.1872  210527      80       NaN
                 TDAG-Net 32.2103 14.1083 19.2237 0.9481 16.0407 20.2232  253825      80    0.2100
TDAG-Net (Input-Splitter) 30.9906 13.7942 18.7958 0.9519 15.7490 19.4574  254209      80    0.1228
                CRISP-Net 30.8963 13.6448 18.5922 0.9522 15.5890 19.3982  257313      80       NaN
      CRISP-Net w/o gates 30.6522 13.6610 18.6143 0.9530 15.5013 19.2450  257313      80       NaN
        CRISP-Net w/o PRI 32.3827 14.1359 19.2613 0.9475 16.1583 20.3315  256929      80       NaN
          CRISP-Net (M=5) 30.0875

## 38 · Figures — training and accuracy

In [42]:
def _have(*names):
    return [n for n in names if n in RESULTS]


@figure("F20", "Training convergence and loss curves", "05_training")
def fig_training_curves():
    rows = []
    for name in EXPERIMENT_QUEUE:
        hp = f"{P['runs']}/{CFG.dataset}__{slug(name)}/history.csv"
        if os.path.exists(hp):
            h = pd.read_csv(hp)
            h["model"] = name
            rows.append(h)
    if not rows:
        return _skip("F20", "no training history on disk")
    hist = pd.concat(rows, ignore_index=True)
    models = [m for m in EXPERIMENT_QUEUE if m in set(hist["model"])]
    ncol = 4
    nrow = int(np.ceil(len(models) / ncol))
    fig, axes = new_fig("double", 1.35 * nrow + 0.35, nrow, ncol, squeeze=False)
    for ax, name in zip(axes.ravel(), models):
        h = hist[hist["model"] == name]
        ax.plot(h["epoch"], h["train_loss"], lw=0.95, color=M3["blue600"], label="train")
        ax.plot(h["epoch"], h["val_loss"], lw=0.95, color=ACCENT, label="tune")
        if len(h):
            b = h.loc[h["val_loss"].idxmin()]
            ax.axvline(b["epoch"], color=MUTED, ls="--", lw=0.6)
            ax.scatter([b["epoch"]], [b["val_loss"]], color=ACCENT, zorder=5, s=8)
        ax.set_title(dname(name), fontsize=6.8, loc="left", fontweight="bold",
                     color=ACCENT if role(name) == "PROPOSED" else INK)
        ax.tick_params(labelsize=5.8)
        ax.set_xlabel("epoch", fontsize=6.2, color=MUTED)
        ax.set_ylabel("loss", fontsize=6.2, color=MUTED)
    axes.ravel()[0].legend(fontsize=5.6, loc="upper right")
    for ax in axes.ravel()[len(models):]:
        ax.set_axis_off()
    _safe_tight(fig)
    return fig, hist


@figure("F21", "Benchmark comparison across all six accuracy metrics", "06_results")
def fig_metric_comparison():
    d = TBL_MAIN.copy()
    if not len(d):
        return _skip("F21", "empty results table")
    d = d[d["Model"].map(lambda m: role(m) in
                         ("BASELINE", "PREDECESSOR", "PROPOSED"))]
    fig, axes = new_fig("double", 4.4, 2, 3)
    for ax, c in zip(axes.ravel(), METRIC_COLS):
        if c not in d.columns:
            ax.set_axis_off()
            continue
        s = d.sort_values(c, ascending=(c != "R2"))
        ax.barh(range(len(s)), s[c], color=[bar_color(k) for k in s["Model"]], height=0.7)
        ax.set_yticks(range(len(s)))
        ax.set_yticklabels([dname(m) for m in s["Model"]], fontsize=6.0)
        for lbl, k in zip(ax.get_yticklabels(), s["Model"]):
            if role(k) == "PROPOSED":
                lbl.set_fontweight("bold")
                lbl.set_color(ACCENT)
        ax.invert_yaxis()
        _finish(ax, f"{c}{' (higher better)' if c == 'R2' else ' (lower better)'}")
        _bar_labels(ax, s[c].to_numpy(), fmt="{:.3f}" if c == "R2" else "{:.2f}", fs=5.6)
        ax.set_xlim(0, s[c].max() * 1.18)
    _panels(axes)
    _safe_tight(fig)
    return fig, d


@figure("F22", "Accuracy decomposed by flow direction", "06_results")
def fig_channel_breakdown():
    if not len(TBL_ENTRY) or not len(TBL_EXIT):
        return _skip("F22", "per-channel results unavailable")
    keep = [m for m in TBL_ENTRY["Model"]
            if role(m) in ("BASELINE", "PREDECESSOR", "PROPOSED")]
    e = TBL_ENTRY.set_index("Model").loc[keep]
    x = TBL_EXIT.set_index("Model").loc[keep]
    fig, axes = new_fig("double", 2.5, 1, 3)
    for ax, c in zip(axes, ["RMSE", "WMAPE", "R2"]):
        order = (e[c] + x[c]).sort_values(ascending=(c != "R2")).index
        y = np.arange(len(order))
        w = 0.38
        ax.barh(y - w / 2, e.loc[order, c], w, color=CHANNEL_COLORS["Entry"], label="entry")
        ax.barh(y + w / 2, x.loc[order, c], w, color=CHANNEL_COLORS["Exit"], label="exit")
        ax.set_yticks(y)
        ax.set_yticklabels([dname(m) for m in order], fontsize=6.0)
        for lbl, k in zip(ax.get_yticklabels(), order):
            if role(k) == "PROPOSED":
                lbl.set_fontweight("bold")
                lbl.set_color(ACCENT)
        ax.invert_yaxis()
        _finish(ax, c, legend=(c == "RMSE"))
    _panels(axes)
    _safe_tight(fig)
    d = pd.DataFrame({"Model": keep})
    for c in ["RMSE", "WMAPE", "R2"]:
        d[f"entry_{c}"] = e.loc[keep, c].to_numpy()
        d[f"exit_{c}"] = x.loc[keep, c].to_numpy()
    return fig, d


@figure("F23", "Error growth over the forecast horizon", "06_results")
def fig_horizon_degradation():
    show = _have(NAME_PROPOSED, "TDAG-Net", "AGCRN", "Graph WaveNet", "TGALSTM")
    if len(show) < 2:
        return _skip("F23", f"need ≥2 models with per-horizon data, have {show}")
    fig, axes = new_fig("double", 2.3, 1, 3)
    frames = []
    for ax, c in zip(axes, ["RMSE", "WMAPE", "R2"]):
        for name in show:
            h = _as_df(RESULTS[name].get("per_horizon"))
            if not len(h) or c not in h.columns:
                continue
            hi = role(name) == "PROPOSED"
            ax.plot(h["minutes_ahead"], h[c], marker="o", ms=2.6,
                    lw=1.7 if hi else 0.95,
                    color=ACCENT if hi else None,
                    alpha=1.0 if hi else 0.72,
                    label=dname(name), zorder=5 if hi else 2)
            if c == "RMSE":
                g = h[["minutes_ahead", "RMSE"]].copy()
                g["model"] = name
                frames.append(g)
        _finish(ax, c, "minutes ahead", c)
    axes[-1].legend(fontsize=5.8, ncol=1, loc="lower left")
    hp = _as_df(RESULTS[NAME_PROPOSED].get("per_horizon"))
    if len(hp):
        v = hp["RMSE"].to_numpy()
        deg = 100 * (v[-1] - v[0]) / max(v[0], 1e-9)
        axes[0].annotate(f"{dname(NAME_PROPOSED)}: {deg:+.1f}% "
                         f"step 1 → {len(v)}",
                         xy=(0.03, 0.93), xycoords="axes fraction",
                         fontsize=6.2, color=ACCENT, fontweight="bold", va="top")
        log(f"F23 horizon RMSE ({NAME_PROPOSED}): "
            + ", ".join(f"{x:.4f}" for x in v) + f"  → degradation {deg:+.2f}%")
    _panels(axes)
    _safe_tight(fig)
    return fig, (pd.concat(frames, ignore_index=True) if frames else None)


@figure("F24", "Observed versus predicted flow at representative stations", "06_results")
def fig_prediction_traces():
    if NAME_PROPOSED not in PREDICTIONS:
        return _skip("F24", "no CRISP-Net predictions")
    pr = PREDICTIONS[NAME_PROPOSED]
    yt, yp = pr["y_true"], pr["y_pred"]
    lo, hi = pr.get("lo"), pr.get("hi")
    if lo is not None and NAME_PROPOSED in CONFORMAL and CONFORMAL[NAME_PROPOSED].fitted:
        lo, hi = CONFORMAL[NAME_PROPOSED].apply(lo, hi, TEST_EVENT_MASK, TEST_PEAK_MASK)
    base_name = next((m for m in ["TDAG-Net", "AGCRN", "Graph WaveNet", "TGALSTM"]
                      if m in PREDICTIONS), None)
    stations = [s for s in CFG.report_stations if s in ST_POS][:4]
    if not stations:
        return _skip("F24", "no reporting stations")
    n = min(len(yt), 2 * BINS_PER_DAY)
    tstamp = TIME_INDEX[IDX_TE[:n] + L]
    fig, axes = new_fig("double", 1.35 * len(stations) + 0.5, len(stations), 1, sharex=True)
    axes = np.atleast_1d(axes)
    rows = []
    for ax, sid in zip(axes, stations):
        ni = ST_POS[sid]
        a = yt[:n, 0, ni, CH_IN]
        p = yp[:n, 0, ni, CH_IN]
        if lo is not None:
            ax.fill_between(tstamp, lo[:n, 0, ni, CH_IN], hi[:n, 0, ni, CH_IN],
                            color=ACCENT, alpha=0.16, lw=0,
                            label=f"{100 * (1 - CFG.alpha):.0f}% PI")
        ax.plot(tstamp, a, lw=1.0, color=INK, label="observed")
        ax.plot(tstamp, p, lw=1.0, color=ACCENT, label=dname(NAME_PROPOSED))
        if base_name:
            pb = PREDICTIONS[base_name]["y_pred"][:n, 0, ni, CH_IN]
            ax.plot(tstamp, pb, lw=0.75, color="#7aa6c9", ls="--", label=base_name, alpha=0.9)
            rows.append(pd.DataFrame({"t": tstamp, "station": sid, "observed": a,
                                      "crispnet": p, "baseline": pb}))
        r = 100 * np.abs(a - p).sum() / max(np.abs(a).sum(), 1e-9)
        _finish(ax, f"Station {sid} — entry, +10 min   (WMAPE {r:.1f}%)",
                None, "taps / 10 min")
    axes[0].legend(ncol=4, loc="upper right", fontsize=5.8)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d %H:%M"))
    for lbl in axes[-1].get_xticklabels():
        lbl.set_rotation(30)
        lbl.set_ha("right")
    _panels(axes)
    _safe_tight(fig)
    return fig, (pd.concat(rows, ignore_index=True) if rows else None)


@figure("F25", "Event-window behaviour and peak forecasting resilience", "06_results")
def fig_event_zoom():
    if NAME_PROPOSED not in PREDICTIONS or not TEST_EVENT_MASK.any():
        return _skip("F25", "no event windows in the test split")
    pr = PREDICTIONS[NAME_PROPOSED]
    yt, yp = pr["y_true"], pr["y_pred"]
    lo, hi = pr.get("lo"), pr.get("hi")
    if lo is not None and NAME_PROPOSED in CONFORMAL and CONFORMAL[NAME_PROPOSED].fitted:
        lo, hi = CONFORMAL[NAME_PROPOSED].apply(lo, hi, TEST_EVENT_MASK, TEST_PEAK_MASK)
    base_name = next((m for m in ["TDAG-Net", "AGCRN", "Graph WaveNet"]
                      if m in PREDICTIONS), None)
    ev_idx = np.where(TEST_EVENT_MASK)[0]
    flat = yt[ev_idx, 0, :, CH_IN]
    pos = np.unravel_index(np.argmax(flat), flat.shape)
    ni = int(pos[1])
    day_of = TENS["day_index"][IDX_TE[ev_idx] + L]
    sel = ev_idx[day_of == day_of[pos[0]]]
    if len(sel) < 4:
        return _skip("F25", "event day has too few windows")
    t = TIME_INDEX[IDX_TE[sel] + L]

    fig, axes = new_fig("double", 2.4, 1, 2, gridspec_kw={"width_ratios": [1.8, 1.0]})
    ax = axes[0]
    if lo is not None:
        ax.fill_between(t, lo[sel, 0, ni, CH_IN], hi[sel, 0, ni, CH_IN],
                        color=ACCENT, alpha=0.18, lw=0,
                        label=f"{100 * (1 - CFG.alpha):.0f}% conformal PI")
    ax.plot(t, yt[sel, 0, ni, CH_IN], lw=1.4, color=INK, label="observed")
    ax.plot(t, yp[sel, 0, ni, CH_IN], lw=1.2, color=ACCENT, label=dname(NAME_PROPOSED))
    if base_name:
        ax.plot(t, PREDICTIONS[base_name]["y_pred"][sel, 0, ni, CH_IN],
                lw=0.85, ls="--", color="#7aa6c9", label=base_name)
    pk = int(np.argmax(yt[sel, 0, ni, CH_IN]))
    ax.annotate(f"peak {yt[sel, 0, ni, CH_IN][pk]:.0f}",
                xy=(t[pk], yt[sel, 0, ni, CH_IN][pk]), xytext=(6, 9),
                textcoords="offset points", color=INK, fontsize=6.2, fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=INK, lw=0.6))
    ax.legend(fontsize=6.0, loc="upper left")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    _finish(ax, f"Station {int(STATION_IDS[ni])}, entry — event day", None, "taps / 10 min")

    ax = axes[1]
    ev_models = [m for m in TBL_EVENT["Model"]
                 if role(m) in ("BASELINE", "PREDECESSOR", "PROPOSED")]
    if ev_models:
        s = TBL_EVENT.set_index("Model").loc[ev_models].sort_values("RMSE")
        ax.barh(range(len(s)), s["RMSE"], color=[bar_color(m) for m in s.index], height=0.7)
        ax.set_yticks(range(len(s)))
        ax.set_yticklabels([dname(m) for m in s.index], fontsize=6.0)
        for lbl, k in zip(ax.get_yticklabels(), s.index):
            if role(k) == "PROPOSED":
                lbl.set_fontweight("bold")
                lbl.set_color(ACCENT)
        ax.invert_yaxis()
        _bar_labels(ax, s["RMSE"].to_numpy(), fmt="{:.2f}", fs=5.8)
        ax.set_xlim(0, s["RMSE"].max() * 1.18)
        _finish(ax, "Event-day RMSE", "RMSE (lower better)")
    _panels(axes)
    _safe_tight(fig)
    return fig, pd.DataFrame({"t": t, "observed": yt[sel, 0, ni, CH_IN],
                              "predicted": yp[sel, 0, ni, CH_IN]})


@figure("F26", "Ablation of the CRISP-Net components", "06_results")
def fig_ablation():
    have = _have(NAME_PROPOSED, *ABLATIONS)
    if len(have) < 2:
        return _skip("F26", "need the full model plus ≥1 ablation")
    # −gates : persona channels enter ungated (== the Input-Splitter input)
    d = pd.DataFrame([{"variant": m, **{c: RESULTS[m]["overall"].get(c, np.nan)
                                        for c in METRIC_COLS}} for m in have])
    full = d[d["variant"] == NAME_PROPOSED].iloc[0]
    for c in ["RMSE", "WMAPE", "EPE"]:
        d[f"d{c}"] = 100 * (d[c] - full[c]) / max(abs(full[c]), 1e-9)
    fig, axes = new_fig("double", 2.2, 1, 3)
    for ax, c, t in zip(axes, ["dRMSE", "dWMAPE", "dEPE"], ["RMSE", "WMAPE", "EPE"]):
        s = d[d["variant"] != NAME_PROPOSED].sort_values(c, ascending=False)
        ax.barh(range(len(s)), s[c],
                color=np.where(s[c] > 0, ACCENT, M3["green"]), height=0.62)
        ax.set_yticks(range(len(s)))
        ax.set_yticklabels([dname(v) for v in s["variant"]], fontsize=6.2)
        ax.axvline(0, color=INK, lw=0.7)
        ax.invert_yaxis()
        _bar_labels(ax, s[c].to_numpy(), fmt="{:+.2f}%", fs=6.0)
        lim = max(np.abs(s[c]).max() * 1.45, 0.5)
        ax.set_xlim(-lim, lim)
        _finish(ax, f"Δ {t} when the module is removed", "% change vs full model")
    _panels(axes)
    _safe_tight(fig)
    return fig, d


@figure("F27", "Conditional coverage under three calibration regimes", "06_results")
def fig_decisive_coverage():
    """The paper's pivotal figure, and the one a reviewer will look at first."""
    d = TBL_DECISIVE if "TBL_DECISIVE" in globals() else pd.DataFrame()
    if not len(d):
        return _skip("F27", "coverage comparison unavailable")
    target = 100 * (1 - CFG.alpha)
    methods = [m for m in ["raw quantile heads", "pooled conformal", "Mondrian ECQ"]
               if m in set(d["method"])]
    cols = {"raw quantile heads": M3["blue100"], "pooled conformal": M3["blue600"],
            "Mondrian ECQ": ACCENT}
    bins = [b for b in BIN_ORDER if b in set(d["bin"])]

    fig, axes = new_fig("double", 2.5, 1, 3,
                        gridspec_kw={"width_ratios": [1.45, 1.15, 0.95]})

    # (a) coverage per bin, grouped by method
    ax = axes[0]
    x = np.arange(len(bins))
    w = 0.8 / max(len(methods), 1)
    for i, m in enumerate(methods):
        sub = d[d["method"] == m].set_index("bin").reindex(bins)
        ax.bar(x + (i - (len(methods) - 1) / 2) * w, sub["PICP"], w,
               color=cols[m], label=m, zorder=3)
    ax.axhline(target, color=M3["red"], ls="--", lw=1.0, zorder=4)
    ax.annotate(f"nominal {target:.0f}%", xy=(len(bins) - 0.55, target), fontsize=5.9,
                color=M3["red"], va="bottom", ha="right")
    ax.set_xticks(x)
    ax.set_xticklabels([b.replace(" · ", "\n") for b in bins], fontsize=6.1)
    lo = min(80.0, float(np.nanmin(d["PICP"])) - 2)
    ax.set_ylim(lo, max(96.0, float(np.nanmax(d["PICP"])) + 2))
    ax.legend(fontsize=5.9, ncol=1, loc="lower left")
    _finish(ax, "Coverage inside each Mondrian bin", None, "PICP (%)")

    # (b) the cost of that coverage — interval width
    ax = axes[1]
    for i, m in enumerate(methods):
        sub = d[d["method"] == m].set_index("bin").reindex(bins)
        ax.plot(np.arange(len(bins)), sub["MPIW"], "-o", ms=3.2, lw=1.3,
                color=cols[m], label=m)
    ax.set_xticks(np.arange(len(bins)))
    ax.set_xticklabels([b.replace(" · ", "\n") for b in bins], fontsize=6.1)
    _finish(ax, "Interval width paid per bin", None, "MPIW (passengers / 10 min)")
    ax.annotate("wider only where the data is harder",
                xy=(0.03, 0.93), xycoords="axes fraction", fontsize=5.8, color=MUTED)

    # (c) the argument in one number: worst-bin deviation from nominal
    ax = axes[2]
    worst = [float(d[d["method"] == m]["|PICP − 90|"].max()) for m in methods]
    ax.barh(np.arange(len(methods)), worst,
            color=[cols[m] for m in methods], height=0.6, zorder=3)
    ax.set_yticks(np.arange(len(methods)))
    ax.set_yticklabels([m.replace(" ", "\n", 1) for m in methods], fontsize=6.0)
    ax.invert_yaxis()
    _bar_labels(ax, np.array(worst), fmt="{:.2f} pp", fs=6.0)
    ax.set_xlim(0, max(worst) * 1.45 if worst else 1)
    _finish(ax, "Worst-bin deviation from nominal", "percentage points")

    _panels(axes)
    _safe_tight(fig)
    return fig, d


@figure("F28", "Residual diagnostics and per-station error", "06_results")
def fig_residuals():
    if NAME_PROPOSED not in PREDICTIONS:
        return _skip("F28", "no CRISP-Net predictions")
    yt = PREDICTIONS[NAME_PROPOSED]["y_true"]
    yp = PREDICTIONS[NAME_PROPOSED]["y_pred"]
    rng = np.random.default_rng(CFG.seed)
    fig, axes = new_fig("double", 2.3, 1, 4)

    a, p = yt.ravel(), yp.ravel()
    sub = rng.choice(len(a), size=min(len(a), 40000), replace=False)
    axes[0].scatter(a[sub], p[sub], s=0.8, alpha=0.12, color=M3["blue600"], lw=0)
    lim = float(np.percentile(a, 99.7))
    axes[0].plot([0, lim], [0, lim], color=ACCENT, lw=0.9)
    axes[0].set_xlim(0, lim)
    axes[0].set_ylim(0, lim)
    r2 = RESULTS[NAME_PROPOSED]["overall"]["R2"]
    _finish(axes[0], f"Predicted vs observed (R² = {r2:.4f})", "observed", "predicted")

    res = p - a
    axes[1].hist(res[sub], bins=70, color=M3["blue600"], edgecolor="none")
    axes[1].axvline(0, color=ACCENT, lw=0.9)
    _finish(axes[1], f"Residuals (bias {res.mean():+.2f})", "predicted − observed", "count")
    _thousands(axes[1])

    for ci, nm in enumerate(CHANNEL_NAMES):
        r = (yp[..., ci] - yt[..., ci]).ravel()
        s = rng.choice(len(r), size=min(len(r), 20000), replace=False)
        axes[2].hist(r[s], bins=60, histtype="step", lw=1.0,
                     color=CHANNEL_COLORS[nm], label=nm, density=True)
    axes[2].axvline(0, color=INK, lw=0.7)
    _finish(axes[2], "Residuals by direction", "error", "density", legend=True)

    st_err = np.sqrt(((yp - yt) ** 2).mean(axis=(0, 1, 3)))
    st_vol = yt.mean(axis=(0, 1, 3))
    axes[3].scatter(st_vol, st_err, s=5, color=M3["blue600"], alpha=0.7, lw=0)
    for i in np.argsort(-st_err)[:4]:
        axes[3].annotate(str(int(STATION_IDS[i])), (st_vol[i], st_err[i]), fontsize=5.6)
    _finish(axes[3], "Station RMSE vs demand", "mean flow", "RMSE")
    _panels(axes)
    _safe_tight(fig)
    return fig, pd.DataFrame({"station_id": STATION_IDS, "rmse": st_err, "mean_flow": st_vol})


## 39 · Figures — uncertainty and calibration

In [43]:
@figure("F29", "Coverage calibration against the nominal level", "07_uncertainty")
def fig_calibration():
    if not len(TBL_UNC):
        return _skip("F29", "no uncertainty results")
    target = 100 * (1 - CFG.alpha)
    fig, axes = new_fig("double", 2.4, 1, 3, gridspec_kw={"width_ratios": [1.2, 1.2, 1.1]})

    # (a) PICP before and after conformal correction
    ax = axes[0]
    d = TBL_UNC.sort_values("PICP")
    y = np.arange(len(d))
    w = 0.38
    ax.barh(y - w / 2, d["PICP_uncal"], w, color=M3["blue100"],
            edgecolor=M3["blue600"], lw=0.5, label="raw quantiles")
    ax.barh(y + w / 2, d["PICP"], w, color=ACCENT, label="after ECQ")
    ax.axvline(target, color=M3["red"], ls="--", lw=1.0,
               label=f"nominal {target:.0f}%")
    ax.set_yticks(y)
    ax.set_yticklabels([dname(m) for m in d["Model"]], fontsize=6.2)
    ax.set_xlim(min(50, np.nanmin(d[["PICP", "PICP_uncal"]].to_numpy()) - 5), 102)
    ax.legend(fontsize=5.9, loc="lower left")
    _finish(ax, "Empirical coverage (PICP)", "coverage (%)")

    # (b) sharpness–coverage trade-off
    ax = axes[1]
    for _, r in TBL_UNC.iterrows():
        c = ACCENT if r["Model"] == NAME_PROPOSED else bar_color(r["Model"])
        ax.scatter(r["MPIW"], r["PICP"], s=42 if r["Model"] == NAME_PROPOSED else 24,
                   color=c, edgecolor="white", lw=0.6, zorder=4)
        ax.annotate(dname(r["Model"]), (r["MPIW"], r["PICP"]), fontsize=5.8,
                    xytext=(4, 3), textcoords="offset points", color=INK)
    ax.axhline(target, color=M3["red"], ls="--", lw=0.9)
    ax.axhspan(target - 1.5, target + 1.5, color=M3["green"], alpha=0.08, lw=0)
    _finish(ax, "Sharpness vs coverage", "MPIW (passengers / 10 min)", "PICP (%)")
    ax.annotate("ideal: narrow and on-target",
                xy=(0.03, 0.06), xycoords="axes fraction", fontsize=5.9, color=MUTED)

    # (c) Winkler score — the joint proper scoring rule
    ax = axes[2]
    s = TBL_UNC.sort_values("Winkler")
    ax.barh(range(len(s)), s["Winkler"],
            color=[ACCENT if m == NAME_PROPOSED else M3["blue600"] for m in s["Model"]],
            height=0.66)
    ax.set_yticks(range(len(s)))
    ax.set_yticklabels([dname(m) for m in s["Model"]], fontsize=6.2)
    ax.invert_yaxis()
    _bar_labels(ax, s["Winkler"].to_numpy(), fmt="{:.1f}", fs=6.0)
    ax.set_xlim(0, s["Winkler"].max() * 1.2)
    _finish(ax, "Winkler interval score", "score (lower better)")
    _panels(axes)
    _safe_tight(fig)
    return fig, TBL_UNC


@figure("F30", "Conditional coverage inside each Mondrian bin", "07_uncertainty")
def fig_conditional_coverage():
    rows = []
    for m in UNC_QUEUE:
        b = RESULTS[m].get("uncertainty_bins")
        if not b:
            continue
        df = pd.DataFrame(b)
        df["model"] = m
        rows.append(df)
    if not rows:
        return _skip("F30", "no per-bin coverage recorded")
    d = pd.concat(rows, ignore_index=True)
    d["bin"] = [BIN_LABELS[(bool(e), bool(p))] for e, p in zip(d["event"], d["peak"])]
    target = 100 * (1 - CFG.alpha)
    bins = [BIN_LABELS[k] for k in [(False, False), (False, True), (True, False), (True, True)]]

    fig, axes = new_fig("double", 2.5, 1, 3, gridspec_kw={"width_ratios": [1.5, 1.2, 1.1]})

    ax = axes[0]
    models = list(dict.fromkeys(d["model"]))
    w = 0.8 / max(len(models), 1)
    for i, m in enumerate(models):
        sub = d[d["model"] == m].set_index("bin").reindex(bins)
        ax.bar(np.arange(len(bins)) + (i - (len(models) - 1) / 2) * w, sub["PICP"], w,
               color=ACCENT if m == NAME_PROPOSED else bar_color(m), label=dname(m))
    ax.axhline(target, color=M3["red"], ls="--", lw=1.0, label=f"nominal {target:.0f}%")
    ax.set_xticks(range(len(bins)))
    ax.set_xticklabels([b.replace(" · ", "\n") for b in bins], fontsize=6.0)
    ax.set_ylim(min(60, np.nanmin(d["PICP"]) - 4), 104)
    ax.legend(fontsize=5.8, ncol=2, loc="lower left")
    _finish(ax, "Coverage per bin", None, "PICP (%)")

    ax = axes[1]
    sub = d[d["model"] == NAME_PROPOSED].set_index("bin").reindex(bins)
    ax.bar(range(len(bins)), sub["MPIW"],
           color=[M3["blue100"], M3["blue600"], "#fdd9b5", ACCENT], width=0.6)
    ax.set_xticks(range(len(bins)))
    ax.set_xticklabels([b.replace(" · ", "\n") for b in bins], fontsize=6.0)
    _bar_labels(ax, sub["MPIW"].to_numpy(), fmt="{:.1f}", horizontal=False, fs=6.0)
    ax.set_ylim(0, np.nanmax(sub["MPIW"]) * 1.22)
    _finish(ax, "Interval width per bin", None, "MPIW")

    ax = axes[2]
    if NAME_PROPOSED in CONFORMAL and CONFORMAL[NAME_PROPOSED].fitted:
        rep = CONFORMAL[NAME_PROPOSED].report()
        q = np.nan_to_num(rep["q_hat"].to_numpy(dtype=float),
                          nan=0.0, posinf=0.0, neginf=0.0)
        ax.barh(range(len(rep)), q,
                color=[M3["green"] if "Mondrian" in mm else M3["amber"]
                       for mm in rep["method"]], height=0.62)
        ax.set_yticks(range(len(rep)))
        ax.set_yticklabels([b.replace(" · ", "\n") for b in rep["bin"]], fontsize=6.0)
        ax.invert_yaxis()
        ax.axvline(0, color=INK, lw=0.7)
        _bar_labels(ax, q, fmt="{:+.2f}", fs=6.0)
        # q̂ is SIGNED: positive widens a too-narrow interval, negative tightens an
        span = max(float(np.abs(q).max()), 1e-6)
        ax.set_xlim(min(-0.18 * span, q.min() * 1.5), max(0.18 * span, q.max() * 1.5))
        ax.legend(handles=[Patch(color=M3["green"], label="own bin"),
                           Patch(color=M3["amber"], label="fallback")],
                  fontsize=5.8, loc="best")
        _finish(ax, r"Calibration constant $\hat{q}$", "widen (+) / tighten (−)")
    else:
        ax.set_axis_off()
    _panels(axes)
    _safe_tight(fig)
    return fig, d


@figure("F31", "Peak containment on event days", "07_uncertainty")
def fig_peak_containment():
    if not len(TBL_UNC) or not TEST_EVENT_MASK.any():
        return _skip("F31", "no event windows or no interval models")
    fig, axes = new_fig("double", 2.4, 1, 3, gridspec_kw={"width_ratios": [1.1, 1.5, 1.1]})

    ax = axes[0]
    s = TBL_UNC.sort_values("PCR", ascending=False)
    ax.barh(range(len(s)), s["PCR"],
            color=[ACCENT if m == NAME_PROPOSED else M3["blue600"] for m in s["Model"]],
            height=0.66)
    ax.set_yticks(range(len(s)))
    ax.set_yticklabels([dname(m) for m in s["Model"]], fontsize=6.2)
    ax.invert_yaxis()
    ax.axvline(100 * (1 - CFG.alpha), color=M3["red"], ls="--", lw=0.9)
    _bar_labels(ax, s["PCR"].to_numpy(), fmt="{:.1f}%", fs=6.0)
    ax.set_xlim(0, 108)
    _finish(ax, "Peak Containment Rate", "% of true peaks ≤ upper bound")

    # (b) where the peaks fall relative to the bound
    ax = axes[1]
    if NAME_PROPOSED in PREDICTIONS and PREDICTIONS[NAME_PROPOSED].get("hi") is not None:
        pr = PREDICTIONS[NAME_PROPOSED]
        lo, hi = pr["lo"], pr["hi"]
        if NAME_PROPOSED in CONFORMAL and CONFORMAL[NAME_PROPOSED].fitted:
            lo, hi = CONFORMAL[NAME_PROPOSED].apply(lo, hi, TEST_EVENT_MASK, TEST_PEAK_MASK)
        yt = pr["y_true"][TEST_EVENT_MASK]
        hh = hi[TEST_EVENT_MASK]
        thr = np.quantile(yt, CFG.event_peak_quantile)
        m = yt >= max(thr, 1.0)
        yv, hv = yt[m], hh[m]
        rng = np.random.default_rng(CFG.seed)
        sub = rng.choice(len(yv), size=min(len(yv), 25000), replace=False)
        contained = yv[sub] <= hv[sub]
        ax.scatter(yv[sub][contained], hv[sub][contained], s=1.2, lw=0,
                   color=M3["green"], alpha=0.25, label="contained")
        ax.scatter(yv[sub][~contained], hv[sub][~contained], s=1.6, lw=0,
                   color=M3["red"], alpha=0.5, label="exceeded")
        lim = float(np.percentile(yv, 99.5))
        ax.plot([0, lim], [0, lim], color=INK, lw=0.8, ls="--")
        ax.set_xlim(0, lim)
        ax.set_ylim(0, lim * 1.35)
        ax.legend(fontsize=5.9, loc="upper left", markerscale=4)
        _finish(ax, "Event-day peaks vs the predicted upper bound",
                "observed peak", "upper bound")

    ax = axes[2]
    ax.axis("off")
    if NAME_PROPOSED in RESULTS and "uncertainty" in RESULTS[NAME_PROPOSED]:
        u = RESULTS[NAME_PROPOSED]["uncertainty"]
        txt = (f"{dname(NAME_PROPOSED)} — event-day peaks\n\n"
               f"PCR            {u.get('PCR', float('nan')):.2f}%\n"
               f"PICP           {u.get('PICP', float('nan')):.2f}%\n"
               f"nominal        {u.get('target_PICP', float('nan')):.0f}%\n"
               f"MPIW           {u.get('MPIW', float('nan')):.2f}\n"
               f"Winkler        {u.get('Winkler', float('nan')):.2f}\n"
               f"peak cells     {u.get('n_peak_cells', 0):,}\n\n"
               "PCR is the operational quantity: an\n"
               "interval can meet nominal coverage\n"
               "overall and still under-bound the\n"
               "peaks, which are the only cells a\n"
               "hold-the-train decision depends on.")
        ax.text(0.0, 0.98, txt, transform=ax.transAxes, fontsize=6.3, va="top",
                ha="left", color=INK, family="monospace",
                bbox=dict(fc=M3["blue50"], ec=M3["outline"], lw=0.5,
                          boxstyle="round,pad=0.5"))
    _panels(axes[:2])
    _safe_tight(fig)
    return fig, TBL_UNC[["Model", "PCR", "PICP", "MPIW", "Winkler"]]


@figure("F32", "Prediction intervals through an event day", "07_uncertainty")
def fig_interval_fan():
    if NAME_PROPOSED not in PREDICTIONS or PREDICTIONS[NAME_PROPOSED].get("hi") is None:
        return _skip("F32", "no interval forecasts")
    pr = PREDICTIONS[NAME_PROPOSED]
    yt, yp, lo, hi = pr["y_true"], pr["y_pred"], pr["lo"], pr["hi"]
    raw_lo, raw_hi = lo.copy(), hi.copy()
    if NAME_PROPOSED in CONFORMAL and CONFORMAL[NAME_PROPOSED].fitted:
        lo, hi = CONFORMAL[NAME_PROPOSED].apply(lo, hi, TEST_EVENT_MASK, TEST_PEAK_MASK)
    if TEST_EVENT_MASK.any():
        ev_idx = np.where(TEST_EVENT_MASK)[0]
        flat = yt[ev_idx, 0, :, CH_IN]
        pos = np.unravel_index(np.argmax(flat), flat.shape)
        ni = int(pos[1])
        day_of = TENS["day_index"][IDX_TE[ev_idx] + L]
        sel = ev_idx[day_of == day_of[pos[0]]]
    else:
        ni = int(np.argmax(yt.mean(axis=(0, 1, 3))))
        sel = np.arange(min(len(yt), BINS_PER_DAY))
    if len(sel) < 4:
        return _skip("F32", "selected day has too few windows")
    t = TIME_INDEX[IDX_TE[sel] + L]

    fig, axes = new_fig("double", 2.9, 2, 1, sharex=True)
    for ci, (ch, nm) in enumerate([(CH_IN, "Entry"), (CH_OUT, "Exit")]):
        ax = axes[ci]
        ax.fill_between(t, raw_lo[sel, 0, ni, ch], raw_hi[sel, 0, ni, ch],
                        color=M3["blue600"], alpha=0.14, lw=0, label="raw quantiles")
        ax.fill_between(t, lo[sel, 0, ni, ch], hi[sel, 0, ni, ch],
                        color=ACCENT, alpha=0.20, lw=0,
                        label=f"conformal {100 * (1 - CFG.alpha):.0f}% PI")
        ax.plot(t, yp[sel, 0, ni, ch], lw=1.0, color=ACCENT, label="median forecast")
        ax.plot(t, yt[sel, 0, ni, ch], lw=1.2, color=INK, label="observed")
        out = (yt[sel, 0, ni, ch] < lo[sel, 0, ni, ch]) | (yt[sel, 0, ni, ch] > hi[sel, 0, ni, ch])
        if out.any():
            ax.scatter(t[out], yt[sel, 0, ni, ch][out], s=12, color=M3["red"],
                       zorder=6, label="outside PI")
        cov = 100 * (~out).mean()
        _finish(ax, f"Station {int(STATION_IDS[ni])} — {nm}   "
                    f"(coverage on this day {cov:.1f}%)", None, "taps / 10 min")
        if ci == 0:
            ax.legend(fontsize=5.8, ncol=5, loc="upper left")
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axes[-1].set_xlabel("time of day", fontsize=7.5, fontweight="bold", color=MUTED)
    _panels(axes)
    _safe_tight(fig)
    return fig, pd.DataFrame({"t": t,
                              "observed_entry": yt[sel, 0, ni, CH_IN],
                              "lo_entry": lo[sel, 0, ni, CH_IN],
                              "hi_entry": hi[sel, 0, ni, CH_IN]})


@figure("F33", "Conformal transfer across architectures", "07_uncertainty")
def fig_conformal_transfer():
    """Contribution 2."""
    if not TRANSFER:
        return _skip("F33", "no transferred baselines")
    target = 100 * (1 - CFG.alpha)
    rows = []
    for n, t in TRANSFER.items():
        u = uncertainty_suite(t["y_true"], t["lo"], t["hi"], TEST_EVENT_MASK)
        rows.append({"model": n, **u})
    if NAME_PROPOSED in RESULTS and "uncertainty" in RESULTS[NAME_PROPOSED]:
        rows.append({"model": NAME_PROPOSED, **RESULTS[NAME_PROPOSED]["uncertainty"]})
    d = pd.DataFrame(rows).sort_values("MPIW").reset_index(drop=True)
    cols = [ACCENT if m == NAME_PROPOSED else M3["blue600"] for m in d["model"]]
    labs = [dname(m) + ("" if m == NAME_PROPOSED else " + ECQ") for m in d["model"]]

    fig, axes = new_fig("double", 2.5, 1, 3)

    ax = axes[0]
    y = np.arange(len(d))
    ax.barh(y, d["PICP"], 0.62, color=cols, zorder=3)
    ax.axvline(target, color=M3["red"], ls="--", lw=1.0, zorder=4)
    ax.axvspan(target - 1.5, target + 1.5, color=M3["green"], alpha=0.10, lw=0, zorder=0)
    ax.set_yticks(y); ax.set_yticklabels(labs, fontsize=6.1); ax.invert_yaxis()
    ax.set_xlim(min(84, float(d["PICP"].min()) - 2), max(94, float(d["PICP"].max()) + 2))
    _bar_labels(ax, d["PICP"].to_numpy(), fmt="{:.1f}%", fs=6.0)
    _finish(ax, f"Coverage after transfer (target {target:.0f}%)", "PICP (%)")

    ax = axes[1]
    ax.barh(y, d["MPIW"], 0.62, color=cols, zorder=3)
    ax.set_yticks(y); ax.set_yticklabels(labs, fontsize=6.1); ax.invert_yaxis()
    _bar_labels(ax, d["MPIW"].to_numpy(), fmt="{:.1f}", fs=6.0)
    ax.set_xlim(0, float(d["MPIW"].max()) * 1.3)
    _finish(ax, "Width paid for that coverage", "MPIW (passengers / 10 min)")

    ax = axes[2]
    ax.barh(y, d["PCR"], 0.62, color=cols, zorder=3)
    ax.axvline(90, color=M3["red"], ls="--", lw=0.9, zorder=4)
    ax.set_yticks(y); ax.set_yticklabels(labs, fontsize=6.1); ax.invert_yaxis()
    _bar_labels(ax, d["PCR"].to_numpy(), fmt="{:.1f}%", fs=6.0)
    ax.set_xlim(0, 108)
    _finish(ax, "Peak containment on event days", "PCR (%)")

    _panels(axes)
    _safe_tight(fig)
    return fig, d


@figure("F34", "Persona attribution: the learned input gates", "07_uncertainty")
def fig_persona_gates():
    """The PRI module's interpretability payload, and the answer to the question our programme has asked in four papers: WHICH behavioural...."""
    m = restore_model(NAME_PROPOSED)
    if m is None or not getattr(m, "use_gates", False):
        return _skip("F34", "no gated CRISP-Net checkpoint")
    g = m.persona_gates()
    K = CFG.n_personas

    fig, axes = new_fig("double", 2.4, 1, 3,
                        gridspec_kw={"width_ratios": [1.3, 1.15, 1.0]})

    # (a) gate value per persona and direction
    ax = axes[0]
    x = np.arange(K)
    w = 0.38
    for i, ch in enumerate(CHANNEL_NAMES):
        sub = g[g["direction"] == ch].set_index("persona").reindex(CFG.persona_names)
        ax.bar(x + (i - 0.5) * w, sub["gate"], w, color=CHANNEL_COLORS[ch],
               label=ch, zorder=3)
    ax.axhline(CFG.gate_init, color=INK, lw=0.8, ls="--", zorder=4)
    ax.annotate("initialisation", xy=(K - 0.55, CFG.gate_init), fontsize=5.8,
                color=MUTED, va="bottom", ha="right")
    ax.set_xticks(x)
    ax.set_xticklabels([p.replace("/", "/\n") for p in CFG.persona_names], fontsize=6.0)
    ax.legend(fontsize=6.0, ncol=2, loc="upper right")
    _finish(ax, "Learned gate per persona channel", None, "softplus(g)")

    # (b) share of each direction's total gate mass
    ax = axes[1]
    piv = g.pivot(index="persona", columns="direction",
                  values="share_of_direction").reindex(CFG.persona_names)
    bot = np.zeros(len(CHANNEL_NAMES))
    for k, p in enumerate(CFG.persona_names):
        vals = 100 * piv.loc[p, CHANNEL_NAMES].to_numpy(dtype=float)
        ax.bar(np.arange(len(CHANNEL_NAMES)), vals, 0.55, bottom=bot,
               color=PERSONA_COLORS[k % len(PERSONA_COLORS)], label=p, zorder=3)
        for j, v in enumerate(vals):
            if v > 6:
                ax.text(j, bot[j] + v / 2, f"{v:.0f}%", ha="center", va="center",
                        fontsize=5.8, color="white", fontweight="semibold")
        bot += vals
    ax.set_xticks(np.arange(len(CHANNEL_NAMES)))
    ax.set_xticklabels(CHANNEL_NAMES, fontsize=6.4)
    ax.set_ylim(0, 118)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.legend(fontsize=5.7, ncol=2, loc="upper center", columnspacing=0.8)
    _finish(ax, "Share of gate mass by direction", None, "% of direction total")

    # (c) gate weight against corpus share — is the model just tracking volume?
    ax = axes[2]
    share = np.asarray(PI_PRIOR, dtype=float).ravel()[:K]
    share = 100 * share / max(share.sum(), 1e-9)
    gm = g.groupby("persona")["gate"].mean().reindex(CFG.persona_names).to_numpy()
    for k, p in enumerate(CFG.persona_names):
        ax.scatter(share[k], gm[k], s=46, color=PERSONA_COLORS[k % len(PERSONA_COLORS)],
                   edgecolor="white", lw=0.6, zorder=4)
        ax.annotate(p.split("/")[0], (share[k], gm[k]), fontsize=5.8,
                    xytext=(4, 3), textcoords="offset points", color=INK)
    if np.isfinite(gm).all() and len(share) == len(gm):
        r = float(np.corrcoef(share, gm)[0, 1]) if np.std(share) > 0 else np.nan
        ax.annotate(f"r = {r:+.2f}", xy=(0.04, 0.92), xycoords="axes fraction",
                    fontsize=6.2, color=INK, fontweight="semibold")
    ax.axhline(CFG.gate_init, color=INK, lw=0.7, ls="--")
    _finish(ax, "Gate vs corpus share", "share of taps (%)", "mean gate")
    ax.annotate("a flat relation means the gates carry\nbehaviour, not volume",
                xy=(0.04, 0.06), xycoords="axes fraction", fontsize=5.7, color=MUTED)

    _panels(axes)
    _safe_tight(fig)
    save_df(g, f"{P['results']}/persona_gates_{CFG.dataset}_{CFG.exp_version}.csv")
    return fig, g


# ── render, in the order the paper presents them ────────────────────────────
RESULT_FIGURES = [fig_training_curves, fig_metric_comparison, fig_channel_breakdown,
                  fig_horizon_degradation, fig_prediction_traces, fig_event_zoom,
                  fig_ablation, fig_decisive_coverage, fig_residuals,
                  fig_calibration, fig_conditional_coverage, fig_peak_containment,
                  fig_interval_fan, fig_conformal_transfer, fig_persona_gates]


def render_result_figures(force: bool = False):
    ok = 0
    for fn in tqdm(RESULT_FIGURES, desc="Result figures"):
        if fn(force=force):
            ok += 1
    log(f"Result figures done: {ok}/{len(RESULT_FIGURES)}")
    return ok


render_result_figures()
figure_status_report()


Result figures:   0%|          | 0/15 [00:00<?, ?it/s]

07:48:44 │ INFO    │ ✓ figure  [F20] done — loaded from disk (force=True to redraw)
07:48:44 │ INFO    │ ✓ figure  [F21] done — loaded from disk (force=True to redraw)
07:48:44 │ INFO    │ ✓ figure  [F22] done — loaded from disk (force=True to redraw)
07:48:44 │ INFO    │ ✓ figure  [F23] done — loaded from disk (force=True to redraw)
07:48:44 │ INFO    │ ✓ figure  [F24] done — loaded from disk (force=True to redraw)
07:48:44 │ INFO    │ ✓ figure  [F25] done — loaded from disk (force=True to redraw)
07:48:44 │ INFO    │ ✓ figure  [F26] done — loaded from disk (force=True to redraw)
07:48:44 │ INFO    │ ✓ figure  [F27] done — loaded from disk (force=True to redraw)
07:48:44 │ INFO    │ ✓ figure  [F28] done — loaded from disk (force=True to redraw)
07:48:45 │ INFO    │ ✓ figure  [F29] done — loaded from disk (force=True to redraw)
07:48:45 │ INFO    │ ✓ figure  [F30] done — loaded from disk (force=True to redraw)
07:48:45 │ INFO    │ ✓ figure  [F31] done — loaded from disk (force=True to 

,figure_id,title,group,func,status,png,pdf,csv,kb,path
0,F01,Ingestion audit and data retention across sour...,01_corpus,fig_ingestion_audit,DONE,True,True,True,401.2,/content/drive/MyDrive/CRISP_Net/figures/01_co...
1,F02,Longitudinal network ridership by direction wi...,01_corpus,fig_daily_ridership,DONE,True,True,True,397.3,/content/drive/MyDrive/CRISP_Net/figures/01_co...
2,F03,Service-hour completeness and data availability,01_corpus,fig_coverage_calendar,DONE,True,True,True,346.2,/content/drive/MyDrive/CRISP_Net/figures/01_co...
3,F04,Ticket-code volume distribution and cumulative...,01_corpus,fig_card_pareto,DONE,True,True,True,273.1,/content/drive/MyDrive/CRISP_Net/figures/01_co...
4,F05,Diurnal temporal signature of each ticket code,01_corpus,fig_card_signature_heatmap,DONE,True,True,True,445.2,/content/drive/MyDrive/CRISP_Net/figures/01_co...
5,F06,Spatial demand concentration across the network,01_corpus,fig_demand_concentration,DONE,True,True,True,244.2,/content/drive/MyDrive/CRISP_Net/figures/01_co...
6,F07,Entry-to-exit propagation across the network,01_corpus,fig_delay_evidence,DONE,True,True,True,270.9,/content/drive/MyDrive/CRISP_Net/figures/01_co...
7,F08,Passenger personas in behavioural feature space,02_persona,fig_persona_space,DONE,True,True,True,214.5,/content/drive/MyDrive/CRISP_Net/figures/02_pe...
8,F09,"Diurnal persona profiles, weekday versus weekend",02_persona,fig_persona_profiles,DONE,True,True,True,390.2,/content/drive/MyDrive/CRISP_Net/figures/02_pe...
9,F10,Soft-assignment prior matrix P0 inherited from...,02_persona,fig_prior_matrix,DONE,True,True,True,448.1,/content/drive/MyDrive/CRISP_Net/figures/02_pe...


## 40 · Figure — persona gates

In [55]:
@figure("F34", "Persona attribution: the learned input gates", "07_uncertainty")
def fig_persona_gates():
    m = restore_model(NAME_PROPOSED)
    if m is None or not getattr(m, "use_gates", False):
        return _skip("F34", "no gated CRISP-Net checkpoint")
    g = m.persona_gates()
    K = CFG.n_personas
    pnames = list(CFG.persona_names)[:K]
    chans = list(CHANNEL_NAMES)

    # explicit, dtype-safe matrices — no .loc[scalar, list] anywhere
    gate_m = np.full((len(chans), K), np.nan)
    share_m = np.full((len(chans), K), np.nan)
    for _, r in g.iterrows():
        if r["direction"] in chans and r["persona"] in pnames:
            i, j = chans.index(r["direction"]), pnames.index(r["persona"])
            gate_m[i, j] = float(r["gate"])
            share_m[i, j] = float(r["share_of_direction"])

    spread = float(np.nanmax(gate_m) - np.nanmin(gate_m))
    inert = spread < 0.25 * CFG.gate_init

    fig, axes = new_fig("double", 2.4, 1, 3,
                        gridspec_kw={"width_ratios": [1.3, 1.15, 1.0]})

    # (a) gate value per persona and direction
    ax = axes[0]
    x = np.arange(K)
    w = 0.38
    for i, ch in enumerate(chans):
        ax.bar(x + (i - 0.5) * w, gate_m[i], w, color=CHANNEL_COLORS[ch],
               label=ch, zorder=3)
    ax.axhline(CFG.gate_init, color=INK, lw=0.9, ls="--", zorder=4)
    ax.annotate("initialisation", xy=(K - 0.55, CFG.gate_init), fontsize=5.8,
                color=MUTED, va="bottom", ha="right")
    ax.set_xticks(x)
    ax.set_xticklabels([p.replace("/", "/\n") for p in pnames], fontsize=6.0)
    lo_, hi_ = np.nanmin(gate_m), np.nanmax(gate_m)
    pad = max(0.12 * (hi_ - lo_), 0.05)
    ax.set_ylim(min(lo_, CFG.gate_init) - pad, max(hi_, CFG.gate_init) + pad)
    ax.legend(fontsize=6.0, ncol=2, loc="upper right")
    _finish(ax, "Learned gate per persona channel", None, "softplus(g)")
    ax.annotate(f"spread {spread:.3f}" + ("  —  gates stayed at identity" if inert else ""),
                xy=(0.03, 0.04), xycoords="axes fraction", fontsize=5.8,
                color=M3["red"] if inert else MUTED)

    # (b) share of each direction's total gate mass
    ax = axes[1]
    bot = np.zeros(len(chans))
    for k, p in enumerate(pnames):
        vals = 100.0 * share_m[:, k]
        ax.bar(np.arange(len(chans)), vals, 0.55, bottom=bot,
               color=PERSONA_COLORS[k % len(PERSONA_COLORS)], label=p, zorder=3)
        for j, v in enumerate(vals):
            if np.isfinite(v) and v > 6:
                ax.text(j, bot[j] + v / 2, f"{v:.0f}%", ha="center", va="center",
                        fontsize=5.8, color="white", fontweight="semibold")
        bot += np.nan_to_num(vals)
    ax.set_xticks(np.arange(len(chans)))
    ax.set_xticklabels(chans, fontsize=6.4)
    ax.set_ylim(0, 118)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.legend(fontsize=5.7, ncol=2, loc="upper center", columnspacing=0.8)
    _finish(ax, "Share of gate mass by direction", None, "% of direction total")
    ax.annotate(f"even split = {100 / K:.0f}% each", xy=(0.03, 0.04),
                xycoords="axes fraction", fontsize=5.8, color=MUTED)

    # (c) gate vs corpus share — DESCRIPTIVE ONLY at K = 4
    ax = axes[2]
    share = np.asarray(PI_PRIOR, dtype=float).ravel()[:K]
    share = 100 * share / max(share.sum(), 1e-9)
    gm = np.nanmean(gate_m, axis=0)
    for k, p in enumerate(pnames):
        ax.scatter(share[k], gm[k], s=46, color=PERSONA_COLORS[k % len(PERSONA_COLORS)],
                   edgecolor="white", lw=0.6, zorder=4)
        ax.annotate(p.split("/")[0], (share[k], gm[k]), fontsize=5.8,
                    xytext=(4, 3), textcoords="offset points", color=INK)
    if np.isfinite(gm).all() and np.std(share) > 0:
        r = float(np.corrcoef(share, gm)[0, 1])
        ax.annotate(f"r = {r:+.2f}  (n = {K})", xy=(0.04, 0.92),
                    xycoords="axes fraction", fontsize=6.2, color=MUTED)
    ax.axhline(CFG.gate_init, color=INK, lw=0.8, ls="--")
    _finish(ax, "Gate vs corpus share", "share of taps (%)", "mean gate")
    ax.annotate("four points: descriptive only,\nno correlation is claimed",
                xy=(0.04, 0.06), xycoords="axes fraction", fontsize=5.7, color=MUTED)

    _panels(axes)
    _safe_tight(fig)
    save_df(g, f"{P['results']}/persona_gates_{CFG.dataset}_{CFG.exp_version}.csv")
    return fig, g


# figures are cached on disk — force=True is required to redraw
fig_persona_gates(force=True)
print("\nGate summary for the Discussion section:")
_g = restore_model(NAME_PROPOSED).persona_gates()
print(_g.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


08:45:04 │ INFO    │ CRISP-Net restored from 2023__v2__CRISP-Net/best.pt
08:45:06 │ INFO    │ 🖼  figure  [F34] Persona attribution: the learned input gates → F34_fig_persona_gates.png

Gate summary for the Discussion section:
08:45:06 │ INFO    │ CRISP-Net restored from 2023__v2__CRISP-Net/best.pt
direction            persona   gate  share_of_direction
    Entry           Commuter 1.1080              0.2637
    Entry             Casual 1.0586              0.2520
    Entry    Tourist/Visitor 1.0050              0.2392
    Entry Concession/Student 1.0295              0.2451
     Exit           Commuter 1.0908              0.2614
     Exit             Casual 1.0599              0.2540
     Exit    Tourist/Visitor 0.9917              0.2376
     Exit Concession/Student 1.0308              0.2470


## 41 · Interpretability export

In [56]:
INTERP: Dict[str, object] = {}

_m = restore_model(NAME_PROPOSED)
if _m is None:
    print("CRISP-Net checkpoint not found — run the experiment runner first.")
else:
    _m.eval()
    print("═" * 72)
    print(f"INTERPRETABILITY — {NAME_PROPOSED}")
    print("═" * 72)

    # ── persona gates ───────────────────────────────────────────────────────
    if getattr(_m, "use_gates", False):
        _g = _m.persona_gates()
        _share = np.asarray(PI_PRIOR, dtype=float).ravel()[:CFG.n_personas]
        _share = 100 * _share / max(_share.sum(), 1e-9)
        _gm = _g.groupby("persona")["gate"].mean().reindex(CFG.persona_names).to_numpy()
        _spread = float(np.ptp(_g["gate"].to_numpy()))
        _r = (float(np.corrcoef(_share, _gm)[0, 1])
              if np.std(_share) > 0 and np.isfinite(_gm).all() else float("nan"))

        print("\nPersona gates (init %.2f)" % CFG.gate_init)
        print(f"  {'persona':<20s} {'entry':>7s} {'exit':>7s} {'share%':>8s}")
        for _k, _p in enumerate(CFG.persona_names):
            _e = float(_g[(_g.direction == "Entry") & (_g.persona == _p)]["gate"].iloc[0])
            _x = float(_g[(_g.direction == "Exit") & (_g.persona == _p)]["gate"].iloc[0])
            print(f"  {_p:<20s} {_e:>7.3f} {_x:>7.3f} {_share[_k]:>8.1f}")
        print(f"  spread {_spread:.3f}   r(gate, share) {_r:+.3f} (n={CFG.n_personas})")

        INTERP["persona_gates"] = _g.to_dict("records")
        INTERP["persona_corpus_share"] = (_share / 100).tolist()
        INTERP["gate_spread"] = _spread
        INTERP["gate_vs_share_corr"] = _r
    else:
        print("\nPersona gates: disabled in this checkpoint.")

    # ── fusion ──────────────────────────────────────────────────────────────
    with torch.no_grad():
        _w = torch.softmax(_m.fuse_w, 0).cpu().numpy()
        _mix = float(torch.sigmoid(_m.mix).cpu())
    INTERP["fusion_weights"] = _w.tolist()
    INTERP["adjacency_mix"] = _mix
    print(f"\nFusion weights   temporal {_w[0]:.2f} · adaptive {_w[1]:.2f} · "
          f"topology {_w[2]:.2f}")
    print(f"Adjacency mix    sigma(mix) = {_mix:.3f}")

    # ── calibration regimes ─────────────────────────────────────────────────
    _dec = TBL_DECISIVE if "TBL_DECISIVE" in globals() else pd.DataFrame()
    if len(_dec):
        _worst = _dec.groupby("method", observed=True)["|PICP − 90|"].max().to_dict()
        _ev_pk = _dec[_dec["bin"] == "event · peak"].set_index("method")["PICP"].to_dict()
        _gain = (_worst.get("raw quantile heads", np.nan)
                 - _worst.get("Mondrian ECQ", np.nan))
        INTERP["worst_bin_deviation"] = {str(k): float(v) for k, v in _worst.items()}
        INTERP["event_peak_coverage"] = {str(k): float(v) for k, v in _ev_pk.items()}
        INTERP["mondrian_gain_pp"] = float(_gain) if np.isfinite(_gain) else None

        print(f"\nWorst-bin deviation from nominal (pp)")
        for _k, _v in _worst.items():
            print(f"  {str(_k):<22s} {_v:5.2f}")
        if np.isfinite(_gain):
            print(f"  Mondrian vs raw        {_gain:+5.2f}")

    # ── calibration provenance ──────────────────────────────────────────────
    if NAME_PROPOSED in CONFORMAL:
        _rep = CONFORMAL[NAME_PROPOSED].report()
        INTERP["conformal_report"] = _rep.to_dict("records")
        _fb = sum("fallback" in str(r) for r in _rep["method"])
        print(f"\nCalibration bins  {len(_rep) - _fb}/{len(_rep)} own quantile, "
              f"{_fb} fallback")
        for _, _r in _rep.iterrows():
            _q = _r.get("q_hat", float("nan"))
            print(f"  {_r['bin']:<18s} n={int(_r['n_windows']):>4d}  "
                  f"q={_q:+.3f}  {_r['method']}")

    # ── transfer ────────────────────────────────────────────────────────────
    if TRANSFER:
        _t = {}
        print("\nConformal transfer (no retraining)")
        print(f"  {'model':<28s} {'PICP':>6s} {'MPIW':>7s} {'Winkler':>8s} {'PCR':>6s}")
        for _n, _p in TRANSFER.items():
            _u = uncertainty_suite(_p["y_true"], _p["lo"], _p["hi"], TEST_EVENT_MASK)
            _t[_n] = {k: float(_u.get(k, np.nan)) for k in UNC_COLS}
            print(f"  {dname(_n) + ' + ECQ':<28s} {_u['PICP']:>6.2f} {_u['MPIW']:>7.2f} "
                  f"{_u['Winkler']:>8.2f} {_u['PCR']:>6.2f}")
        if NAME_PROPOSED in RESULTS and "uncertainty" in RESULTS[NAME_PROPOSED]:
            _u = RESULTS[NAME_PROPOSED]["uncertainty"]
            print(f"  {dname(NAME_PROPOSED):<28s} {_u['PICP']:>6.2f} {_u['MPIW']:>7.2f} "
                  f"{_u['Winkler']:>8.2f} {_u['PCR']:>6.2f}")
        INTERP["transfer"] = _t

    INTERP["model"] = NAME_PROPOSED
    INTERP["n_seeds"] = len(CFG.report_seeds)
    INTERP["params"] = int(sum(p.numel() for p in _m.parameters() if p.requires_grad))
    print(f"\nParameters {INTERP['params']:,} · seeds {INTERP['n_seeds']}")
    print("═" * 72)

    save_json(INTERP, f"{P['results']}/interpretability_{CFG.dataset}_"
                      f"{CFG.exp_version}.json")
    log(f"Interpretability exported — {len(INTERP)} entries.")
    del _m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

08:45:11 │ INFO    │ CRISP-Net restored from 2023__v2__CRISP-Net/best.pt
════════════════════════════════════════════════════════════════════════
INTERPRETABILITY — CRISP-Net
════════════════════════════════════════════════════════════════════════

Persona gates (init 1.00)
  persona                entry    exit   share%
  Commuter               1.108   1.091     31.5
  Casual                 1.059   1.060     42.2
  Tourist/Visitor        1.005   0.992     11.5
  Concession/Student     1.029   1.031     14.8
  spread 0.116   r(gate, share) +0.742 (n=4)

Fusion weights   temporal 0.43 · adaptive 0.35 · topology 0.22
Adjacency mix    sigma(mix) = 0.622

Worst-bin deviation from nominal (pp)
  raw quantile heads      1.15
  pooled conformal        0.91
  Mondrian ECQ            1.04
  Mondrian vs raw        +0.11

Calibration bins  4/4 own quantile, 0 fallback
  normal · off-peak  n= 134  q=+0.148  Mondrian bin (n_win=134)
  normal · peak      n=  48  q=-0.168  Mondrian bin (n_win=48)
  

## 42 · Packaging

In [57]:
def _md(df):
    try:
        return df.to_markdown(index=False, floatfmt=".4f")
    except Exception:
        return "```\n" + df.to_string(index=False) + "\n```"


def build_summary() -> str:
    lines = []
    A = lines.append
    A(f"# {CFG.paper} — Results Summary ({CFG.dataset} corpus)\n")
    A(f"_Generated {datetime.now().strftime('%Y-%m-%d %H:%M')} · "
      f"device {DEVICE} · seed {CFG.seed}_\n")

    A("\n## 1. Dataset\n")
    A("| Property | Value |\n|---|---|")
    A(f"| Corpus | Nanjing Metro AFC {CFG.dataset} |")
    # `_audit` comes from the Stage-A cell; guard so a partial re-run (e.g. resuming
    # straight into training from cached tensors) still produces a summary
    _aud = globals().get("_audit")
    if isinstance(_aud, pd.DataFrame) and len(_aud):
        A(f"| Source files ingested | {len(_aud)} |")
        A(f"| Raw records read | {int(_aud['rows_read'].sum()):,} |")
        A(f"| Records retained | {int(_aud['rows_kept'].sum()):,} |")
    A(f"| Directional events (entry + exit) | {int(X_total.sum()):,} |")
    A(f"| Stations (excluding {list(CFG.exclude_station_ids)}) | {N_STATIONS} |")
    A(f"| Lines | {sorted(set(STATION_LINE.values())) if STATION_LINE else 'n/a'} |")
    A(f"| Usable days | {N_DAYS} ({TENS['days'][0]} → {TENS['days'][-1]}) |")
    A(f"| Interval | {CFG.freq_minutes} min, {CFG.service_start}–{CFG.service_end} |")
    A(f"| Ticket channels | {N_CARD} (top {CFG.n_card_channels} + OTHER, "
      f"{TENS['card_coverage'] * 100:.2f}% coverage) |")
    A(f"| Windows train / tune / calib / test | {len(IDX_TR):,} / {len(IDX_VA):,} / "
      f"{len(IDX_CAL):,} / {len(IDX_TE):,} |")
    A(f"| Event days flagged | {len(EVENTS['event_days'])} "
      f"({int(TEST_EVENT_MASK.sum())} test windows) |")
    A(f"| Context vector | z_t ∈ ℝ^{DZ} — {EXO['names']} |")

    A("\n## 2. Components, and what each one measured\n")
    A("| Component | Question it answers | Status |")
    A("|---|---|---|")
    _gs = INTERP.get("gate_spread")
    _gc = INTERP.get("gate_vs_share_corr")
    A(f"| PRI | which personas drive the forecast | gate spread "
      f"{_gs:.3f}, corr with corpus share {_gc:+.2f} |"
      if _gs is not None else "| PRI | which personas drive the forecast | — |")
    A("| Bidirectional head | exit channel discarded in Paper 3 | entry and exit "
      "predicted jointly and calibrated separately |")
    _mg = INTERP.get("mondrian_gain_pp")
    A(f"| ECQ | forecasts carry no risk bound | worst-bin coverage improved "
      f"{_mg:+.2f} pp over raw heads |"
      if _mg is not None else "| ECQ | forecasts carry no risk bound | — |")
    _tr = INTERP.get("transfer") or {}
    A(f"| Transfer | is calibration architecture-specific | {len(_tr)} point "
      f"baselines calibrated without retraining |")
    A("\n**Removed after measurement (CALM-Net pilot, quoted not rerun):** "
      "context-driven dynamic graph (3.72% edge drift), learned delay kernel "
      "(r = −0.897 against measured propagation), FiLM persona conditioning "
      "(γ spread 0.045). The encoder control that attributes the pilot's "
      "regression to the encoder rather than to these modules scored "
      "RMSE 48.076 ± 2.608.\n")

    A("\n## 3. Main results (test set, both channels)\n")
    if len(TBL_MAIN):
        A(_md(TBL_MAIN))
    if len(TBL_EVENT):
        A("\n### Event days only\n")
        A(_md(TBL_EVENT))
    if len(TBL_UNC):
        A("\n## 4. Uncertainty quantification\n")
        A(_md(TBL_UNC))
    if NAME_PROPOSED in RESULTS and RESULTS[NAME_PROPOSED].get("uncertainty_bins"):
        A("\n### Conditional coverage per Mondrian bin\n")
        _b = pd.DataFrame(RESULTS[NAME_PROPOSED]["uncertainty_bins"])
        _b["bin"] = [BIN_LABELS[(bool(e), bool(p))]
                     for e, p in zip(_b["event"], _b["peak"])]
        A(_md(_b[["bin", "n_windows", "PICP", "MPIW", "Winkler"]]))

    A("\n## 5. Training configuration (identical to Paper 3 except the pinball term)\n")
    A("| Hyper-parameter | Value |\n|---|---|")
    for k in ["seq_len", "horizon", "batch_size", "epochs", "lr", "weight_decay",
              "huber_delta", "pinball_weight", "grad_clip", "early_stop_patience",
              "seed", "report_seeds", "alpha", "per_channel_conformal",
              "quantiles", "calib_frac", "ensemble_members"]:
        A(f"| {k} | {getattr(CFG, k)} |")

    A("\n## 6. Figures\n")
    if os.path.exists(FIGURE_INDEX):
        A(_md(pd.read_csv(FIGURE_INDEX)[["figure_id", "group", "title"]]))
    return "\n".join(lines)


SUMMARY = build_summary()
with open(f"{P['exports']}/RESULTS_SUMMARY.md", "w", encoding="utf-8") as f:
    f.write(SUMMARY)
print(SUMMARY[:2600])
print("\n… full summary written to exports/RESULTS_SUMMARY.md")

# ── asset manifest ───────────────────────────────────────────────────────────
_assets = []
for _root, _dirs, _files in os.walk(P["root"]):
    if "checkpoints" in _root:
        continue
    for _f in _files:
        _p = os.path.join(_root, _f)
        with contextlib.suppress(OSError):
            _assets.append({"path": os.path.relpath(_p, P["root"]),
                            "type": os.path.splitext(_f)[1],
                            "kb": round(os.path.getsize(_p) / 1024, 1)})
ASSETS = pd.DataFrame(_assets)
if len(ASSETS):
    save_df(ASSETS, f"{P['exports']}/asset_manifest.csv")
    print(f"\nAssets catalogued: {len(ASSETS)} files, "
          f"{ASSETS['kb'].sum() / 1024:.1f} MB (checkpoints excluded)")
    print(ASSETS.groupby("type")["kb"].agg(["count", "sum"])
          .sort_values("sum", ascending=False).head(12).to_string())

# ── zip bundle: figures, tables, metrics, summary ───────────────────────────
def build_bundle(max_mb: float = 512.0):
    """Zip the paper deliverables."""
    import zipfile
    out = f"{P['exports']}/CRISPNet_paper_bundle_{CFG.dataset}.zip"
    out_real = os.path.realpath(out)

    entries, total = [], 0
    for sub in ["figures", "results/tables", "results/metrics",
                "results/conformal", "exports"]:
        base = os.path.join(P["root"], sub)
        if not os.path.isdir(base):
            continue
        for r, _d, fs in os.walk(base):
            for f in fs:
                if f.endswith(".zip") or f.endswith(".tmp") or ".tmp_" in f:
                    continue
                p = os.path.join(r, f)
                if os.path.realpath(p) == out_real:
                    continue
                try:
                    total += os.path.getsize(p)
                except OSError:
                    continue
                entries.append((p, os.path.relpath(p, P["root"])))

    if not entries:
        log("Nothing to bundle yet.", "warning")
        return None

    size_mb = total / 1e6
    if size_mb > max_mb:
        log(f"Bundle inputs total {size_mb:.0f} MB, above the {max_mb:.0f} MB cap - "
            f"skipping to protect Drive quota. Artefacts remain under {P['root']}.",
            "warning")
        return None

    try:
        free = shutil.disk_usage(P["exports"]).free
        if free < total * 1.3:
            log(f"Only {free / 1e6:.0f} MB free for a ~{size_mb:.0f} MB bundle - "
                f"skipping. Artefacts remain under {P['root']}.", "warning")
            return None
    except OSError:
        pass

    try:
        with atomic_path(out) as tmp:
            with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
                for src_path, arcname in entries:
                    z.write(src_path, arcname)
        return out
    except OSError as exc:
        log(f"Bundle not written ({type(exc).__name__}: {exc}). Every figure, table and "
            f"metric file is already on disk under {P['root']} - only the convenience "
            f"zip is missing.", "warning")
        return None


_bundle = build_bundle()
if _bundle:
    print(f"\n[bundle] {_bundle}  ({os.path.getsize(_bundle) / 1e6:.1f} MB)")
else:
    print(f"\n[bundle] skipped - individual artefacts remain under {P['root']}")
with contextlib.suppress(OSError):
    state_set("pipeline_complete", True)
log(f"PIPELINE COMPLETE — {CFG.paper}.")


# CRISP-Net — Results Summary (2023 corpus)

_Generated 2026-08-17 08:45 · device cuda · seed 42_


## 1. Dataset

| Property | Value |
|---|---|
| Corpus | Nanjing Metro AFC 2023 |
| Source files ingested | 17 |
| Raw records read | 95,800,909 |
| Records retained | 95,733,909 |
| Directional events (entry + exit) | 190,911,927 |
| Stations (excluding [333]) | 191 |
| Lines | [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13] |
| Usable days | 57 (2023-03-01 → 2023-05-31) |
| Interval | 10 min, 06:00–23:50 |
| Ticket channels | 25 (top 24 + OTHER, 99.58% coverage) |
| Windows train / tune / calib / test | 3,640 / 364 / 364 / 819 |
| Event days flagged | 24 (182 test windows) |
| Context vector | z_t ∈ ℝ^8 — ['tod_sin', 'tod_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_peak', 'event_flag', 'event_countdown'] |

## 2. Components, and what each one measured

| Component | Question it answers | Status |
|---|---|---|
| PRI | which personas drive the forecast | gate spread 0.116, corr with corpu